# 06 GHSL Multi-City Parameterized Pipeline

**Purpose:** Convert the validated Bengaluru Spark/Sedona workflow into one reusable pipeline for the remaining metropolitan regions.

## Cities and order

1. Hyderabad
2. Pune metropolitan region
3. Chennai metropolitan region
4. Mumbai metropolitan region
5. Delhi NCR
6. Ahmedabad
7. Surat

Jaipur is configured separately as a held-out generalization city. Bengaluru remains the frozen reference implementation.

## Safety contract

- Do not rerun or modify validated Bengaluru outputs.
- Do not launch a city batch until its extent and raw-input inventory pass.
- Process `1975`, `2000`, and `2020` first as a three-epoch proof.
- Run the remaining epochs only after the proof passes.
- Validate existing HDFS output before skipping it.
- Stop at the first unexplained failure.
- Use Spark, Sedona, HDFS, and Parquet. Do not replace raster stages with local Pandas/Rasterio processing.

In [1]:
# CELL 1: COLAB BOOTSTRAP AND REMOTE EXECUTION HELPER
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path('/content/drive/MyDrive')
REMOTE_EXEC = DRIVE_ROOT / 'remote_exec.sh'
DRIVE_KEY = DRIVE_ROOT / 'hadoop-hdfs.pem'
LOCAL_KEY = Path('/content/hadoop-hdfs.pem')

if not REMOTE_EXEC.is_file():
    raise FileNotFoundError(f'Remote helper not found: {REMOTE_EXEC}')
os.chmod(REMOTE_EXEC, 0o700)

if not LOCAL_KEY.is_file():
    if not DRIVE_KEY.is_file():
        raise FileNotFoundError(f'SSH key not found: {DRIVE_KEY}')
    shutil.copy2(DRIVE_KEY, LOCAL_KEY)
os.chmod(LOCAL_KEY, 0o600)

if oct(LOCAL_KEY.stat().st_mode & 0o777) != '0o600':
    raise RuntimeError('SSH key permissions must be 0o600')

def run_remote(command, timeout=300, show_output=True):
    result = subprocess.run(
        ['bash', str(REMOTE_EXEC), command],
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    if show_output and result.stdout:
        print(result.stdout)
    if show_output and result.stderr:
        print('REMOTE STDERR:')
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f'Remote command failed with return code {result.returncode}.'
        )
    return result

print('Google Drive: MOUNTED')
print('Remote helper:', REMOTE_EXEC)
print('SSH key:', LOCAL_KEY)
print('CELL 1: SUCCESS')


Mounted at /content/drive
Google Drive: MOUNTED
Remote helper: /content/drive/MyDrive/remote_exec.sh
SSH key: /content/hadoop-hdfs.pem
CELL 1: SUCCESS


In [2]:
# CELL 2: PROJECT-WIDE CONSTANTS AND CITY MANIFEST
# Extents are intentionally not guessed. They must be populated by the
# standardized extent-discovery stage and then frozen before processing.

HISTORICAL_EPOCHS = [1975, 1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020]
PROOF_EPOCHS = [1975, 2000, 2020]
REMAINING_EPOCHS = [1980, 1985, 1990, 1995, 2005, 2010, 2015]
PRODUCTS = ['population', 'smod', 'built_surface']

HDFS_ROOT = 'hdfs://localhost:9000'
HDFS_CATALOG = f'{HDFS_ROOT}/data/ghsl/metadata/india/geotiff_catalog'
HDFS_CITY_ROOT = f'{HDFS_ROOT}/data/ghsl/curated/cities'
REMOTE_WORK_ROOT = '/home/ubuntu/ghsl_multicity'

DEFAULT_SPARK = {
    'population_driver_memory': '5g',
    'smod_driver_memory': '3g',
    'built_driver_memory': '3g',
    'enrichment_driver_memory': '2g',
    'analytics_driver_memory': '3g',
    'shuffle_partitions': 4,
    'default_parallelism': 2,
}

CITY_CONFIGS = {
    'hyderabad': {
        'city_name': 'Hyderabad Metropolitan Region',
        'model_role': 'training',
        'processing_order': 1,
        'seed_lon': 78.4867,
        'seed_lat': 17.3850,
    },
    'pune_metro': {
        'city_name': 'Pune Metropolitan Region',
        'model_role': 'training',
        'processing_order': 2,
        'seed_lon': 73.8567,
        'seed_lat': 18.5204,
    },
    'chennai_metro': {
        'city_name': 'Chennai Metropolitan Region',
        'model_role': 'training',
        'processing_order': 3,
        'seed_lon': 80.2707,
        'seed_lat': 13.0827,
    },
    'mumbai_metro': {
        'city_name': 'Mumbai Metropolitan Region',
        'model_role': 'training',
        'processing_order': 4,
        'seed_lon': 72.8777,
        'seed_lat': 19.0760,
    },
    'delhi_ncr': {
        'city_name': 'Delhi NCR',
        'model_role': 'training',
        'processing_order': 5,
        'seed_lon': 77.2090,
        'seed_lat': 28.6139,
    },
    'ahmedabad': {
        'city_name': 'Ahmedabad Metropolitan Region',
        'model_role': 'training',
        'processing_order': 6,
        'seed_lon': 72.5714,
        'seed_lat': 23.0225,
    },
    'surat': {
        'city_name': 'Surat Metropolitan Region',
        'model_role': 'training',
        'processing_order': 7,
        'seed_lon': 72.8311,
        'seed_lat': 21.1702,
    },
    'jaipur': {
        'city_name': 'Jaipur Metropolitan Region',
        'model_role': 'held_out',
        'processing_order': 8,
        'seed_lon': 75.7873,
        'seed_lat': 26.9124,
    },
}

for slug, config in CITY_CONFIGS.items():
    config.update({
        'city_slug': slug,
        'extent_method': '2020_urban_footprint_plus_30km_buffer_snapped_to_1km',
        'buffer_m': 30000,
        'target_crs': 'ESRI:54009',
        'grid_resolution_m': 1000,
        'extent_mollweide': None,
        'expected_grid_cells': None,
        'hdfs_output_base': f'{HDFS_CITY_ROOT}/city={slug}',
        'remote_work_dir': f'{REMOTE_WORK_ROOT}/{slug}',
        'spark': dict(DEFAULT_SPARK),
    })

ACTIVE_CITY_SLUG = 'hyderabad'
ACTIVE_CITY = CITY_CONFIGS[ACTIVE_CITY_SLUG]

print('Active city:', ACTIVE_CITY['city_name'])
print('Role:', ACTIVE_CITY['model_role'])
print('Output base:', ACTIVE_CITY['hdfs_output_base'])
print('Extent frozen:', ACTIVE_CITY['extent_mollweide'] is not None)
print('CELL 2: SUCCESS')


Active city: Hyderabad Metropolitan Region
Role: training
Output base: hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad
Extent frozen: False
CELL 2: SUCCESS


In [ ]:
# CELL 3: VALIDATE CITY CONFIGURATION WITHOUT PROCESSING
required_keys = {
    'city_name', 'city_slug', 'model_role', 'processing_order',
    'seed_lon', 'seed_lat', 'extent_method', 'buffer_m',
    'target_crs', 'grid_resolution_m', 'extent_mollweide',
    'expected_grid_cells', 'hdfs_output_base', 'remote_work_dir', 'spark'
}

missing = sorted(required_keys - set(ACTIVE_CITY))
if missing:
    raise RuntimeError(f'Missing configuration keys: {missing}')

if ACTIVE_CITY['model_role'] not in {'training', 'held_out'}:
    raise RuntimeError('model_role must be training or held_out')
if ACTIVE_CITY['grid_resolution_m'] != 1000:
    raise RuntimeError('This pipeline expects a 1 km canonical grid')
if ACTIVE_CITY['buffer_m'] <= 0:
    raise RuntimeError('buffer_m must be positive')

print('Configuration structure: PASS')
print('Historical epochs:', HISTORICAL_EPOCHS)
print('Proof epochs:', PROOF_EPOCHS)
print('Remaining epochs:', REMAINING_EPOCHS)
print('No Spark processing executed.')
print('CELL 3: SUCCESS')


Configuration structure: PASS
Historical epochs: [1975, 1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020]
Proof epochs: [1975, 2000, 2020]
Remaining epochs: [1980, 1985, 1990, 1995, 2005, 2010, 2015]
No Spark processing executed.
CELL 3: SUCCESS


In [3]:
# ============================================================
# CELL 3B: FIXED, COMPARABLE MULTI-CITY EXTENTS
#
# This cell:
#   - Selects the active city
#   - Converts its WGS84 seed to Mollweide
#   - Creates a fixed 70 km x 70 km extent
#   - Snaps the extent to the global 1 km grid
#   - Sets 4,900 expected grid cells
#
# No Spark job is launched.
# No HDFS data is modified.
# ============================================================

import json
import math
import subprocess
import sys

# Install pyproj only if it is unavailable.
try:
    from pyproj import Transformer
except ImportError:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "pyproj",
        ],
        check=True,
    )

    from pyproj import Transformer


# ============================================================
# ACTIVE CITY
#
# Change only this value after the current city is complete.
# Processing order:
#   hyderabad
#   pune_metro
#   chennai_metro
#   mumbai_metro
#   delhi_ncr
#   ahmedabad
#   surat
# ============================================================

ACTIVE_CITY_SLUG = "hyderabad"


# ============================================================
# STANDARDIZED ANALYSIS EXTENT
#
# 70 km x 70 km at 1 km resolution:
#
#   70 columns x 70 rows = 4,900 grid cells
#
# The extent is fixed across all epochs.
# ============================================================

SIDE_M = 70_000
GRID_RESOLUTION_M = 1_000

GRID_COLUMNS = SIDE_M // GRID_RESOLUTION_M
GRID_ROWS = SIDE_M // GRID_RESOLUTION_M

EXPECTED_GRID_CELLS = (
    GRID_COLUMNS * GRID_ROWS
)

if EXPECTED_GRID_CELLS != 4_900:
    raise RuntimeError(
        "Expected a 70 x 70 grid containing "
        f"4,900 cells, calculated "
        f"{EXPECTED_GRID_CELLS}."
    )


# ============================================================
# WGS84 TO MOLLWEIDE TRANSFORMATION
# ============================================================

transformer = Transformer.from_crs(
    "EPSG:4326",
    "ESRI:54009",
    always_xy=True,
)


# ============================================================
# BUILD EXTENTS FOR ALL CONFIGURED CITIES
# ============================================================

for city_slug, city_config in CITY_CONFIGS.items():

    seed_lon = float(
        city_config["seed_lon"]
    )

    seed_lat = float(
        city_config["seed_lat"]
    )

    centre_x, centre_y = transformer.transform(
        seed_lon,
        seed_lat,
    )

    if not (
        math.isfinite(centre_x)
        and math.isfinite(centre_y)
    ):
        raise RuntimeError(
            f"Invalid Mollweide coordinates for "
            f"{city_slug}: "
            f"x={centre_x}, y={centre_y}"
        )

    half_side_m = SIDE_M / 2.0

    raw_min_x = centre_x - half_side_m
    raw_min_y = centre_y - half_side_m

    # Snap the lower-left corner downward to the 1 km grid.
    min_x = (
        math.floor(
            raw_min_x / GRID_RESOLUTION_M
        )
        * GRID_RESOLUTION_M
    )

    min_y = (
        math.floor(
            raw_min_y / GRID_RESOLUTION_M
        )
        * GRID_RESOLUTION_M
    )

    # Retain an exact 70 km width and height.
    max_x = min_x + SIDE_M
    max_y = min_y + SIDE_M

    width_m = max_x - min_x
    height_m = max_y - min_y

    derived_columns = int(
        width_m / GRID_RESOLUTION_M
    )

    derived_rows = int(
        height_m / GRID_RESOLUTION_M
    )

    derived_grid_cells = (
        derived_columns * derived_rows
    )

    if derived_grid_cells != EXPECTED_GRID_CELLS:
        raise RuntimeError(
            f"{city_slug}: derived grid contains "
            f"{derived_grid_cells} cells; expected "
            f"{EXPECTED_GRID_CELLS}."
        )

    city_config["extent_mollweide"] = {
        "min_x": float(min_x),
        "min_y": float(min_y),
        "max_x": float(max_x),
        "max_y": float(max_y),
    }

    city_config["expected_grid_cells"] = int(
        EXPECTED_GRID_CELLS
    )

    city_config[
        "expected_complete_cells"
    ] = int(EXPECTED_GRID_CELLS)

    city_config[
        "expected_partial_cells"
    ] = 0

    city_config["grid_columns"] = int(
        derived_columns
    )

    city_config["grid_rows"] = int(
        derived_rows
    )

    city_config["analysis_width_m"] = float(
        width_m
    )

    city_config["analysis_height_m"] = float(
        height_m
    )


# ============================================================
# ACTIVATE AND VALIDATE THE SELECTED CITY
# ============================================================

if ACTIVE_CITY_SLUG not in CITY_CONFIGS:
    raise KeyError(
        f"Unknown city slug: {ACTIVE_CITY_SLUG}. "
        f"Available cities: "
        f"{sorted(CITY_CONFIGS.keys())}"
    )

ACTIVE_CITY = CITY_CONFIGS[
    ACTIVE_CITY_SLUG
]

extent = ACTIVE_CITY[
    "extent_mollweide"
]

required_extent_keys = {
    "min_x",
    "min_y",
    "max_x",
    "max_y",
}

if set(extent.keys()) != required_extent_keys:
    raise RuntimeError(
        "Extent keys are incorrect: "
        f"{sorted(extent.keys())}"
    )

if not (
    extent["min_x"] < extent["max_x"]
    and extent["min_y"] < extent["max_y"]
):
    raise RuntimeError(
        "The calculated city extent is invalid."
    )

if (
    ACTIVE_CITY["expected_grid_cells"]
    != EXPECTED_GRID_CELLS
):
    raise RuntimeError(
        "Active-city expected grid-cell count "
        "does not match the standardized grid."
    )


# ============================================================
# FINAL REPORT
# ============================================================

print("=" * 80)
print("ACTIVE MULTI-CITY CONFIGURATION")
print("=" * 80)

print(
    "City:",
    ACTIVE_CITY["city_name"],
)

print(
    "City slug:",
    ACTIVE_CITY["city_slug"],
)

print(
    "Model role:",
    ACTIVE_CITY["model_role"],
)

print(
    "Seed longitude:",
    ACTIVE_CITY["seed_lon"],
)

print(
    "Seed latitude:",
    ACTIVE_CITY["seed_lat"],
)

print(
    "Target CRS:",
    ACTIVE_CITY["target_crs"],
)

print(
    "Grid resolution:",
    f"{GRID_RESOLUTION_M} m",
)

print(
    "Grid dimensions:",
    f"{GRID_COLUMNS} columns x "
    f"{GRID_ROWS} rows",
)

print(
    "Expected grid cells:",
    ACTIVE_CITY["expected_grid_cells"],
)

print(
    "Mollweide extent:",
    json.dumps(
        extent,
        indent=2,
    ),
)

print(
    "HDFS output base:",
    ACTIVE_CITY["hdfs_output_base"],
)

print(
    "Remote work directory:",
    ACTIVE_CITY["remote_work_dir"],
)

print()
print("Configuration extent: FROZEN")
print("No Spark processing executed.")
print("No HDFS data modified.")
print("CELL 3B: SUCCESS")

ACTIVE MULTI-CITY CONFIGURATION
City: Hyderabad Metropolitan Region
City slug: hyderabad
Model role: training
Seed longitude: 78.4867
Seed latitude: 17.385
Target CRS: ESRI:54009
Grid resolution: 1000 m
Grid dimensions: 70 columns x 70 rows
Expected grid cells: 4900
Mollweide extent: {
  "min_x": 7607000.0,
  "min_y": 2101000.0,
  "max_x": 7677000.0,
  "max_y": 2171000.0
}
HDFS output base: hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad
Remote work directory: /home/ubuntu/ghsl_multicity/hyderabad

Configuration extent: FROZEN
No Spark processing executed.
No HDFS data modified.
CELL 3B: SUCCESS


Hyderabad configuration is now frozen in the current Colab runtime.

CRS:                 ESRI:54009
Resolution:          1,000 m
Dimensions:          70 × 70
Expected cells:      4,900

min_x:               7,607,000
min_y:               2,101,000
max_x:               7,677,000
max_y:               2,171,000

The extent is exactly:

Plain Text
70,000 m × 70,000 m
= 4,900 one-square-kilometre cells

In [ ]:
# CELL 4: PREFLIGHT FINAL PROVEN ARTIFACTS
cmd="""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop
export PATH=${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}
for d in NameNode DataNode SecondaryNameNode; do jps|awk '{print $2}'|grep -qx $d; done
hdfs dfsadmin -safemode get|grep -qi 'Safe mode is OFF'
test -f /home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py
for f in population_epoch_template.py smod_epoch_template.py built_surface_epoch_template.py; do test -f /home/ubuntu/ghsl_notebook05_templates/$f; done
echo PREFLIGHT_SUCCESS
"""
run_remote(cmd,timeout=300)
print('CELL 4: SUCCESS')


PREFLIGHT_SUCCESS

REMOTE STDERR:

CELL 4: SUCCESS


In [ ]:
# ============================================================
# CELL 5: PARAMETERIZE THE PROVEN BENGALURU DRIVER
#
# Creates city-specific copies for Hyderabad.
#
# Does not:
#   - modify the Bengaluru source files
#   - start Spark
#   - write anything to HDFS
# ============================================================

import base64
import hashlib
import json
import textwrap


# ------------------------------------------------------------
# Validate required notebook state.
# ------------------------------------------------------------

required_variables = [
    "ACTIVE_CITY",
    "ACTIVE_CITY_SLUG",
    "run_remote",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Missing notebook variables: "
        + ", ".join(missing_variables)
        + ". Rerun Cells 1, 2, and 3B."
    )


required_city_fields = {
    "city_name",
    "city_slug",
    "extent_mollweide",
    "expected_grid_cells",
    "expected_complete_cells",
    "expected_partial_cells",
    "remote_work_dir",
}

missing_city_fields = sorted(
    required_city_fields
    - set(ACTIVE_CITY.keys())
)

if missing_city_fields:
    raise RuntimeError(
        "Active-city configuration is missing: "
        + ", ".join(missing_city_fields)
    )


# ------------------------------------------------------------
# Encode the frozen Hyderabad configuration.
# ------------------------------------------------------------

config_json = json.dumps(
    ACTIVE_CITY,
    sort_keys=True,
)

config_b64 = base64.b64encode(
    config_json.encode("utf-8")
).decode("ascii")


# ------------------------------------------------------------
# Build the remote preparation program.
#
# Important:
# This source uses a normal triple-quoted string rather than
# a nested f-triple-quoted source block.
# ------------------------------------------------------------

remote_lines = [
    "from pathlib import Path",
    "import base64",
    "import json",
    "import py_compile",
    "import re",
    "",
    (
        "config = json.loads("
        "base64.b64decode("
        + repr(config_b64)
        + ").decode('utf-8'))"
    ),
    "",
    "city_slug = config['city_slug']",
    "extent = config['extent_mollweide']",
    (
        "expected_rows = "
        "int(config['expected_grid_cells'])"
    ),
    (
        "expected_complete = "
        "int(config['expected_complete_cells'])"
    ),
    (
        "expected_partial = "
        "int(config['expected_partial_cells'])"
    ),
    "",
    (
        "source_template_root = "
        "Path('/home/ubuntu/"
        "ghsl_notebook05_templates')"
    ),
    (
        "source_driver = "
        "Path('/home/ubuntu/"
        "ghsl_notebook05_batch/"
        "run_intermediate_epochs.py')"
    ),
    "",
    (
        "city_root = "
        "Path(config['remote_work_dir'])"
    ),
    "template_root = city_root / 'templates'",
    "program_root = city_root / 'programs'",
    "batch_root = city_root / 'batch'",
    "",
    (
        "for directory in "
        "[city_root, template_root, "
        "program_root, batch_root]:"
    ),
    "    directory.mkdir(",
    "        parents=True,",
    "        exist_ok=True,",
    "    )",
    "",
    "if not source_driver.is_file():",
    "    raise FileNotFoundError(source_driver)",
    "",
    "template_names = [",
    "    'population_epoch_template.py',",
    "    'smod_epoch_template.py',",
    "    'built_surface_epoch_template.py',",
    "]",
    "",
    "for template_name in template_names:",
    (
        "    source_path = "
        "source_template_root / template_name"
    ),
    "    if not source_path.is_file():",
    "        raise FileNotFoundError(source_path)",
    "",
    "replacement_pairs = [",
    "    (",
    (
        "        '/data/ghsl/curated/"
        "pilots/bengaluru',"
    ),
    (
        "        '/data/ghsl/curated/"
        "cities/city=' + city_slug,"
    ),
    "    ),",
    "    ('bengaluru', city_slug),",
    "    (",
    "        '7624177.350136818',",
    "        str(extent['min_x']),",
    "    ),",
    "    (",
    "        '1571479.656526347',",
    "        str(extent['min_y']),",
    "    ),",
    "    (",
    "        '7683017.20414696',",
    "        str(extent['max_x']),",
    "    ),",
    "    (",
    "        '1632690.7573197454',",
    "        str(extent['max_y']),",
    "    ),",
    "    ('3658', str(expected_rows)),",
    "    ('3480', str(expected_complete)),",
    "    ('178', str(expected_partial)),",
    "]",
    "",
    "def patch_text(text):",
    "    result = text",
    "    for old, new in replacement_pairs:",
    "        result = result.replace(old, new)",
    "    return result",
    "",
    "# Create city-specific templates.",
    "for template_name in template_names:",
    (
        "    source_path = "
        "source_template_root / template_name"
    ),
    (
        "    destination_path = "
        "template_root / template_name"
    ),
    "    text = source_path.read_text(",
    "        encoding='utf-8',",
    "    )",
    "    text = patch_text(text)",
    "    compile(",
    "        text,",
    "        str(destination_path),",
    "        'exec',",
    "    )",
    "    destination_path.write_text(",
    "        text,",
    "        encoding='utf-8',",
    "    )",
    "    py_compile.compile(",
    "        str(destination_path),",
    "        doraise=True,",
    "    )",
    "",
    "# Create city-specific historical driver.",
    "driver_text = source_driver.read_text(",
    "    encoding='utf-8',",
    ")",
    "",
    "# Replace intermediate-only list with all ten epochs.",
    "epoch_pattern = (",
    "    r'EPOCHS\\s*=\\s*\\['",
    "    r'.*?'",
    "    r'\\]\\s*'",
    ")",
    "",
    "epoch_replacement = (",
    "    'EPOCHS = [\\n'",
    "    '    1975,\\n'",
    "    '    1980,\\n'",
    "    '    1985,\\n'",
    "    '    1990,\\n'",
    "    '    1995,\\n'",
    "    '    2000,\\n'",
    "    '    2005,\\n'",
    "    '    2010,\\n'",
    "    '    2015,\\n'",
    "    '    2020,\\n'",
    "    ']\\n\\n'",
    ")",
    "",
    "driver_text, replacement_count = re.subn(",
    "    epoch_pattern,",
    "    epoch_replacement,",
    "    driver_text,",
    "    count=1,",
    "    flags=re.DOTALL,",
    ")",
    "",
    "if replacement_count != 1:",
    "    raise RuntimeError(",
    (
        "        'Could not uniquely replace "
        "the epoch list.'"
    ),
    "    )",
    "",
    "driver_text = patch_text(driver_text)",
    "",
    "driver_text = driver_text.replace(",
    (
        "    '/home/ubuntu/"
        "ghsl_notebook05_templates',"
    ),
    "    str(template_root),",
    ")",
    "",
    "driver_text = driver_text.replace(",
    (
        "    '/home/ubuntu/"
        "ghsl_notebook05_programs',"
    ),
    "    str(program_root),",
    ")",
    "",
    "driver_text = driver_text.replace(",
    (
        "    '/home/ubuntu/"
        "ghsl_notebook05_batch',"
    ),
    "    str(batch_root),",
    ")",
    "",
    "city_driver = (",
    "    batch_root / 'run_city_epochs.py'",
    ")",
    "",
    "compile(",
    "    driver_text,",
    "    str(city_driver),",
    "    'exec',",
    ")",
    "",
    "city_driver.write_text(",
    "    driver_text,",
    "    encoding='utf-8',",
    ")",
    "",
    "py_compile.compile(",
    "    str(city_driver),",
    "    doraise=True,",
    ")",
    "",
    "config_path = city_root / 'city_config.json'",
    "config_path.write_text(",
    "    json.dumps(",
    "        config,",
    "        indent=2,",
    "        sort_keys=True,",
    "    ) + '\\n',",
    "    encoding='utf-8',",
    ")",
    "",
    "# Final static checks.",
    "required_fragments = [",
    "    'EPOCHS = [',",
    "    '1975',",
    "    '2020',",
    "    'EXPECTED_ROWS = 4900',",
    (
        "    '/data/ghsl/curated/"
        "cities/city=' + city_slug,"
    ),
    "    str(template_root),",
    "    str(program_root),",
    "    str(batch_root),",
    "]",
    "",
    "missing_fragments = [",
    "    fragment",
    "    for fragment in required_fragments",
    "    if fragment not in driver_text",
    "]",
    "",
    "if missing_fragments:",
    "    raise RuntimeError(",
    (
        "        'Generated driver is missing: '"
    ),
    "        + repr(missing_fragments)",
    "    )",
    "",
    "print('=' * 80)",
    "print('CITY DRIVER PREPARATION')",
    "print('=' * 80)",
    "print('City:', config['city_name'])",
    "print('City slug:', city_slug)",
    "print('Expected rows:', expected_rows)",
    "print('Template root:', template_root)",
    "print('Program root:', program_root)",
    "print('Batch root:', batch_root)",
    "print('Driver:', city_driver)",
    "print('Config:', config_path)",
    "print()",
    "print('Remote compilation: PASS')",
    "print('Original Bengaluru files modified: NO')",
    "print('Spark launched: NO')",
    "print('HDFS modified: NO')",
    "print('PREPARE CITY DRIVER: SUCCESS')",
]

remote_program = "\n".join(
    remote_lines
) + "\n"


# ============================================================
# COMPILE LOCALLY BEFORE TRANSFER
# ============================================================

compile(
    remote_program,
    "prepare_city_driver.py",
    "exec",
)

local_sha256 = hashlib.sha256(
    remote_program.encode("utf-8")
).hexdigest()

print(
    "LOCAL PREPARATION COMPILE: PASS"
)

print(
    "Expected SHA-256:",
    local_sha256,
)


# ============================================================
# BASE64 TRANSFER
# ============================================================

remote_program_b64 = base64.b64encode(
    remote_program.encode("utf-8")
).decode("ascii")

remote_command = f"""
set -e

PREPARER="/tmp/prepare_city_driver.py"
EXPECTED_SHA256="{local_sha256}"

echo "{remote_program_b64}" \
| base64 -d \
> "${{PREPARER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PREPARER}}" \
    | awk '{{print $1}}'
)"

echo "Remote SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Checksum mismatch."
    exit 20
fi

echo "REMOTE CHECKSUM: PASS"

python3 -m py_compile "${{PREPARER}}"

echo "REMOTE PREPARATION COMPILE: PASS"

python3 "${{PREPARER}}"
"""


# ============================================================
# RUN PREPARATION
# ============================================================

result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 5 RESULT")
print("=" * 80)

print(
    "Return code:",
    result.returncode,
)

print(
    "Generated driver:",
    (
        ACTIVE_CITY["remote_work_dir"]
        + "/batch/run_city_epochs.py"
    ),
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 5: SUCCESS")

LOCAL PREPARATION COMPILE: PASS
Expected SHA-256: 2eda3c7a7b29db876ada7b18440db68834462643072e7bfb397fe20b986ce7e4
Remote SHA-256: 2eda3c7a7b29db876ada7b18440db68834462643072e7bfb397fe20b986ce7e4
REMOTE CHECKSUM: PASS
REMOTE PREPARATION COMPILE: PASS
CITY DRIVER PREPARATION
City: Hyderabad Metropolitan Region
City slug: hyderabad
Expected rows: 4900
Template root: /home/ubuntu/ghsl_multicity/hyderabad/templates
Program root: /home/ubuntu/ghsl_multicity/hyderabad/programs
Batch root: /home/ubuntu/ghsl_multicity/hyderabad/batch
Driver: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py
Config: /home/ubuntu/ghsl_multicity/hyderabad/city_config.json

Remote compilation: PASS
Original Bengaluru files modified: NO
Spark launched: NO
HDFS modified: NO
PREPARE CITY DRIVER: SUCCESS

REMOTE STDERR:


CELL 5 RESULT
Return code: 0
Generated driver: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py
Spark launched: NO
HDFS modified: NO
CELL 5: SUCCESS


In [21]:
# CELL 6: STATIC VALIDATION
root=ACTIVE_CITY['remote_work_dir']; slug=ACTIVE_CITY_SLUG
cmd=f"""set -e
D='{root}/batch/run_city_epochs.py'
python3 -m py_compile "$D"
for f in {root}/templates/*.py; do python3 -m py_compile "$f"; done
grep -q 'EXPECTED_ROWS = 4900' "$D"
grep -q 'city={slug}' "$D"
echo STATIC_VALIDATION_SUCCESS
"""
run_remote(cmd,timeout=180)
print('CELL 6: SUCCESS')


STATIC_VALIDATION_SUCCESS

REMOTE STDERR:

CELL 6: SUCCESS


In [22]:
# CELL 7: DETACHED LAUNCH
root=ACTIVE_CITY['remote_work_dir']
cmd=f"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop
export PATH=${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}
PID='{root}/batch/city.pid'; LOG='{root}/batch/city.log'; D='{root}/batch/run_city_epochs.py'
if [ -f "$PID" ] && kill -0 $(cat "$PID") 2>/dev/null; then echo JOB_ALREADY_RUNNING; exit 30; fi
nohup python3 "$D" >"$LOG" 2>&1 </dev/null & echo $! >"$PID"
sleep 2; kill -0 $(cat "$PID")
echo PID=$(cat "$PID"); echo LOG=$LOG; echo DETACHED_LAUNCH_SUCCESS
"""
run_remote(cmd,timeout=60)
print('Safe to close Colab. Monitor with Cell 8.')


PID=122114
LOG=/home/ubuntu/ghsl_multicity/hyderabad/batch/city.log
DETACHED_LAUNCH_SUCCESS

REMOTE STDERR:

Safe to close Colab. Monitor with Cell 8.


In [ ]:
# ============================================================
# CELL 7A: DIAGNOSE FAILED DETACHED HYDERABAD LAUNCH
#
# Read-only:
#   - Does not relaunch the job
#   - Does not modify HDFS
#   - Does not modify the generated driver
# ============================================================

import shlex

root = ACTIVE_CITY["remote_work_dir"]

diagnostic_command = f"""
set +e

PID_FILE={shlex.quote(root + "/batch/city.pid")}
LOG_FILE={shlex.quote(root + "/batch/city.log")}
DRIVER={shlex.quote(root + "/batch/run_city_epochs.py")}
MANIFEST={shlex.quote(root + "/batch/batch_manifest.tsv")}

echo "============================================================"
echo "DETACHED PROCESS STATUS"
echo "============================================================"

if [ -f "$PID_FILE" ]; then
    PID="$(cat "$PID_FILE")"
    echo "PID file: $PID_FILE"
    echo "Recorded PID: $PID"

    if kill -0 "$PID" 2>/dev/null; then
        echo "Process status: RUNNING"
    else
        echo "Process status: STOPPED"
    fi
else
    echo "PID file: MISSING"
fi

echo
echo "============================================================"
echo "DRIVER STATUS"
echo "============================================================"

if [ -f "$DRIVER" ]; then
    echo "Driver exists: YES"
    ls -lh "$DRIVER"
    sha256sum "$DRIVER"

    echo
    echo "Python compilation:"
    python3 -m py_compile "$DRIVER"
    echo "Driver compilation: PASS"
else
    echo "Driver exists: NO"
fi

echo
echo "============================================================"
echo "COMPLETE DETACHED LOG"
echo "============================================================"

if [ -f "$LOG_FILE" ]; then
    echo "Log file: $LOG_FILE"
    echo "Log size:"
    ls -lh "$LOG_FILE"

    echo
    cat "$LOG_FILE"
else
    echo "Log file: MISSING"
fi

echo
echo "============================================================"
echo "BATCH MANIFEST"
echo "============================================================"

if [ -f "$MANIFEST" ]; then
    cat "$MANIFEST"
else
    echo "Manifest not created"
fi

echo
echo "============================================================"
echo "RECENT CITY FILES"
echo "============================================================"

find {shlex.quote(root)} \
    -maxdepth 3 \
    -type f \
    -printf '%TY-%Tm-%Td %TH:%TM:%TS %10s %p\\n' \
    2>/dev/null \
    | sort \
    | tail -40

echo
echo "============================================================"
echo "HYDERABAD HDFS OUTPUTS CREATED SO FAR"
echo "============================================================"

hdfs dfs -find \
    /data/ghsl/curated/cities/city=hyderabad \
    2>/dev/null \
    | sort \
    | tail -100

exit 0
"""

diagnostic_result = run_remote(
    diagnostic_command,
    timeout=300,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 7A RESULT")
print("=" * 80)
print("Diagnostic return code:", diagnostic_result.returncode)
print("No job was relaunched.")
print("No HDFS data was modified.")

DETACHED PROCESS STATUS
PID file: /home/ubuntu/ghsl_multicity/hyderabad/batch/city.pid
Recorded PID: 50698
Process status: STOPPED

DRIVER STATUS
Driver exists: YES
-rw-rw-r-- 1 ubuntu ubuntu 35K Nov  7 14:45 /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py
b6a2e7baeae96173991a5f028079965820422eef86c2f266f987b629f0d427c9  /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py

Python compilation:
Driver compilation: PASS

COMPLETE DETACHED LOG
Log file: /home/ubuntu/ghsl_multicity/hyderabad/batch/city.log
Log size:
-rw-rw-r-- 1 ubuntu ubuntu 481 Nov  7 14:48 /home/ubuntu/ghsl_multicity/hyderabad/batch/city.log

Traceback (most recent call last):
  File "/home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py", line 1468, in <module>
    verify_template(
  File "/home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py", line 270, in verify_template
    raise RuntimeError(
RuntimeError: population template checksum mismatch.
Expected: f764cd402f7dafe007

In [ ]:
# ============================================================
# CELL 7B REPLACEMENT:
# UPDATE HYDERABAD TEMPLATE CHECKSUMS DIRECTLY
#
# This cell:
#   - Computes actual Hyderabad template checksums
#   - Replaces the three exact Bengaluru checksum literals
#     inside the Hyderabad driver only
#   - Preserves the original Bengaluru driver and templates
#   - Creates a Hyderabad-driver backup
#   - Compiles and verifies the patched driver
#
# It does not:
#   - start Spark
#   - launch the city batch
#   - modify HDFS
# ============================================================

import base64
import hashlib
import textwrap


# ------------------------------------------------------------
# Validate notebook state.
# ------------------------------------------------------------

for required_name in [
    "ACTIVE_CITY",
    "ACTIVE_CITY_SLUG",
    "run_remote",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is unavailable. "
            "Rerun Cells 1, 2, and 3B."
        )


remote_root = ACTIVE_CITY[
    "remote_work_dir"
]

driver_path = (
    remote_root
    + "/batch/run_city_epochs.py"
)

template_root = (
    remote_root
    + "/templates"
)


# ------------------------------------------------------------
# Exact original Bengaluru checksums.
#
# These remain unchanged in the original Bengaluru driver.
# They are replaced only in the Hyderabad copy.
# ------------------------------------------------------------

bengaluru_checksums = {
    "population": (
        "f764cd402f7dafe00734c3f7afda7e947"
        "f597a46eb3e2d5618c4831f028b93aa"
    ),
    "smod": (
        "361fea01efcde8ffb3f5d85067749352"
        "f354e8ad1bef85f2b9c9e2bdb5dc085c"
    ),
    "built_surface": (
        "d21f49e979bec0d20dea8e4210738250"
        "9620a750fad89f77ce9331ea23dba87a"
    ),
}


# ------------------------------------------------------------
# Remote checksum patcher.
# ------------------------------------------------------------

remote_patcher = textwrap.dedent(
    f"""
    from pathlib import Path
    import hashlib
    import json
    import py_compile
    import shutil
    import time


    DRIVER = Path({driver_path!r})

    TEMPLATE_ROOT = Path({template_root!r})

    TEMPLATE_PATHS = {{
        "population": (
            TEMPLATE_ROOT
            / "population_epoch_template.py"
        ),
        "smod": (
            TEMPLATE_ROOT
            / "smod_epoch_template.py"
        ),
        "built_surface": (
            TEMPLATE_ROOT
            / "built_surface_epoch_template.py"
        ),
    }}

    BENGALURU_CHECKSUMS = {{
        "population": (
            "f764cd402f7dafe00734c3f7afda7e947"
            "f597a46eb3e2d5618c4831f028b93aa"
        ),
        "smod": (
            "361fea01efcde8ffb3f5d85067749352"
            "f354e8ad1bef85f2b9c9e2bdb5dc085c"
        ),
        "built_surface": (
            "d21f49e979bec0d20dea8e4210738250"
            "9620a750fad89f77ce9331ea23dba87a"
        ),
    }}


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            while True:
                chunk = handle.read(
                    1024 * 1024
                )

                if not chunk:
                    break

                digest.update(chunk)

        return digest.hexdigest()


    # --------------------------------------------------------
    # Validate Hyderabad files.
    # --------------------------------------------------------

    if not DRIVER.is_file():
        raise FileNotFoundError(
            f"Hyderabad driver missing: {{DRIVER}}"
        )

    for product, template_path in (
        TEMPLATE_PATHS.items()
    ):
        if not template_path.is_file():
            raise FileNotFoundError(
                f"{{product}} template missing: "
                f"{{template_path}}"
            )


    # --------------------------------------------------------
    # Calculate actual Hyderabad checksums.
    # --------------------------------------------------------

    hyderabad_checksums = {{
        product: sha256_file(template_path)
        for product, template_path
        in TEMPLATE_PATHS.items()
    }}


    # --------------------------------------------------------
    # Read and back up Hyderabad driver.
    # --------------------------------------------------------

    original_text = DRIVER.read_text(
        encoding="utf-8"
    )

    timestamp = time.strftime(
        "%Y%m%dT%H%M%SZ",
        time.gmtime(),
    )

    backup_path = DRIVER.with_name(
        DRIVER.name
        + ".before_hyderabad_checksum_patch."
        + timestamp
    )

    shutil.copy2(
        DRIVER,
        backup_path,
    )


    # --------------------------------------------------------
    # Replace exact checksum literals.
    #
    # This handles two cases:
    #   1. Original Bengaluru checksum still present
    #   2. Hyderabad checksum already present from a prior
    #      successful run
    # --------------------------------------------------------

    patched_text = original_text
    patch_report = {{}}

    for product in [
        "population",
        "smod",
        "built_surface",
    ]:
        old_checksum = (
            BENGALURU_CHECKSUMS[product]
        )

        new_checksum = (
            hyderabad_checksums[product]
        )

        old_count = patched_text.count(
            old_checksum
        )

        new_count_before = patched_text.count(
            new_checksum
        )

        if old_count == 1:
            patched_text = patched_text.replace(
                old_checksum,
                new_checksum,
                1,
            )

            action = (
                "replaced_bengaluru_checksum"
            )

        elif (
            old_count == 0
            and new_count_before >= 1
        ):
            action = (
                "hyderabad_checksum_already_present"
            )

        elif old_count == 0:
            raise RuntimeError(
                f"Could not locate the expected "
                f"Bengaluru checksum for {{product}}, "
                f"and the Hyderabad checksum is also "
                f"absent."
            )

        else:
            raise RuntimeError(
                f"Expected at most one Bengaluru "
                f"checksum occurrence for {{product}}, "
                f"found {{old_count}}."
            )

        patch_report[product] = {{
            "action": action,
            "old_occurrences": old_count,
            "hyderabad_checksum": new_checksum,
        }}


    # --------------------------------------------------------
    # Verify all three Hyderabad checksums now occur.
    # --------------------------------------------------------

    for product, checksum in (
        hyderabad_checksums.items()
    ):
        occurrence_count = (
            patched_text.count(checksum)
        )

        if occurrence_count < 1:
            raise RuntimeError(
                f"Hyderabad checksum for {{product}} "
                f"is missing after patch."
            )


    # --------------------------------------------------------
    # Compile in memory before writing.
    # --------------------------------------------------------

    compile(
        patched_text,
        str(DRIVER),
        "exec",
    )


    # --------------------------------------------------------
    # Save and compile patched Hyderabad driver.
    # --------------------------------------------------------

    DRIVER.write_text(
        patched_text,
        encoding="utf-8",
    )

    py_compile.compile(
        str(DRIVER),
        doraise=True,
    )


    # --------------------------------------------------------
    # Verify saved driver and templates.
    # --------------------------------------------------------

    saved_text = DRIVER.read_text(
        encoding="utf-8"
    )

    for product, template_path in (
        TEMPLATE_PATHS.items()
    ):
        actual_checksum = sha256_file(
            template_path
        )

        expected_checksum = (
            hyderabad_checksums[product]
        )

        if actual_checksum != expected_checksum:
            raise RuntimeError(
                f"Template changed during patch for "
                f"{{product}}."
            )

        if expected_checksum not in saved_text:
            raise RuntimeError(
                f"Saved driver does not contain "
                f"the Hyderabad checksum for "
                f"{{product}}."
            )


    # --------------------------------------------------------
    # Save city checksum manifest.
    # --------------------------------------------------------

    manifest_path = (
        DRIVER.parent
        / "template_checksums.json"
    )

    manifest_path.write_text(
        json.dumps(
            {{
                "city_slug": "hyderabad",
                "driver": str(DRIVER),
                "templates": {{
                    product: {{
                        "path": str(
                            TEMPLATE_PATHS[product]
                        ),
                        "sha256": checksum,
                    }}
                    for product, checksum
                    in hyderabad_checksums.items()
                }},
                "patch_report": patch_report,
                "patched_at_utc": (
                    time.strftime(
                        "%Y-%m-%dT%H:%M:%SZ",
                        time.gmtime(),
                    )
                ),
            }},
            indent=2,
            sort_keys=True,
        )
        + "\\n",
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # Final report.
    # --------------------------------------------------------

    print("=" * 80)
    print("HYDERABAD TEMPLATE CHECKSUM PATCH")
    print("=" * 80)

    print("Driver:", DRIVER)
    print("Backup:", backup_path)
    print("Manifest:", manifest_path)

    print()

    for product in [
        "population",
        "smod",
        "built_surface",
    ]:
        print(
            f"{{product}}:"
        )

        print(
            "  Action:",
            patch_report[product]["action"],
        )

        print(
            "  SHA-256:",
            hyderabad_checksums[product],
        )

    print()
    print("Driver compilation: PASS")
    print("Saved checksum verification: PASS")
    print("Original Bengaluru driver modified: NO")
    print("Original Bengaluru templates modified: NO")
    print("Spark launched: NO")
    print("HDFS modified: NO")
    print("HYDERABAD CHECKSUM PATCH: SUCCESS")
    """
).strip() + "\n"


# ------------------------------------------------------------
# Compile the exact patcher locally.
# ------------------------------------------------------------

compile(
    remote_patcher,
    "patch_hyderabad_checksums.py",
    "exec",
)

local_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print("LOCAL PATCHER COMPILE: PASS")
print("Expected patcher SHA-256:", local_sha256)


# ------------------------------------------------------------
# Transfer with Base64 and verify.
# ------------------------------------------------------------

encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")

remote_command = f"""
set -e

PATCHER="/tmp/patch_hyderabad_checksums.py"
EXPECTED_SHA256="{local_sha256}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}"
"""

patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 7B REPLACEMENT RESULT")
print("=" * 80)

print("Return code:", patch_result.returncode)
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 7B: SUCCESS")

LOCAL PATCHER COMPILE: PASS
Expected patcher SHA-256: 93f588679e2fe68216f6444c538c367f9ec5f46811d01dad28bce92f27725e75
Remote patcher SHA-256: 93f588679e2fe68216f6444c538c367f9ec5f46811d01dad28bce92f27725e75
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS

REMOTE STDERR:
Traceback (most recent call last):
  File "/tmp/patch_hyderabad_checksums.py", line 169, in <module>
    raise RuntimeError(
    ...<4 lines>...
    )
RuntimeError: Could not locate the expected Bengaluru checksum for population, and the Hyderabad checksum is also absent.



RuntimeError: Remote command failed with return code 1.

In [ ]:
# ============================================================
# CELL 7C: INSPECT ACTUAL CHECKSUM STRUCTURE IN CITY DRIVER
#
# Read-only:
#   - Does not patch any file
#   - Does not launch Spark
#   - Does not modify HDFS
# ============================================================

import shlex

driver_path = (
    ACTIVE_CITY["remote_work_dir"]
    + "/batch/run_city_epochs.py"
)

template_root = (
    ACTIVE_CITY["remote_work_dir"]
    + "/templates"
)

inspection_command = f"""
set -e

DRIVER={shlex.quote(driver_path)}
TEMPLATE_ROOT={shlex.quote(template_root)}

echo "============================================================"
echo "ACTUAL HYDERABAD TEMPLATE CHECKSUMS"
echo "============================================================"

sha256sum \
    "$TEMPLATE_ROOT/population_epoch_template.py" \
    "$TEMPLATE_ROOT/smod_epoch_template.py" \
    "$TEMPLATE_ROOT/built_surface_epoch_template.py"

echo
echo "============================================================"
echo "CHECKSUM-RELATED DRIVER LINES"
echo "============================================================"

grep -n -i \
    -E 'checksum|sha256|template_hash|template.*hash' \
    "$DRIVER" \
    || true

echo
echo "============================================================"
echo "VERIFY_TEMPLATE FUNCTION"
echo "============================================================"

START_LINE="$(
    grep -n \
        'def verify_template' \
        "$DRIVER" \
    | head -1 \
    | cut -d: -f1
)"

if [ -n "$START_LINE" ]; then
    END_LINE=$((START_LINE + 100))

    sed -n \
        "${{START_LINE}},${{END_LINE}}p" \
        "$DRIVER"
else
    echo "verify_template function not found"
fi

echo
echo "============================================================"
echo "VERIFY_TEMPLATE CALLS"
echo "============================================================"

grep -n -A12 -B8 \
    'verify_template(' \
    "$DRIVER" \
    || true

echo
echo "============================================================"
echo "FIRST 340 DRIVER LINES"
echo "============================================================"

sed -n '1,340p' "$DRIVER"

echo
echo "============================================================"
echo "CHECKSUM INSPECTION COMPLETE"
echo "============================================================"
"""

inspection_result = run_remote(
    inspection_command,
    timeout=300,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 7C RESULT")
print("=" * 80)

print(
    "Return code:",
    inspection_result.returncode,
)

print("Driver modified: NO")
print("Spark launched: NO")
print("HDFS modified: NO")

ACTUAL HYDERABAD TEMPLATE CHECKSUMS
d3d0b3c5a92180531258a3f13b176c8ed14e9babd8cdfdf88a2e57cab19856ef  /home/ubuntu/ghsl_multicity/hyderabad/templates/population_epoch_template.py
ccf9a02016143676776af1d10eb31e30b3c37377fa85952f76200f9d99b6325c  /home/ubuntu/ghsl_multicity/hyderabad/templates/smod_epoch_template.py
6f805892684e89f5646c482c10be6e7bd9066a678a74e35f1f9776617087c3cd  /home/ubuntu/ghsl_multicity/hyderabad/templates/built_surface_epoch_template.py

CHECKSUM-RELATED DRIVER LINES
72:EXPECTED_TEMPLATE_SHA256 = {
104:def sha256_file(path):
105:    digest = hashlib.sha256()
261:    actual_checksum = sha256_file(
265:    expected_checksum = (
266:        EXPECTED_TEMPLATE_SHA256[product]
269:    if actual_checksum != expected_checksum:
271:            f"{product} template checksum mismatch.\n"
272:            f"Expected: {expected_checksum}\n"
273:            f"Actual:   {actual_checksum}"
1473:print("Protected template checksums: PASS")

VERIFY_TEMPLATE FUNCTION
def verify_templat

In [ ]:
# ============================================================
# CELL 7D: REPLACE HYDERABAD CHECKSUM DICTIONARY EXACTLY
#
# Updates only:
#   /home/ubuntu/ghsl_multicity/hyderabad/batch/
#   run_city_epochs.py
#
# Does not:
#   - modify the original Bengaluru driver
#   - modify original Bengaluru templates
#   - start Spark
#   - modify HDFS
# ============================================================

import base64
import hashlib
import textwrap


# ------------------------------------------------------------
# Confirm notebook state.
# ------------------------------------------------------------

for required_name in [
    "ACTIVE_CITY",
    "ACTIVE_CITY_SLUG",
    "run_remote",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is unavailable. "
            "Rerun Cells 1, 2, and 3B."
        )


if ACTIVE_CITY_SLUG != "hyderabad":
    raise RuntimeError(
        "This patch is specifically for the current "
        "Hyderabad driver."
    )


driver_path = (
    ACTIVE_CITY["remote_work_dir"]
    + "/batch/run_city_epochs.py"
)

template_root = (
    ACTIVE_CITY["remote_work_dir"]
    + "/templates"
)


# ------------------------------------------------------------
# Checksums measured from the actual Hyderabad templates.
# ------------------------------------------------------------

expected_hyderabad_checksums = {
    "population": (
        "d3d0b3c5a92180531258a3f13b176c8e"
        "d14e9babd8cdfdf88a2e57cab19856ef"
    ),
    "smod": (
        "ccf9a02016143676776af1d10eb31e30"
        "b3c37377fa85952f76200f9d99b6325c"
    ),
    "built_surface": (
        "6f805892684e89f5646c482c10be6e7b"
        "d9066a678a74e35f1f9776617087c3cd"
    ),
}


# ------------------------------------------------------------
# Remote patch program.
# ------------------------------------------------------------

remote_patcher = textwrap.dedent(
    f'''
    from pathlib import Path
    import hashlib
    import json
    import py_compile
    import re
    import shutil
    import time


    DRIVER = Path({driver_path!r})

    TEMPLATE_ROOT = Path({template_root!r})

    TEMPLATE_PATHS = {{
        "population": (
            TEMPLATE_ROOT
            / "population_epoch_template.py"
        ),
        "smod": (
            TEMPLATE_ROOT
            / "smod_epoch_template.py"
        ),
        "built_surface": (
            TEMPLATE_ROOT
            / "built_surface_epoch_template.py"
        ),
    }}

    EXPECTED_HYDERABAD_CHECKSUMS = {{
        "population": (
            "d3d0b3c5a92180531258a3f13b176c8e"
            "d14e9babd8cdfdf88a2e57cab19856ef"
        ),
        "smod": (
            "ccf9a02016143676776af1d10eb31e30"
            "b3c37377fa85952f76200f9d99b6325c"
        ),
        "built_surface": (
            "6f805892684e89f5646c482c10be6e7b"
            "d9066a678a74e35f1f9776617087c3cd"
        ),
    }}


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(1024 * 1024),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    # --------------------------------------------------------
    # Validate files and measured checksums first.
    # --------------------------------------------------------

    if not DRIVER.is_file():
        raise FileNotFoundError(
            f"Driver does not exist: {{DRIVER}}"
        )

    measured_checksums = {{}}

    for product, template_path in (
        TEMPLATE_PATHS.items()
    ):
        if not template_path.is_file():
            raise FileNotFoundError(
                f"Missing {{product}} template: "
                f"{{template_path}}"
            )

        measured_checksum = sha256_file(
            template_path
        )

        measured_checksums[product] = (
            measured_checksum
        )

        expected_checksum = (
            EXPECTED_HYDERABAD_CHECKSUMS[
                product
            ]
        )

        if measured_checksum != expected_checksum:
            raise RuntimeError(
                f"{{product}} template checksum "
                f"changed unexpectedly.\\n"
                f"Expected: {{expected_checksum}}\\n"
                f"Actual:   {{measured_checksum}}"
            )


    # --------------------------------------------------------
    # Read and back up only the Hyderabad driver.
    # --------------------------------------------------------

    original_text = DRIVER.read_text(
        encoding="utf-8"
    )

    timestamp = time.strftime(
        "%Y%m%dT%H%M%SZ",
        time.gmtime(),
    )

    backup_path = DRIVER.with_name(
        DRIVER.name
        + ".before_checksum_dictionary_patch."
        + timestamp
    )

    shutil.copy2(
        DRIVER,
        backup_path,
    )


    # --------------------------------------------------------
    # Construct the replacement dictionary.
    # --------------------------------------------------------

    replacement_dictionary = """EXPECTED_TEMPLATE_SHA256 = {{
        "population": (
            "d3d0b3c5a92180531258a3f13b176c8e"
            "d14e9babd8cdfdf88a2e57cab19856ef"
        ),
        "smod": (
            "ccf9a02016143676776af1d10eb31e30"
            "b3c37377fa85952f76200f9d99b6325c"
        ),
        "built_surface": (
            "6f805892684e89f5646c482c10be6e7b"
            "d9066a678a74e35f1f9776617087c3cd"
        ),
    }}
    """


    # --------------------------------------------------------
    # Replace precisely from the checksum dictionary through
    # the line immediately before SEDONA_PACKAGES.
    # --------------------------------------------------------

    dictionary_pattern = (
        r"EXPECTED_TEMPLATE_SHA256\\s*=\\s*\\{{"
        r".*?"
        r"\\}}\\s*"
        r"(?=SEDONA_PACKAGES\\s*=)"
    )

    patched_text, replacement_count = (
        re.subn(
            dictionary_pattern,
            replacement_dictionary,
            original_text,
            count=1,
            flags=re.DOTALL,
        )
    )

    if replacement_count != 1:
        raise RuntimeError(
            "Expected exactly one checksum dictionary "
            f"replacement, found {{replacement_count}}."
        )


    # --------------------------------------------------------
    # Compile before writing.
    # --------------------------------------------------------

    compile(
        patched_text,
        str(DRIVER),
        "exec",
    )


    # --------------------------------------------------------
    # Write and compile the patched Hyderabad driver.
    # --------------------------------------------------------

    DRIVER.write_text(
        patched_text,
        encoding="utf-8",
    )

    py_compile.compile(
        str(DRIVER),
        doraise=True,
    )


    # --------------------------------------------------------
    # Execute only the driver's checksum dictionary safely.
    #
    # Parse the saved source with AST rather than importing
    # the driver, because importing would start the batch.
    # --------------------------------------------------------

    import ast

    saved_text = DRIVER.read_text(
        encoding="utf-8"
    )

    parsed = ast.parse(saved_text)

    saved_dictionary = None

    for node in parsed.body:
        if not isinstance(node, ast.Assign):
            continue

        for target in node.targets:
            if (
                isinstance(target, ast.Name)
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
            ):
                saved_dictionary = (
                    ast.literal_eval(node.value)
                )

    if saved_dictionary is None:
        raise RuntimeError(
            "Saved checksum dictionary was not found."
        )

    if (
        saved_dictionary
        != EXPECTED_HYDERABAD_CHECKSUMS
    ):
        raise RuntimeError(
            "Saved checksum dictionary does not match "
            "the measured Hyderabad checksums."
        )


    # --------------------------------------------------------
    # Final independent verification.
    # --------------------------------------------------------

    for product, template_path in (
        TEMPLATE_PATHS.items()
    ):
        actual_checksum = sha256_file(
            template_path
        )

        driver_checksum = (
            saved_dictionary[product]
        )

        if actual_checksum != driver_checksum:
            raise RuntimeError(
                f"Final checksum mismatch for "
                f"{{product}}.\\n"
                f"Driver:   {{driver_checksum}}\\n"
                f"Template: {{actual_checksum}}"
            )


    # --------------------------------------------------------
    # Write separate Hyderabad checksum manifest.
    # --------------------------------------------------------

    manifest_path = (
        DRIVER.parent
        / "template_checksums.json"
    )

    manifest_path.write_text(
        json.dumps(
            {{
                "city_slug": "hyderabad",
                "driver": str(DRIVER),
                "driver_sha256": sha256_file(
                    DRIVER
                ),
                "templates": {{
                    product: {{
                        "path": str(
                            TEMPLATE_PATHS[product]
                        ),
                        "sha256": checksum,
                    }}
                    for product, checksum
                    in measured_checksums.items()
                }},
                "backup": str(backup_path),
                "patched_at_utc": (
                    time.strftime(
                        "%Y-%m-%dT%H:%M:%SZ",
                        time.gmtime(),
                    )
                ),
            }},
            indent=2,
            sort_keys=True,
        )
        + "\\n",
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # Report.
    # --------------------------------------------------------

    print("=" * 80)
    print("HYDERABAD CHECKSUM DICTIONARY PATCH")
    print("=" * 80)

    print("Driver:", DRIVER)
    print("Backup:", backup_path)
    print("Manifest:", manifest_path)

    print()

    for product in [
        "population",
        "smod",
        "built_surface",
    ]:
        print(
            f"{{product}}:",
            saved_dictionary[product],
        )

    print()
    print("Dictionary replacement count: 1")
    print("Driver compilation: PASS")
    print("AST dictionary validation: PASS")
    print("Template checksum verification: PASS")
    print("Original Bengaluru driver modified: NO")
    print("Original Bengaluru templates modified: NO")
    print("Spark launched: NO")
    print("HDFS modified: NO")
    print("HYDERABAD CHECKSUM DICTIONARY: SUCCESS")
    '''
).strip() + "\n"


# ------------------------------------------------------------
# Compile locally.
# ------------------------------------------------------------

compile(
    remote_patcher,
    "patch_hyderabad_checksum_dictionary.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print("LOCAL PATCHER COMPILE: PASS")
print("Expected patcher SHA-256:", patcher_sha256)


# ------------------------------------------------------------
# Transfer and run remotely.
# ------------------------------------------------------------

encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")

remote_command = f"""
set -e

PATCHER="/tmp/patch_hyderabad_checksum_dictionary.py"
EXPECTED_SHA256="{patcher_sha256}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}"
"""

patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 7D RESULT")
print("=" * 80)

print("Return code:", patch_result.returncode)
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 7D: SUCCESS")

LOCAL PATCHER COMPILE: PASS
Expected patcher SHA-256: ada40fadc54f214250f3f2fc2f5661df8db98ae8a619da82adb7a069c609a873
Remote patcher SHA-256: ada40fadc54f214250f3f2fc2f5661df8db98ae8a619da82adb7a069c609a873
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
HYDERABAD CHECKSUM DICTIONARY PATCH
Driver: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py
Backup: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py.before_checksum_dictionary_patch.20261107T145431Z
Manifest: /home/ubuntu/ghsl_multicity/hyderabad/batch/template_checksums.json

population: d3d0b3c5a92180531258a3f13b176c8ed14e9babd8cdfdf88a2e57cab19856ef
smod: ccf9a02016143676776af1d10eb31e30b3c37377fa85952f76200f9d99b6325c
built_surface: 6f805892684e89f5646c482c10be6e7bd9066a678a74e35f1f9776617087c3cd

Dictionary replacement count: 1
Driver compilation: PASS
AST dictionary validation: PASS
Template checksum verification: PASS
Original Bengaluru driver modified: NO
Original Bengaluru templates mod

In [ ]:
# ============================================================
# CELL 7E: PATCH HYDERABAD TILE COUNT AND REFRESH CHECKSUMS
#
# Fixes:
#   Hyderabad selected tiles = 16
#   Bengaluru template expectation = 12
#
# Updates only Hyderabad copies:
#   population_epoch_template.py
#   built_surface_epoch_template.py
#   run_city_epochs.py checksum dictionary
#
# Does not modify:
#   original Bengaluru templates
#   original Bengaluru driver
#   HDFS data
#
# Does not launch Spark.
# ============================================================

import base64
import hashlib
import textwrap


# ------------------------------------------------------------
# Validate notebook state.
# ------------------------------------------------------------

for required_name in [
    "ACTIVE_CITY",
    "ACTIVE_CITY_SLUG",
    "run_remote",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is unavailable. "
            "Rerun Cells 1, 2, and 3B."
        )

if ACTIVE_CITY_SLUG != "hyderabad":
    raise RuntimeError(
        "This patch is for Hyderabad only."
    )


remote_root = ACTIVE_CITY["remote_work_dir"]

driver_path = (
    remote_root
    + "/batch/run_city_epochs.py"
)

template_root = (
    remote_root
    + "/templates"
)


# ------------------------------------------------------------
# Build exact remote patch program.
# ------------------------------------------------------------

remote_patcher = textwrap.dedent(
    f'''
    from pathlib import Path
    import ast
    import hashlib
    import json
    import py_compile
    import shutil
    import time


    DRIVER = Path({driver_path!r})

    TEMPLATE_ROOT = Path({template_root!r})

    POPULATION_TEMPLATE = (
        TEMPLATE_ROOT
        / "population_epoch_template.py"
    )

    SMOD_TEMPLATE = (
        TEMPLATE_ROOT
        / "smod_epoch_template.py"
    )

    BUILT_TEMPLATE = (
        TEMPLATE_ROOT
        / "built_surface_epoch_template.py"
    )

    TEMPLATE_PATHS = {{
        "population": POPULATION_TEMPLATE,
        "smod": SMOD_TEMPLATE,
        "built_surface": BUILT_TEMPLATE,
    }}

    NEW_EXPECTED_SELECTED_TILES = 16


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(1024 * 1024),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    def backup(path, description):
        timestamp = time.strftime(
            "%Y%m%dT%H%M%SZ",
            time.gmtime(),
        )

        backup_path = path.with_name(
            path.name
            + ".before_"
            + description
            + "."
            + timestamp
        )

        shutil.copy2(
            path,
            backup_path,
        )

        return backup_path


    # --------------------------------------------------------
    # Validate files.
    # --------------------------------------------------------

    for required_path in [
        DRIVER,
        POPULATION_TEMPLATE,
        SMOD_TEMPLATE,
        BUILT_TEMPLATE,
    ]:
        if not required_path.is_file():
            raise FileNotFoundError(
                required_path
            )


    # --------------------------------------------------------
    # Back up Hyderabad-specific files.
    # --------------------------------------------------------

    driver_backup = backup(
        DRIVER,
        "tile_count_patch",
    )

    population_backup = backup(
        POPULATION_TEMPLATE,
        "tile_count_patch",
    )

    built_backup = backup(
        BUILT_TEMPLATE,
        "tile_count_patch",
    )


    # --------------------------------------------------------
    # Patch population and built-surface templates.
    #
    # SMOD operates at the 1 km product level and does not use
    # this 100 m selected-tile expectation.
    # --------------------------------------------------------

    patched_templates = {{}}

    for product, template_path in [
        ("population", POPULATION_TEMPLATE),
        ("built_surface", BUILT_TEMPLATE),
    ]:
        original_text = template_path.read_text(
            encoding="utf-8"
        )

        old_assignment = (
            "EXPECTED_SELECTED_TILES = 12"
        )

        new_assignment = (
            "EXPECTED_SELECTED_TILES = "
            + str(NEW_EXPECTED_SELECTED_TILES)
        )

        old_assignment_count = (
            original_text.count(old_assignment)
        )

        new_assignment_count = (
            original_text.count(new_assignment)
        )

        if old_assignment_count == 1:
            patched_text = original_text.replace(
                old_assignment,
                new_assignment,
                1,
            )

            patch_action = "12_to_16"

        elif (
            old_assignment_count == 0
            and new_assignment_count == 1
        ):
            patched_text = original_text

            patch_action = "already_16"

        else:
            raise RuntimeError(
                f"{{product}} template has an unexpected "
                f"tile-count structure. "
                f"old={{old_assignment_count}}, "
                f"new={{new_assignment_count}}"
            )

        # Update diagnostic text where present.
        patched_text = patched_text.replace(
            "Expected 12 selected tiles",
            "Expected 16 selected tiles",
        )

        compile(
            patched_text,
            str(template_path),
            "exec",
        )

        template_path.write_text(
            patched_text,
            encoding="utf-8",
        )

        py_compile.compile(
            str(template_path),
            doraise=True,
        )

        patched_templates[product] = {{
            "path": str(template_path),
            "action": patch_action,
        }}


    # --------------------------------------------------------
    # Calculate all three current Hyderabad template hashes.
    # --------------------------------------------------------

    current_checksums = {{
        product: sha256_file(path)
        for product, path
        in TEMPLATE_PATHS.items()
    }}


    # --------------------------------------------------------
    # Replace EXPECTED_TEMPLATE_SHA256 using AST line bounds.
    #
    # This avoids assumptions about indentation or where the
    # next configuration variable is located.
    # --------------------------------------------------------

    driver_text = DRIVER.read_text(
        encoding="utf-8"
    )

    parsed_driver = ast.parse(
        driver_text
    )

    checksum_assignment = None

    for node in parsed_driver.body:
        if not isinstance(node, ast.Assign):
            continue

        for target in node.targets:
            if (
                isinstance(target, ast.Name)
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
            ):
                checksum_assignment = node
                break

        if checksum_assignment is not None:
            break

    if checksum_assignment is None:
        raise RuntimeError(
            "EXPECTED_TEMPLATE_SHA256 assignment "
            "was not found in the city driver."
        )

    if not hasattr(
        checksum_assignment,
        "end_lineno",
    ):
        raise RuntimeError(
            "Python AST does not provide end_lineno."
        )

    driver_lines = driver_text.splitlines(
        keepends=True
    )

    start_index = (
        checksum_assignment.lineno - 1
    )

    end_index = (
        checksum_assignment.end_lineno
    )

    replacement_source = (
        "EXPECTED_TEMPLATE_SHA256 = {{\\n"
        "    \\"population\\": "
        + repr(current_checksums["population"])
        + ",\\n"
        "    \\"smod\\": "
        + repr(current_checksums["smod"])
        + ",\\n"
        "    \\"built_surface\\": "
        + repr(
            current_checksums["built_surface"]
        )
        + ",\\n"
        "}}\\n"
    )

    replacement_lines = (
        replacement_source.splitlines(
            keepends=True
        )
    )

    patched_driver_lines = (
        driver_lines[:start_index]
        + replacement_lines
        + driver_lines[end_index:]
    )

    patched_driver_text = "".join(
        patched_driver_lines
    )


    # --------------------------------------------------------
    # Compile before saving.
    # --------------------------------------------------------

    compile(
        patched_driver_text,
        str(DRIVER),
        "exec",
    )

    DRIVER.write_text(
        patched_driver_text,
        encoding="utf-8",
    )

    py_compile.compile(
        str(DRIVER),
        doraise=True,
    )


    # --------------------------------------------------------
    # Verify the saved dictionary through AST.
    # --------------------------------------------------------

    saved_driver_text = DRIVER.read_text(
        encoding="utf-8"
    )

    saved_ast = ast.parse(
        saved_driver_text
    )

    saved_checksums = None

    for node in saved_ast.body:
        if not isinstance(node, ast.Assign):
            continue

        for target in node.targets:
            if (
                isinstance(target, ast.Name)
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
            ):
                saved_checksums = (
                    ast.literal_eval(node.value)
                )
                break

        if saved_checksums is not None:
            break

    if saved_checksums != current_checksums:
        raise RuntimeError(
            "Saved driver checksum dictionary does "
            "not match the patched templates."
        )


    # --------------------------------------------------------
    # Verify selected-tile settings.
    # --------------------------------------------------------

    for product, template_path in [
        ("population", POPULATION_TEMPLATE),
        ("built_surface", BUILT_TEMPLATE),
    ]:
        saved_template = template_path.read_text(
            encoding="utf-8"
        )

        expected_assignment = (
            "EXPECTED_SELECTED_TILES = 16"
        )

        if (
            saved_template.count(
                expected_assignment
            )
            != 1
        ):
            raise RuntimeError(
                f"{{product}} does not contain exactly "
                f"one 16-tile assignment."
            )


    # --------------------------------------------------------
    # Write a permanent Hyderabad patch manifest.
    # --------------------------------------------------------

    manifest_path = (
        DRIVER.parent
        / "hyderabad_tile_count_patch.json"
    )

    manifest = {{
        "city_slug": "hyderabad",
        "expected_selected_tiles": (
            NEW_EXPECTED_SELECTED_TILES
        ),
        "templates": {{
            product: {{
                "path": str(path),
                "sha256": current_checksums[
                    product
                ],
                "patch_action": (
                    patched_templates.get(
                        product,
                        {{}},
                    ).get(
                        "action",
                        "checksum_only",
                    )
                ),
            }}
            for product, path
            in TEMPLATE_PATHS.items()
        }},
        "driver": {{
            "path": str(DRIVER),
            "sha256": sha256_file(DRIVER),
            "backup": str(driver_backup),
        }},
        "population_backup": str(
            population_backup
        ),
        "built_backup": str(
            built_backup
        ),
        "patched_at_utc": (
            time.strftime(
                "%Y-%m-%dT%H:%M:%SZ",
                time.gmtime(),
            )
        ),
    }}

    manifest_path.write_text(
        json.dumps(
            manifest,
            indent=2,
            sort_keys=True,
        )
        + "\\n",
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # Final report.
    # --------------------------------------------------------

    print("=" * 80)
    print("HYDERABAD TILE-COUNT PATCH")
    print("=" * 80)

    print(
        "Expected selected tiles:",
        NEW_EXPECTED_SELECTED_TILES,
    )

    print()
    print(
        "Population action:",
        patched_templates[
            "population"
        ]["action"],
    )

    print(
        "Built-surface action:",
        patched_templates[
            "built_surface"
        ]["action"],
    )

    print()
    print(
        "Population SHA-256:",
        current_checksums["population"],
    )

    print(
        "SMOD SHA-256:",
        current_checksums["smod"],
    )

    print(
        "Built SHA-256:",
        current_checksums[
            "built_surface"
        ],
    )

    print()
    print("Driver:", DRIVER)
    print("Driver backup:", driver_backup)
    print("Manifest:", manifest_path)

    print()
    print("Template compilation: PASS")
    print("Driver compilation: PASS")
    print("Checksum dictionary: PASS")
    print("Expected-tile verification: PASS")
    print("Original Bengaluru files modified: NO")
    print("Spark launched: NO")
    print("HDFS modified: NO")
    print("HYDERABAD TILE PATCH: SUCCESS")
    '''
).strip() + "\n"


# ------------------------------------------------------------
# Compile exact patcher locally.
# ------------------------------------------------------------

compile(
    remote_patcher,
    "patch_hyderabad_tile_count.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print("LOCAL TILE PATCHER COMPILE: PASS")
print("Expected patcher SHA-256:", patcher_sha256)


# ------------------------------------------------------------
# Transfer using Base64 and verify.
# ------------------------------------------------------------

encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")

remote_command = f"""
set -e

PATCHER="/tmp/patch_hyderabad_tile_count.py"
EXPECTED_SHA256="{patcher_sha256}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}"
"""

patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 7E RESULT")
print("=" * 80)

print("Return code:", patch_result.returncode)
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 7E: SUCCESS")

LOCAL TILE PATCHER COMPILE: PASS
Expected patcher SHA-256: 9584f35fde828a578c7a2df5f447e7a6f9f26b7547faacb0f0707eb4cb9ca6fe
Remote patcher SHA-256: 9584f35fde828a578c7a2df5f447e7a6f9f26b7547faacb0f0707eb4cb9ca6fe
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
HYDERABAD TILE-COUNT PATCH
Expected selected tiles: 16

Population action: 12_to_16
Built-surface action: 12_to_16

Population SHA-256: 85db8772adc3edebcf03bcf74577681685783cab6d296444c3f29f7c081f4ef9
SMOD SHA-256: ccf9a02016143676776af1d10eb31e30b3c37377fa85952f76200f9d99b6325c
Built SHA-256: 0df3cdccf6c8515659eee3acd490bf41b4c959668ef272d6e75f6f95ff1e8bb2

Driver: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py
Driver backup: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py.before_tile_count_patch.20261107T150256Z
Manifest: /home/ubuntu/ghsl_multicity/hyderabad/batch/hyderabad_tile_count_patch.json

Template compilation: PASS
Driver compilation: PASS
Checksum dictionary: PASS
Expected-til

In [ ]:
# ============================================================
# CELL 7F: REMOVE ENDPOINT-ONLY 1975 GRID COMPARISON
#
# The copied Bengaluru template compares the current output
# against Bengaluru 1975. That is valid for the old 2020
# endpoint prototype, but invalid for a generalized template,
# especially while generating Hyderabad 1975 itself.
#
# This cell:
#   1. Removes only that endpoint-specific validation block
#   2. Compiles the Hyderabad population template
#   3. Recalculates the population-template checksum
#   4. Updates the Hyderabad driver's checksum dictionary
#
# It does not launch Spark or modify HDFS.
# ============================================================

import base64
import hashlib
import textwrap


root = ACTIVE_CITY["remote_work_dir"]

population_template = (
    root
    + "/templates/population_epoch_template.py"
)

driver_path = (
    root
    + "/batch/run_city_epochs.py"
)


remote_patcher = textwrap.dedent(
    f'''
    from pathlib import Path
    import ast
    import hashlib
    import json
    import py_compile
    import shutil
    import time


    TEMPLATE = Path(
        {population_template!r}
    )

    DRIVER = Path(
        {driver_path!r}
    )


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(1024 * 1024),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    def timestamp():
        return time.strftime(
            "%Y%m%dT%H%M%SZ",
            time.gmtime(),
        )


    if not TEMPLATE.is_file():
        raise FileNotFoundError(TEMPLATE)

    if not DRIVER.is_file():
        raise FileNotFoundError(DRIVER)


    # --------------------------------------------------------
    # Back up Hyderabad-specific files.
    # --------------------------------------------------------

    template_backup = TEMPLATE.with_name(
        TEMPLATE.name
        + ".before_endpoint_validation_removal."
        + timestamp()
    )

    driver_backup = DRIVER.with_name(
        DRIVER.name
        + ".before_population_checksum_refresh."
        + timestamp()
    )

    shutil.copy2(
        TEMPLATE,
        template_backup,
    )

    shutil.copy2(
        DRIVER,
        driver_backup,
    )


    # --------------------------------------------------------
    # Remove the endpoint-only block using exact comment
    # markers from the proven Bengaluru template.
    # --------------------------------------------------------

    template_text = TEMPLATE.read_text(
        encoding="utf-8"
    )

    start_marker = (
        "        # "
        "====================================================\\n"
        "        # CONFIRM GRID IDS MATCH 1975 POPULATION\\n"
        "        # "
        "====================================================\\n"
    )

    end_marker = (
        "        # "
        "====================================================\\n"
        "        # DISPLAY SUMMARY\\n"
        "        # "
        "====================================================\\n"
    )

    start_count = template_text.count(
        start_marker
    )

    end_count = template_text.count(
        end_marker
    )

    if start_count == 1 and end_count == 1:
        start_index = template_text.index(
            start_marker
        )

        end_index = template_text.index(
            end_marker,
            start_index,
        )

        patched_template = (
            template_text[:start_index]
            + "        # Generalized multi-city template:\\n"
            + "        # endpoint-only 1975 key comparison "
            + "removed.\\n\\n"
            + template_text[end_index:]
        )

        patch_action = (
            "endpoint_validation_removed"
        )

    elif (
        "endpoint-only 1975 key comparison removed"
        in template_text
    ):
        patched_template = template_text

        patch_action = "already_removed"

    else:
        raise RuntimeError(
            "Could not uniquely locate the endpoint-only "
            "1975 key-comparison block. "
            f"start_count={{start_count}}, "
            f"end_count={{end_count}}"
        )


    # --------------------------------------------------------
    # Confirm the problematic read is gone.
    # --------------------------------------------------------

    forbidden_fragments = [
        "population_1975_path =",
        ".parquet(population_1975_path)",
        "1975-2020 GRID KEY COMPARISON",
        "The 2020 grid keys do not match 1975",
    ]

    remaining_forbidden = [
        fragment
        for fragment in forbidden_fragments
        if fragment in patched_template
    ]

    if remaining_forbidden:
        raise RuntimeError(
            "Endpoint-only validation fragments remain: "
            + repr(remaining_forbidden)
        )


    # --------------------------------------------------------
    # Compile and save Hyderabad population template.
    # --------------------------------------------------------

    compile(
        patched_template,
        str(TEMPLATE),
        "exec",
    )

    TEMPLATE.write_text(
        patched_template,
        encoding="utf-8",
    )

    py_compile.compile(
        str(TEMPLATE),
        doraise=True,
    )

    new_population_checksum = (
        sha256_file(TEMPLATE)
    )


    # --------------------------------------------------------
    # Read existing checksum dictionary from Hyderabad driver.
    # --------------------------------------------------------

    driver_text = DRIVER.read_text(
        encoding="utf-8"
    )

    driver_ast = ast.parse(
        driver_text
    )

    checksum_node = None
    current_checksums = None

    for node in driver_ast.body:
        if not isinstance(node, ast.Assign):
            continue

        for target in node.targets:
            if (
                isinstance(target, ast.Name)
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
            ):
                checksum_node = node
                current_checksums = (
                    ast.literal_eval(node.value)
                )
                break

        if checksum_node is not None:
            break

    if checksum_node is None:
        raise RuntimeError(
            "EXPECTED_TEMPLATE_SHA256 was not found."
        )


    # --------------------------------------------------------
    # Update only the Hyderabad population checksum.
    # --------------------------------------------------------

    current_checksums["population"] = (
        new_population_checksum
    )

    replacement_source = (
        "EXPECTED_TEMPLATE_SHA256 = "
        + repr(current_checksums)
        + "\\n"
    )

    driver_lines = driver_text.splitlines(
        keepends=True
    )

    start_index = (
        checksum_node.lineno - 1
    )

    end_index = (
        checksum_node.end_lineno
    )

    patched_driver = "".join(
        driver_lines[:start_index]
        + [replacement_source]
        + driver_lines[end_index:]
    )


    # --------------------------------------------------------
    # Compile and save Hyderabad driver.
    # --------------------------------------------------------

    compile(
        patched_driver,
        str(DRIVER),
        "exec",
    )

    DRIVER.write_text(
        patched_driver,
        encoding="utf-8",
    )

    py_compile.compile(
        str(DRIVER),
        doraise=True,
    )


    # --------------------------------------------------------
    # Final verification using saved source.
    # --------------------------------------------------------

    saved_template = TEMPLATE.read_text(
        encoding="utf-8"
    )

    saved_driver = DRIVER.read_text(
        encoding="utf-8"
    )

    for fragment in forbidden_fragments:
        if fragment in saved_template:
            raise RuntimeError(
                f"Forbidden fragment remains: {{fragment}}"
            )

    if (
        new_population_checksum
        not in saved_driver
    ):
        raise RuntimeError(
            "New population checksum is absent "
            "from the saved Hyderabad driver."
        )


    # --------------------------------------------------------
    # Remove stale generated population program.
    #
    # The driver will regenerate it from the corrected
    # template on the next launch.
    # --------------------------------------------------------

    stale_program = Path(
        {root!r}
    ) / "programs/epoch_1975/population_1975.py"

    stale_program_removed = False

    if stale_program.is_file():
        stale_program.unlink()
        stale_program_removed = True


    # --------------------------------------------------------
    # Save patch record.
    # --------------------------------------------------------

    manifest_path = (
        DRIVER.parent
        / "population_template_generalization.json"
    )

    manifest_path.write_text(
        json.dumps(
            {{
                "city_slug": "hyderabad",
                "action": patch_action,
                "population_template": str(TEMPLATE),
                "population_template_sha256": (
                    new_population_checksum
                ),
                "template_backup": str(
                    template_backup
                ),
                "driver_backup": str(
                    driver_backup
                ),
                "stale_program_removed": (
                    stale_program_removed
                ),
                "patched_at_utc": (
                    time.strftime(
                        "%Y-%m-%dT%H:%M:%SZ",
                        time.gmtime(),
                    )
                ),
            }},
            indent=2,
            sort_keys=True,
        )
        + "\\n",
        encoding="utf-8",
    )


    print("=" * 80)
    print("POPULATION TEMPLATE GENERALIZATION")
    print("=" * 80)

    print("Action:", patch_action)
    print("Template:", TEMPLATE)
    print("Template backup:", template_backup)
    print("Driver:", DRIVER)
    print("Driver backup:", driver_backup)
    print(
        "New population SHA-256:",
        new_population_checksum,
    )
    print(
        "Stale generated program removed:",
        stale_program_removed,
    )
    print("Manifest:", manifest_path)

    print()
    print("Endpoint-only comparison removed: PASS")
    print("Population template compilation: PASS")
    print("Driver compilation: PASS")
    print("Checksum refresh: PASS")
    print("Original Bengaluru files modified: NO")
    print("Spark launched: NO")
    print("HDFS modified: NO")
    print("POPULATION GENERALIZATION: SUCCESS")
    '''
).strip() + "\n"


# ============================================================
# LOCAL COMPILE AND TRANSFER
# ============================================================

compile(
    remote_patcher,
    "generalize_population_template.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print("LOCAL GENERALIZATION PATCHER COMPILE: PASS")
print("Expected patcher SHA-256:", patcher_sha256)


encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

PATCHER="/tmp/generalize_population_template.py"
EXPECTED_SHA256="{patcher_sha256}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 7F RESULT")
print("=" * 80)

print("Return code:", patch_result.returncode)
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 7F: SUCCESS")

LOCAL GENERALIZATION PATCHER COMPILE: PASS
Expected patcher SHA-256: 0b3ca056de5fc17db69b557ebfc0525e4998df48a241012cc2fec97208bce939
Remote patcher SHA-256: 0b3ca056de5fc17db69b557ebfc0525e4998df48a241012cc2fec97208bce939
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS

REMOTE STDERR:
Traceback (most recent call last):
  File "/tmp/generalize_population_template.py", line 137, in <module>
    raise RuntimeError(
    ...<4 lines>...
    )
RuntimeError: Could not uniquely locate the endpoint-only 1975 key-comparison block. start_count=0, end_count=0



RuntimeError: Remote command failed with return code 1.

In [ ]:
# ============================================================
# CELL 7G REPLACEMENT:
# INSPECT ACTUAL POPULATION TEMPLATE BLOCK
#
# Completely read-only.
# ============================================================

import base64
import hashlib
import textwrap


root = ACTIVE_CITY["remote_work_dir"]

population_template = (
    root
    + "/templates/population_epoch_template.py"
)

generated_program = (
    root
    + "/programs/epoch_1975/population_1975.py"
)


remote_inspector = textwrap.dedent(
    """
    from pathlib import Path
    import sys


    TEMPLATE = Path(sys.argv[1])
    PROGRAM = Path(sys.argv[2])


    if not TEMPLATE.is_file():
        raise FileNotFoundError(TEMPLATE)


    def print_context(
        path,
        needle,
        before=30,
        after=100,
    ):
        lines = path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()

        matches = [
            index + 1
            for index, line in enumerate(lines)
            if needle in line
        ]

        print(
            repr(needle)
            + ": "
            + repr(matches)
        )

        for line_number in matches:
            start = max(
                1,
                line_number - before,
            )

            end = min(
                len(lines),
                line_number + after,
            )

            print(
                "--- "
                + path.name
                + ": lines "
                + str(start)
                + "-"
                + str(end)
                + " ---"
            )

            for current in range(
                start,
                end + 1,
            ):
                print(
                    str(current).rjust(5)
                    + ": "
                    + repr(
                        lines[current - 1]
                    )
                )


    print("=" * 80)
    print("TEMPLATE FILE")
    print("=" * 80)

    print("Path:", TEMPLATE)
    print("Bytes:", TEMPLATE.stat().st_size)


    print()
    print("=" * 80)
    print("TARGETED TEMPLATE CONTEXT")
    print("=" * 80)

    template_needles = [
        "population_1975_path",
        ".parquet(population_1975_path)",
        "1975-2020 GRID KEY COMPARISON",
        "CONFIRM GRID IDS MATCH 1975",
        "DISPLAY SUMMARY",
    ]

    for needle in template_needles:
        print()
        print_context(
            TEMPLATE,
            needle,
            before=15,
            after=30,
        )


    print()
    print("=" * 80)
    print("GENERATED PROGRAM CONTEXT")
    print("=" * 80)

    if PROGRAM.is_file():
        print("Path:", PROGRAM)
        print("Bytes:", PROGRAM.stat().st_size)

        for needle in [
            "population_1975_path",
            ".parquet(population_1975_path)",
            "DISPLAY SUMMARY",
        ]:
            print()
            print_context(
                PROGRAM,
                needle,
                before=15,
                after=30,
            )
    else:
        print(
            "Generated program does not exist:",
            PROGRAM,
        )


    print()
    print("=" * 80)
    print("INSPECTION COMPLETE")
    print("=" * 80)
    """
).strip() + "\n"


# Compile locally before transfer.
compile(
    remote_inspector,
    "inspect_population_template.py",
    "exec",
)

inspector_sha256 = hashlib.sha256(
    remote_inspector.encode("utf-8")
).hexdigest()

print("LOCAL INSPECTOR COMPILE: PASS")
print("Expected SHA-256:", inspector_sha256)


# Transfer exact source.
encoded_inspector = base64.b64encode(
    remote_inspector.encode("utf-8")
).decode("ascii")

remote_command = f"""
set -e

INSPECTOR="/tmp/inspect_population_template.py"
EXPECTED_SHA256="{inspector_sha256}"
TEMPLATE="{population_template}"
PROGRAM="{generated_program}"

echo "{encoded_inspector}" \
| base64 -d \
> "${{INSPECTOR}}"

ACTUAL_SHA256="$(
    sha256sum "${{INSPECTOR}}" \
    | awk '{{print $1}}'
)"

echo "Remote inspector SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Inspector checksum mismatch."
    exit 20
fi

echo "REMOTE INSPECTOR CHECKSUM: PASS"

python3 -m py_compile "${{INSPECTOR}}"

echo "REMOTE INSPECTOR COMPILE: PASS"

python3 "${{INSPECTOR}}" \
    "${{TEMPLATE}}" \
    "${{PROGRAM}}"
"""

inspection_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 7G RESULT")
print("=" * 80)

print(
    "Return code:",
    inspection_result.returncode,
)

print("Files modified: NO")
print("Spark launched: NO")
print("HDFS modified: NO")

LOCAL INSPECTOR COMPILE: PASS
Expected SHA-256: 115e8194422ff9206add720c5fa566bab28445442b0aeb44078ea984b8a43185
Remote inspector SHA-256: 115e8194422ff9206add720c5fa566bab28445442b0aeb44078ea984b8a43185
REMOTE INSPECTOR CHECKSUM: PASS
REMOTE INSPECTOR COMPILE: PASS
TEMPLATE FILE
Path: /home/ubuntu/ghsl_multicity/hyderabad/templates/population_epoch_template.py
Bytes: 23392

TARGETED TEMPLATE CONTEXT

'population_1975_path': [788, 798]
--- population_epoch_template.py: lines 773-818 ---
  773: '        conservation_difference'
  774: '        > allowed_difference'
  775: '    ):'
  776: '        raise RuntimeError('
  777: '            "2020 population conservation failed. "'
  778: '            f"Source={source_total}, "'
  779: '            f"aggregated={aggregated_total}, "'
  780: '            f"difference={conservation_difference}."'
  781: '        )'
  782: ''
  783: ''
  784: '    # ===================================================='
  785: '    # CONFIRM GRID IDS MATCH 1975 

In [ ]:
# ============================================================
# CELL 7H: REMOVE EXACT ENDPOINT-ONLY POPULATION BLOCK
#
# Based on inspected Hyderabad template:
#   Start marker: CONFIRM GRID IDS MATCH 1975 POPULATION
#   End marker:   DISPLAY SUMMARY
#
# Updates only Hyderabad files.
# Does not launch Spark or modify HDFS.
# ============================================================

import base64
import hashlib


root = ACTIVE_CITY["remote_work_dir"]

template_path = (
    root
    + "/templates/population_epoch_template.py"
)

driver_path = (
    root
    + "/batch/run_city_epochs.py"
)

generated_program_path = (
    root
    + "/programs/epoch_1975/population_1975.py"
)


# ============================================================
# BUILD REMOTE PATCHER WITHOUT NESTED TRIPLE-QUOTED SOURCE
# ============================================================

remote_lines = [
    "from pathlib import Path",
    "import ast",
    "import hashlib",
    "import json",
    "import py_compile",
    "import shutil",
    "import time",
    "",
    f"TEMPLATE = Path({template_path!r})",
    f"DRIVER = Path({driver_path!r})",
    (
        "STALE_PROGRAM = Path("
        + repr(generated_program_path)
        + ")"
    ),
    "",
    "def sha256_file(path):",
    "    digest = hashlib.sha256()",
    "    with path.open('rb') as handle:",
    "        for block in iter(",
    "            lambda: handle.read(1024 * 1024),",
    "            b'',",
    "        ):",
    "            digest.update(block)",
    "    return digest.hexdigest()",
    "",
    "def timestamp():",
    "    return time.strftime(",
    "        '%Y%m%dT%H%M%SZ',",
    "        time.gmtime(),",
    "    )",
    "",
    "if not TEMPLATE.is_file():",
    "    raise FileNotFoundError(TEMPLATE)",
    "",
    "if not DRIVER.is_file():",
    "    raise FileNotFoundError(DRIVER)",
    "",
    "# Back up Hyderabad-specific files.",
    "template_backup = TEMPLATE.with_name(",
    "    TEMPLATE.name",
    "    + '.before_cross_epoch_removal.'",
    "    + timestamp()",
    ")",
    "",
    "driver_backup = DRIVER.with_name(",
    "    DRIVER.name",
    "    + '.before_population_checksum_refresh.'",
    "    + timestamp()",
    ")",
    "",
    "shutil.copy2(TEMPLATE, template_backup)",
    "shutil.copy2(DRIVER, driver_backup)",
    "",
    "# Read actual template lines.",
    "template_lines = TEMPLATE.read_text(",
    "    encoding='utf-8'",
    ").splitlines(keepends=True)",
    "",
    "start_matches = [",
    "    index",
    "    for index, line in enumerate(template_lines)",
    "    if 'CONFIRM GRID IDS MATCH 1975 POPULATION'",
    "    in line",
    "]",
    "",
    "end_matches = [",
    "    index",
    "    for index, line in enumerate(template_lines)",
    "    if 'DISPLAY SUMMARY' in line",
    "]",
    "",
    "already_removed = any(",
    "    'endpoint-only cross-epoch comparison removed'",
    "    in line",
    "    for line in template_lines",
    ")",
    "",
    "if already_removed:",
    "    patched_template = ''.join(template_lines)",
    "    patch_action = 'already_removed'",
    "",
    "else:",
    "    if len(start_matches) != 1:",
    "        raise RuntimeError(",
    "            'Expected one start marker, found '",
    "            + str(len(start_matches))",
    "        )",
    "",
    "    if len(end_matches) != 1:",
    "        raise RuntimeError(",
    "            'Expected one end marker, found '",
    "            + str(len(end_matches))",
    "        )",
    "",
    "    start_marker_index = start_matches[0]",
    "    end_marker_index = end_matches[0]",
    "",
    "    if end_marker_index <= start_marker_index:",
    "        raise RuntimeError(",
    "            'DISPLAY SUMMARY occurs before the '",
    "            'cross-epoch block.'",
    "        )",
    "",
    "    # Remove the separator immediately before the",
    "    # CONFIRM marker as well as its closing blank lines.",
    "    remove_start = max(",
    "        0,",
    "        start_marker_index - 1,",
    "    )",
    "",
    "    remove_end = max(",
    "        remove_start,",
    "        end_marker_index - 1,",
    "    )",
    "",
    "    replacement_lines = [",
    "        '    # Generalized multi-city template:\\n',",
    "        '    # endpoint-only cross-epoch comparison removed.\\n',",
    "        '\\n',",
    "    ]",
    "",
    "    patched_lines = (",
    "        template_lines[:remove_start]",
    "        + replacement_lines",
    "        + template_lines[remove_end:]",
    "    )",
    "",
    "    patched_template = ''.join(patched_lines)",
    "    patch_action = 'removed_exact_inspected_block'",
    "",
    "# Verify the problematic logic is gone.",
    "forbidden_fragments = [",
    "    'population_1975_path =',",
    "    '.parquet(population_1975_path)',",
    "    '1975-2020 GRID KEY COMPARISON',",
    "    'The 2020 grid keys do not match 1975',",
    "]",
    "",
    "remaining = [",
    "    fragment",
    "    for fragment in forbidden_fragments",
    "    if fragment in patched_template",
    "]",
    "",
    "if remaining:",
    "    raise RuntimeError(",
    "        'Forbidden endpoint fragments remain: '",
    "        + repr(remaining)",
    "    )",
    "",
    "# Keep DISPLAY SUMMARY and output-writing logic.",
    "if 'DISPLAY SUMMARY' not in patched_template:",
    "    raise RuntimeError(",
    "        'DISPLAY SUMMARY was accidentally removed.'",
    "    )",
    "",
    "# Compile and save template.",
    "compile(",
    "    patched_template,",
    "    str(TEMPLATE),",
    "    'exec',",
    ")",
    "",
    "TEMPLATE.write_text(",
    "    patched_template,",
    "    encoding='utf-8',",
    ")",
    "",
    "py_compile.compile(",
    "    str(TEMPLATE),",
    "    doraise=True,",
    ")",
    "",
    "population_checksum = sha256_file(TEMPLATE)",
    "",
    "# Parse driver and find checksum dictionary.",
    "driver_text = DRIVER.read_text(",
    "    encoding='utf-8'",
    ")",
    "",
    "driver_ast = ast.parse(driver_text)",
    "checksum_node = None",
    "checksum_dictionary = None",
    "",
    "for node in driver_ast.body:",
    "    if not isinstance(node, ast.Assign):",
    "        continue",
    "",
    "    for target in node.targets:",
    "        if (",
    "            isinstance(target, ast.Name)",
    "            and target.id",
    "            == 'EXPECTED_TEMPLATE_SHA256'",
    "        ):",
    "            checksum_node = node",
    "            checksum_dictionary = ast.literal_eval(",
    "                node.value",
    "            )",
    "            break",
    "",
    "    if checksum_node is not None:",
    "        break",
    "",
    "if checksum_node is None:",
    "    raise RuntimeError(",
    "        'Checksum dictionary was not found.'",
    "    )",
    "",
    "# Update population only; preserve SMOD and built hashes.",
    "checksum_dictionary['population'] = (",
    "    population_checksum",
    ")",
    "",
    "replacement_source = (",
    "    'EXPECTED_TEMPLATE_SHA256 = '",
    "    + repr(checksum_dictionary)",
    "    + '\\n'",
    ")",
    "",
    "driver_lines = driver_text.splitlines(",
    "    keepends=True",
    ")",
    "",
    "start_index = checksum_node.lineno - 1",
    "end_index = checksum_node.end_lineno",
    "",
    "patched_driver = ''.join(",
    "    driver_lines[:start_index]",
    "    + [replacement_source]",
    "    + driver_lines[end_index:]",
    ")",
    "",
    "compile(",
    "    patched_driver,",
    "    str(DRIVER),",
    "    'exec',",
    ")",
    "",
    "DRIVER.write_text(",
    "    patched_driver,",
    "    encoding='utf-8',",
    ")",
    "",
    "py_compile.compile(",
    "    str(DRIVER),",
    "    doraise=True,",
    ")",
    "",
    "# Remove stale program generated from old template.",
    "stale_program_removed = False",
    "",
    "if STALE_PROGRAM.is_file():",
    "    STALE_PROGRAM.unlink()",
    "    stale_program_removed = True",
    "",
    "# Verify final saved content.",
    "saved_template = TEMPLATE.read_text(",
    "    encoding='utf-8'",
    ")",
    "",
    "saved_driver = DRIVER.read_text(",
    "    encoding='utf-8'",
    ")",
    "",
    "if population_checksum not in saved_driver:",
    "    raise RuntimeError(",
    "        'New population checksum is absent '",
    "        'from the saved driver.'",
    "    )",
    "",
    "for fragment in forbidden_fragments:",
    "    if fragment in saved_template:",
    "        raise RuntimeError(",
    "            'Forbidden fragment remains: '",
    "            + fragment",
    "        )",
    "",
    "# Save permanent patch manifest.",
    "manifest_path = DRIVER.parent / (",
    "    'population_cross_epoch_patch.json'",
    ")",
    "",
    "manifest_path.write_text(",
    "    json.dumps(",
    "        {",
    "            'city_slug': 'hyderabad',",
    "            'action': patch_action,",
    "            'population_template': str(TEMPLATE),",
    "            'population_template_sha256':",
    "                population_checksum,",
    "            'template_backup':",
    "                str(template_backup),",
    "            'driver_backup':",
    "                str(driver_backup),",
    "            'stale_program_removed':",
    "                stale_program_removed,",
    "            'patched_at_utc':",
    "                time.strftime(",
    "                    '%Y-%m-%dT%H:%M:%SZ',",
    "                    time.gmtime(),",
    "                ),",
    "        },",
    "        indent=2,",
    "        sort_keys=True,",
    "    ) + '\\n',",
    "    encoding='utf-8',",
    ")",
    "",
    "print('=' * 80)",
    "print('POPULATION CROSS-EPOCH PATCH')",
    "print('=' * 80)",
    "print('Action:', patch_action)",
    "print('Template:', TEMPLATE)",
    "print('Template backup:', template_backup)",
    "print('Driver:', DRIVER)",
    "print('Driver backup:', driver_backup)",
    "print('Population SHA-256:', population_checksum)",
    "print('Stale program removed:', stale_program_removed)",
    "print('Manifest:', manifest_path)",
    "print()",
    "print('Endpoint-only block removed: PASS')",
    "print('DISPLAY SUMMARY retained: PASS')",
    "print('Template compilation: PASS')",
    "print('Driver compilation: PASS')",
    "print('Checksum refresh: PASS')",
    "print('Original Bengaluru files modified: NO')",
    "print('Spark launched: NO')",
    "print('HDFS modified: NO')",
    "print('POPULATION CROSS-EPOCH PATCH: SUCCESS')",
]


remote_patcher = "\n".join(
    remote_lines
) + "\n"


# ============================================================
# COMPILE AND TRANSFER
# ============================================================

compile(
    remote_patcher,
    "patch_population_cross_epoch.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print("LOCAL PATCHER COMPILE: PASS")
print("Expected SHA-256:", patcher_sha256)


encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

PATCHER="/tmp/patch_population_cross_epoch.py"
EXPECTED_SHA256="{patcher_sha256}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 7H RESULT")
print("=" * 80)

print("Return code:", patch_result.returncode)
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 7H: SUCCESS")

LOCAL PATCHER COMPILE: PASS
Expected SHA-256: 1ebedfb681dbbd32d8f49784a6f3708b57c01cfc87b9c8af5ce118b4f6fce134
Remote patcher SHA-256: 1ebedfb681dbbd32d8f49784a6f3708b57c01cfc87b9c8af5ce118b4f6fce134
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
POPULATION CROSS-EPOCH PATCH
Action: already_removed
Template: /home/ubuntu/ghsl_multicity/hyderabad/templates/population_epoch_template.py
Template backup: /home/ubuntu/ghsl_multicity/hyderabad/templates/population_epoch_template.py.before_cross_epoch_removal.20261107T151257Z
Driver: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py
Driver backup: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py.before_population_checksum_refresh.20261107T151257Z
Population SHA-256: 4e500a7a613f0aa373ae645dfcaed1b1ac709d1a241537475245cae418bb4415
Stale program removed: False
Manifest: /home/ubuntu/ghsl_multicity/hyderabad/batch/population_cross_epoch_patch.json

Endpoint-only block removed: PASS
DISPLAY SUMMARY retained:

In [ ]:
# ============================================================
# CELL 7K: GENERALIZE HYDERABAD POPULATION GENERATOR
#
# Removes two endpoint-only expectations from the Hyderabad
# driver:
#
#   1. population_2020_keys_df must occur three times
#   2. epoch=1975 path must remain in every generated program
#
# Updates only:
#   /home/ubuntu/ghsl_multicity/hyderabad/batch/
#   run_city_epochs.py
#
# Does not modify Bengaluru or HDFS.
# Does not launch Spark.
# ============================================================

import base64
import hashlib


root = ACTIVE_CITY["remote_work_dir"]

driver_path = (
    root
    + "/batch/run_city_epochs.py"
)

stale_program_path = (
    root
    + "/programs/epoch_1975/population_1975.py"
)


remote_lines = [
    "from pathlib import Path",
    "import hashlib",
    "import json",
    "import py_compile",
    "import shutil",
    "import time",
    "",
    f"DRIVER = Path({driver_path!r})",
    f"STALE_PROGRAM = Path({stale_program_path!r})",
    "",
    "if not DRIVER.is_file():",
    "    raise FileNotFoundError(DRIVER)",
    "",
    "driver_text = DRIVER.read_text(",
    "    encoding='utf-8'",
    ")",
    "",
    "timestamp = time.strftime(",
    "    '%Y%m%dT%H%M%SZ',",
    "    time.gmtime(),",
    ")",
    "",
    "backup_path = DRIVER.with_name(",
    "    DRIVER.name",
    "    + '.before_population_generator_generalization.'",
    "    + timestamp",
    ")",
    "",
    "shutil.copy2(DRIVER, backup_path)",
    "",
    "# Exact endpoint-only replacement block.",
    "target_key_block = '''        program = replace_exact_count(",
    "            program,",
    "            \"population_2020_keys_df\",",
    "            f\"population_{epoch}_keys_df\",",
    "            3,",
    "            \"Population target key variable\",",
    "        )",
    "",
    "'''",
    "",
    "target_key_count = driver_text.count(",
    "    target_key_block",
    ")",
    "",
    "if target_key_count == 1:",
    "    driver_text = driver_text.replace(",
    "        target_key_block,",
    "        (",
    "            '        # Multi-city population template '\\",
    "            '\\n'",
    "            '        # has no endpoint-only cross-epoch '\\",
    "            '\\n'",
    "            '        # key-variable replacement.\\n\\n'",
    "        ),",
    "        1,",
    "    )",
    "    target_action = 'removed'",
    "elif target_key_count == 0 and (",
    "    'has no endpoint-only cross-epoch'",
    "    in driver_text",
    "):",
    "    target_action = 'already_removed'",
    "else:",
    "    raise RuntimeError(",
    "        'Unexpected population target-key block count: '",
    "        + str(target_key_count)",
    "    )",
    "",
    "# Remove the endpoint-reference requirement from",
    "# Population require_fragments only.",
    "endpoint_requirement = '''                '    \"epoch=1975/\"',",
    "'''",
    "",
    "endpoint_requirement_count = driver_text.count(",
    "    endpoint_requirement",
    ")",
    "",
    "if endpoint_requirement_count == 1:",
    "    driver_text = driver_text.replace(",
    "        endpoint_requirement,",
    "        '',",
    "        1,",
    "    )",
    "    requirement_action = 'removed'",
    "elif endpoint_requirement_count == 0:",
    "    requirement_action = 'already_removed'",
    "else:",
    "    raise RuntimeError(",
    "        'Expected at most one population endpoint-path '",
    "        'requirement, found '",
    "        + str(endpoint_requirement_count)",
    "    )",
    "",
    "# Verify forbidden generator expectation is absent.",
    "forbidden = [",
    "    '\"population_2020_keys_df\",',",
    "    '\"Population target key variable\",',",
    "]",
    "",
    "remaining = [",
    "    marker",
    "    for marker in forbidden",
    "    if marker in driver_text",
    "]",
    "",
    "if remaining:",
    "    raise RuntimeError(",
    "        'Endpoint-only generator markers remain: '",
    "        + repr(remaining)",
    "    )",
    "",
    "# The active output-path replacement must remain.",
    "required = [",
    "    '\"Population output path\",',",
    "    '\"Population active epoch\",',",
    "    'generate_population_program',",
    "    'EXPECTED_TEMPLATE_SHA256',",
    "]",
    "",
    "missing = [",
    "    marker",
    "    for marker in required",
    "    if marker not in driver_text",
    "]",
    "",
    "if missing:",
    "    raise RuntimeError(",
    "        'Required population-generator logic missing: '",
    "        + repr(missing)",
    "    )",
    "",
    "# Compile before writing.",
    "compile(",
    "    driver_text,",
    "    str(DRIVER),",
    "    'exec',",
    ")",
    "",
    "DRIVER.write_text(",
    "    driver_text,",
    "    encoding='utf-8',",
    ")",
    "",
    "py_compile.compile(",
    "    str(DRIVER),",
    "    doraise=True,",
    ")",
    "",
    "# Delete stale generated program if one exists.",
    "stale_program_removed = False",
    "",
    "if STALE_PROGRAM.is_file():",
    "    STALE_PROGRAM.unlink()",
    "    stale_program_removed = True",
    "",
    "# Save patch manifest.",
    "manifest_path = DRIVER.parent / (",
    "    'population_generator_generalization.json'",
    ")",
    "",
    "manifest_path.write_text(",
    "    json.dumps(",
    "        {",
    "            'city_slug': 'hyderabad',",
    "            'driver': str(DRIVER),",
    "            'backup': str(backup_path),",
    "            'target_key_action': target_action,",
    "            'endpoint_requirement_action':",
    "                requirement_action,",
    "            'stale_program_removed':",
    "                stale_program_removed,",
    "            'patched_at_utc':",
    "                time.strftime(",
    "                    '%Y-%m-%dT%H:%M:%SZ',",
    "                    time.gmtime(),",
    "                ),",
    "        },",
    "        indent=2,",
    "        sort_keys=True,",
    "    ) + '\\n',",
    "    encoding='utf-8',",
    ")",
    "",
    "print('=' * 80)",
    "print('POPULATION GENERATOR GENERALIZATION')",
    "print('=' * 80)",
    "print('Driver:', DRIVER)",
    "print('Backup:', backup_path)",
    "print('Target-key block:', target_action)",
    "print(",
    "    '1975 endpoint requirement:',",
    "    requirement_action,",
    ")",
    "print(",
    "    'Stale generated program removed:',",
    "    stale_program_removed,",
    ")",
    "print('Manifest:', manifest_path)",
    "print()",
    "print('Driver compilation: PASS')",
    "print('Active epoch replacement retained: PASS')",
    "print('Output-path replacement retained: PASS')",
    "print('Endpoint key-variable replacement absent: PASS')",
    "print('Original Bengaluru driver modified: NO')",
    "print('Spark launched: NO')",
    "print('HDFS modified: NO')",
    "print('POPULATION GENERATOR GENERALIZATION: SUCCESS')",
]


remote_patcher = "\n".join(
    remote_lines
) + "\n"


# ============================================================
# COMPILE AND TRANSFER
# ============================================================

compile(
    remote_patcher,
    "generalize_population_generator.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print("LOCAL GENERATOR PATCHER COMPILE: PASS")
print("Expected SHA-256:", patcher_sha256)


encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

PATCHER="/tmp/generalize_population_generator.py"
EXPECTED_SHA256="{patcher_sha256}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 7K RESULT")
print("=" * 80)

print("Return code:", patch_result.returncode)
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 7K: SUCCESS")

LOCAL GENERATOR PATCHER COMPILE: PASS
Expected SHA-256: 7ad91ef0e070572950ebae87a0a6bc75d4b97b9a6ebc5a723d8b2910c848186e
Remote patcher SHA-256: 7ad91ef0e070572950ebae87a0a6bc75d4b97b9a6ebc5a723d8b2910c848186e
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS

REMOTE STDERR:
Traceback (most recent call last):
  File "/tmp/generalize_population_generator.py", line 65, in <module>
    raise RuntimeError(
    ...<2 lines>...
    )
RuntimeError: Unexpected population target-key block count: 0



RuntimeError: Remote command failed with return code 1.

In [ ]:
# ============================================================
# CELL 7L: AST-BASED POPULATION GENERATOR GENERALIZATION
#
# Removes from Hyderabad's driver:
#   1. replace_exact_count(... "population_2020_keys_df" ...)
#   2. Population require_fragments entry for epoch=1975
#
# Uses Python AST line positions, not fragile text matching.
#
# Does not modify Bengaluru.
# Does not launch Spark.
# Does not modify HDFS.
# ============================================================

import base64
import hashlib

root = ACTIVE_CITY["remote_work_dir"]

driver_path = (
    root
    + "/batch/run_city_epochs.py"
)

stale_program_path = (
    root
    + "/programs/epoch_1975/population_1975.py"
)


remote_lines = [
    "from pathlib import Path",
    "import ast",
    "import hashlib",
    "import json",
    "import py_compile",
    "import shutil",
    "import time",
    "",
    f"DRIVER = Path({driver_path!r})",
    f"STALE_PROGRAM = Path({stale_program_path!r})",
    "",
    "if not DRIVER.is_file():",
    "    raise FileNotFoundError(DRIVER)",
    "",
    "source = DRIVER.read_text(encoding='utf-8')",
    "tree = ast.parse(source)",
    "",
    "# Find generate_population_program only.",
    "population_function = None",
    "",
    "for node in tree.body:",
    "    if (",
    "        isinstance(node, ast.FunctionDef)",
    "        and node.name == 'generate_population_program'",
    "    ):",
    "        population_function = node",
    "        break",
    "",
    "if population_function is None:",
    "    raise RuntimeError(",
    "        'generate_population_program was not found.'",
    "    )",
    "",
    "# --------------------------------------------------------",
    "# Locate endpoint-only replace_exact_count call.",
    "# --------------------------------------------------------",
    "",
    "target_replace_node = None",
    "",
    "for node in ast.walk(population_function):",
    "    if not isinstance(node, ast.Expr):",
    "        continue",
    "",
    "    call = node.value",
    "",
    "    if not isinstance(call, ast.Call):",
    "        continue",
    "",
    "    if not isinstance(call.func, ast.Name):",
    "        continue",
    "",
    "    if call.func.id != 'replace_exact_count':",
    "        continue",
    "",
    "    string_values = [",
    "        child.value",
    "        for child in ast.walk(call)",
    "        if (",
    "            isinstance(child, ast.Constant)",
    "            and isinstance(child.value, str)",
    "        )",
    "    ]",
    "",
    "    if 'population_2020_keys_df' in string_values:",
    "        if target_replace_node is not None:",
    "            raise RuntimeError(",
    "                'Multiple target-key replacement calls found.'",
    "            )",
    "",
    "        target_replace_node = node",
    "",
    "if target_replace_node is None:",
    "    # Idempotent success if already removed.",
    "    target_action = 'already_removed'",
    "else:",
    "    target_action = 'removed_by_ast'",
    "",
    "# --------------------------------------------------------",
    "# Locate epoch=1975 entry inside population require_fragments.",
    "# --------------------------------------------------------",
    "",
    "endpoint_fragment_node = None",
    "",
    "for node in ast.walk(population_function):",
    "    if not isinstance(node, ast.Call):",
    "        continue",
    "",
    "    function_name = None",
    "",
    "    if isinstance(node.func, ast.Name):",
    "        function_name = node.func.id",
    "",
    "    if function_name != 'require_fragments':",
    "        continue",
    "",
    "    for child in ast.walk(node):",
    "        if (",
    "            isinstance(child, ast.Constant)",
    "            and isinstance(child.value, str)",
    "            and child.value == '    \"epoch=1975/\"'",
    "        ):",
    "            if endpoint_fragment_node is not None:",
    "                raise RuntimeError(",
    "                    'Multiple population epoch=1975 '",
    "                    'requirements found.'",
    "                )",
    "",
    "            endpoint_fragment_node = child",
    "",
    "if endpoint_fragment_node is None:",
    "    requirement_action = 'already_removed'",
    "else:",
    "    requirement_action = 'removed_by_ast'",
    "",
    "# At least one target must still exist unless previously patched.",
    "if (",
    "    target_replace_node is None",
    "    and endpoint_fragment_node is None",
    "    and 'AST population generalization applied'",
    "    not in source",
    "):",
    "    raise RuntimeError(",
    "        'Neither endpoint-only target was found, and no '",
    "        'prior AST patch marker exists.'",
    "    )",
    "",
    "# --------------------------------------------------------",
    "# Back up current Hyderabad driver.",
    "# --------------------------------------------------------",
    "",
    "timestamp = time.strftime(",
    "    '%Y%m%dT%H%M%SZ',",
    "    time.gmtime(),",
    ")",
    "",
    "backup = DRIVER.with_name(",
    "    DRIVER.name",
    "    + '.before_ast_population_generalization.'",
    "    + timestamp",
    ")",
    "",
    "shutil.copy2(DRIVER, backup)",
    "",
    "# --------------------------------------------------------",
    "# Apply edits by line range, from bottom to top.",
    "# --------------------------------------------------------",
    "",
    "lines = source.splitlines(keepends=True)",
    "edits = []",
    "",
    "if target_replace_node is not None:",
    "    edits.append((",
    "        target_replace_node.lineno - 1,",
    "        target_replace_node.end_lineno,",
    "        [",
    "            '        # AST population generalization applied:\\n',",
    "            '        # endpoint-only key-variable replacement removed.\\n',",
    "            '\\n',",
    "        ],",
    "    ))",
    "",
    "if endpoint_fragment_node is not None:",
    "    fragment_start = endpoint_fragment_node.lineno - 1",
    "    fragment_end = endpoint_fragment_node.end_lineno",
    "",
    "    # Remove the complete list-entry line.",
    "    edits.append((",
    "        fragment_start,",
    "        fragment_end,",
    "        [],",
    "    ))",
    "",
    "for start, end, replacement in sorted(",
    "    edits,",
    "    key=lambda item: item[0],",
    "    reverse=True,",
    "):",
    "    lines[start:end] = replacement",
    "",
    "patched = ''.join(lines)",
    "",
    "# --------------------------------------------------------",
    "# Verify endpoint-only generation logic is gone.",
    "# --------------------------------------------------------",
    "",
    "patched_tree = ast.parse(patched)",
    "",
    "for node in ast.walk(patched_tree):",
    "    if not isinstance(node, ast.Call):",
    "        continue",
    "",
    "    values = [",
    "        child.value",
    "        for child in ast.walk(node)",
    "        if (",
    "            isinstance(child, ast.Constant)",
    "            and isinstance(child.value, str)",
    "        )",
    "    ]",
    "",
    "    if 'population_2020_keys_df' in values:",
    "        raise RuntimeError(",
    "            'population_2020_keys_df replacement remains.'",
    "        )",
    "",
    "    if '    \"epoch=1975/\"' in values:",
    "        raise RuntimeError(",
    "            'Population epoch=1975 requirement remains.'",
    "        )",
    "",
    "# Confirm essential generation logic remains.",
    "required_markers = [",
    "    'def generate_population_program(',",
    "    'Population output path',",
    "    'Population active epoch',",
    "    'Population catalog view',",
    "    'Population catalog SQL reference',",
    "    'RS_FromGeoTiff',",
    "    '.parquet(OUTPUT_PATH)',",
    "]",
    "",
    "missing = [",
    "    marker",
    "    for marker in required_markers",
    "    if marker not in patched",
    "]",
    "",
    "if missing:",
    "    raise RuntimeError(",
    "        'Required population logic was lost: '",
    "        + repr(missing)",
    "    )",
    "",
    "# Compile and save.",
    "compile(patched, str(DRIVER), 'exec')",
    "",
    "DRIVER.write_text(",
    "    patched,",
    "    encoding='utf-8',",
    ")",
    "",
    "py_compile.compile(",
    "    str(DRIVER),",
    "    doraise=True,",
    ")",
    "",
    "# Remove stale generated program.",
    "stale_removed = False",
    "",
    "if STALE_PROGRAM.is_file():",
    "    STALE_PROGRAM.unlink()",
    "    stale_removed = True",
    "",
    "# Write patch manifest.",
    "manifest = DRIVER.parent / (",
    "    'ast_population_generator_patch.json'",
    ")",
    "",
    "manifest.write_text(",
    "    json.dumps(",
    "        {",
    "            'city_slug': 'hyderabad',",
    "            'driver': str(DRIVER),",
    "            'backup': str(backup),",
    "            'target_action': target_action,",
    "            'requirement_action': requirement_action,",
    "            'stale_program_removed': stale_removed,",
    "            'patched_at_utc': time.strftime(",
    "                '%Y-%m-%dT%H:%M:%SZ',",
    "                time.gmtime(),",
    "            ),",
    "        },",
    "        indent=2,",
    "        sort_keys=True,",
    "    ) + '\\n',",
    "    encoding='utf-8',",
    ")",
    "",
    "print('=' * 80)",
    "print('AST POPULATION GENERATOR PATCH')",
    "print('=' * 80)",
    "print('Driver:', DRIVER)",
    "print('Backup:', backup)",
    "print('Target-key action:', target_action)",
    "print('1975 requirement action:', requirement_action)",
    "print('Stale program removed:', stale_removed)",
    "print('Manifest:', manifest)",
    "print()",
    "print('AST verification: PASS')",
    "print('Driver compilation: PASS')",
    "print('Required population logic retained: PASS')",
    "print('Original Bengaluru driver modified: NO')",
    "print('Spark launched: NO')",
    "print('HDFS modified: NO')",
    "print('AST POPULATION GENERALIZATION: SUCCESS')",
]


remote_patcher = "\n".join(
    remote_lines
) + "\n"


# ============================================================
# COMPILE AND TRANSFER
# ============================================================

compile(
    remote_patcher,
    "ast_population_generator_patch.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print("LOCAL AST PATCHER COMPILE: PASS")
print("Expected SHA-256:", patcher_sha256)


encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

PATCHER="/tmp/ast_population_generator_patch.py"
EXPECTED_SHA256="{patcher_sha256}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 7L RESULT")
print("=" * 80)

print("Return code:", patch_result.returncode)
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 7L: SUCCESS")

LOCAL AST PATCHER COMPILE: PASS
Expected SHA-256: 97f15f2ebdec9b9ce5e5190d2adfd8f6da87e073063998f58dfef47126ac632e
Remote patcher SHA-256: 97f15f2ebdec9b9ce5e5190d2adfd8f6da87e073063998f58dfef47126ac632e
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS

REMOTE STDERR:
Traceback (most recent call last):
  File "/tmp/ast_population_generator_patch.py", line 202, in <module>
    raise RuntimeError(
        'population_2020_keys_df replacement remains.'
    )
RuntimeError: population_2020_keys_df replacement remains.



RuntimeError: Remote command failed with return code 1.

In [ ]:
# ============================================================
# CELL 7M: CORRECT AST POPULATION GENERATOR PATCH
#
# Fix:
#   program = replace_exact_count(...) is ast.Assign,
#   not ast.Expr.
#
# Updates only the Hyderabad driver.
# Does not launch Spark or modify HDFS.
# ============================================================

import base64
import hashlib


root = ACTIVE_CITY["remote_work_dir"]

driver_path = (
    root
    + "/batch/run_city_epochs.py"
)

stale_program_path = (
    root
    + "/programs/epoch_1975/population_1975.py"
)


remote_lines = [
    "from pathlib import Path",
    "import ast",
    "import json",
    "import py_compile",
    "import shutil",
    "import time",
    "",
    f"DRIVER = Path({driver_path!r})",
    f"STALE_PROGRAM = Path({stale_program_path!r})",
    "",
    "if not DRIVER.is_file():",
    "    raise FileNotFoundError(DRIVER)",
    "",
    "source = DRIVER.read_text(encoding='utf-8')",
    "tree = ast.parse(source)",
    "",
    "# Locate generate_population_program.",
    "population_function = None",
    "",
    "for node in tree.body:",
    "    if (",
    "        isinstance(node, ast.FunctionDef)",
    "        and node.name == 'generate_population_program'",
    "    ):",
    "        population_function = node",
    "        break",
    "",
    "if population_function is None:",
    "    raise RuntimeError(",
    "        'generate_population_program was not found.'",
    "    )",
    "",
    "# Locate the complete statement containing the",
    "# endpoint-only replace_exact_count call.",
    "target_statement = None",
    "",
    "for statement in population_function.body:",
    "    if not isinstance(",
    "        statement,",
    "        (ast.Assign, ast.Expr),",
    "    ):",
    "        continue",
    "",
    "    calls = [",
    "        node",
    "        for node in ast.walk(statement)",
    "        if isinstance(node, ast.Call)",
    "    ]",
    "",
    "    for call in calls:",
    "        if not (",
    "            isinstance(call.func, ast.Name)",
    "            and call.func.id",
    "            == 'replace_exact_count'",
    "        ):",
    "            continue",
    "",
    "        string_values = [",
    "            child.value",
    "            for child in ast.walk(call)",
    "            if (",
    "                isinstance(child, ast.Constant)",
    "                and isinstance(child.value, str)",
    "            )",
    "        ]",
    "",
    "        if (",
    "            'population_2020_keys_df'",
    "            in string_values",
    "        ):",
    "            if target_statement is not None:",
    "                raise RuntimeError(",
    "                    'Multiple population target-key '",
    "                    'statements were found.'",
    "                )",
    "",
    "            target_statement = statement",
    "",
    "if target_statement is None:",
    "    if (",
    "        'endpoint-only target-key replacement removed'",
    "        in source",
    "    ):",
    "        action = 'already_removed'",
    "    else:",
    "        raise RuntimeError(",
    "            'Population target-key statement was not found.'",
    "        )",
    "else:",
    "    action = 'removed_ast_assign'",
    "",
    "# Back up current Hyderabad driver.",
    "timestamp = time.strftime(",
    "    '%Y%m%dT%H%M%SZ',",
    "    time.gmtime(),",
    ")",
    "",
    "backup = DRIVER.with_name(",
    "    DRIVER.name",
    "    + '.before_correct_ast_patch.'",
    "    + timestamp",
    ")",
    "",
    "shutil.copy2(DRIVER, backup)",
    "",
    "if target_statement is not None:",
    "    lines = source.splitlines(keepends=True)",
    "",
    "    start_index = target_statement.lineno - 1",
    "    end_index = target_statement.end_lineno",
    "",
    "    replacement = [",
    "        '        # Multi-city generalization:\\n',",
    "        '        # endpoint-only target-key replacement removed.\\n',",
    "        '\\n',",
    "    ]",
    "",
    "    lines[start_index:end_index] = replacement",
    "    patched = ''.join(lines)",
    "else:",
    "    patched = source",
    "",
    "# Parse patched driver and inspect only the",
    "# population generator.",
    "patched_tree = ast.parse(patched)",
    "patched_population_function = None",
    "",
    "for node in patched_tree.body:",
    "    if (",
    "        isinstance(node, ast.FunctionDef)",
    "        and node.name == 'generate_population_program'",
    "    ):",
    "        patched_population_function = node",
    "        break",
    "",
    "if patched_population_function is None:",
    "    raise RuntimeError(",
    "        'Patched population generator is missing.'",
    "    )",
    "",
    "# Confirm no active call in this function still contains",
    "# population_2020_keys_df.",
    "remaining_calls = []",
    "",
    "for call in ast.walk(patched_population_function):",
    "    if not isinstance(call, ast.Call):",
    "        continue",
    "",
    "    values = [",
    "        child.value",
    "        for child in ast.walk(call)",
    "        if (",
    "            isinstance(child, ast.Constant)",
    "            and isinstance(child.value, str)",
    "        )",
    "    ]",
    "",
    "    if 'population_2020_keys_df' in values:",
    "        remaining_calls.append(call.lineno)",
    "",
    "if remaining_calls:",
    "    raise RuntimeError(",
    "        'Active population target-key calls remain at '",
    "        + repr(remaining_calls)",
    "    )",
    "",
    "# Essential generator transformations must remain.",
    "required_markers = [",
    "    'Population output path',",
    "    'Population active epoch',",
    "    'Population application name',",
    "    'Population catalog view',",
    "    'Population catalog SQL reference',",
    "]",
    "",
    "missing = [",
    "    marker",
    "    for marker in required_markers",
    "    if marker not in patched",
    "]",
    "",
    "if missing:",
    "    raise RuntimeError(",
    "        'Required generator logic is missing: '",
    "        + repr(missing)",
    "    )",
    "",
    "# Compile and save.",
    "compile(patched, str(DRIVER), 'exec')",
    "",
    "DRIVER.write_text(",
    "    patched,",
    "    encoding='utf-8',",
    ")",
    "",
    "py_compile.compile(",
    "    str(DRIVER),",
    "    doraise=True,",
    ")",
    "",
    "# Delete stale generated population program.",
    "stale_removed = False",
    "",
    "if STALE_PROGRAM.is_file():",
    "    STALE_PROGRAM.unlink()",
    "    stale_removed = True",
    "",
    "# Save patch manifest.",
    "manifest = DRIVER.parent / (",
    "    'correct_ast_population_patch.json'",
    ")",
    "",
    "manifest.write_text(",
    "    json.dumps(",
    "        {",
    "            'city_slug': 'hyderabad',",
    "            'action': action,",
    "            'driver': str(DRIVER),",
    "            'backup': str(backup),",
    "            'stale_program_removed': stale_removed,",
    "            'patched_at_utc': time.strftime(",
    "                '%Y-%m-%dT%H:%M:%SZ',",
    "                time.gmtime(),",
    "            ),",
    "        },",
    "        indent=2,",
    "        sort_keys=True,",
    "    ) + '\\n',",
    "    encoding='utf-8',",
    ")",
    "",
    "print('=' * 80)",
    "print('CORRECT AST POPULATION PATCH')",
    "print('=' * 80)",
    "print('Action:', action)",
    "print('Driver:', DRIVER)",
    "print('Backup:', backup)",
    "print('Stale program removed:', stale_removed)",
    "print('Manifest:', manifest)",
    "print()",
    "print('Population generator AST verification: PASS')",
    "print('Driver compilation: PASS')",
    "print('Required transformations retained: PASS')",
    "print('Original Bengaluru driver modified: NO')",
    "print('Spark launched: NO')",
    "print('HDFS modified: NO')",
    "print('CORRECT AST PATCH: SUCCESS')",
]


remote_patcher = "\n".join(
    remote_lines
) + "\n"


# ============================================================
# LOCAL COMPILE AND TRANSFER
# ============================================================

compile(
    remote_patcher,
    "correct_ast_population_patch.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print("LOCAL CORRECT AST PATCHER COMPILE: PASS")
print("Expected SHA-256:", patcher_sha256)


encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

PATCHER="/tmp/correct_ast_population_patch.py"
EXPECTED_SHA256="{patcher_sha256}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 7M RESULT")
print("=" * 80)

print("Return code:", patch_result.returncode)
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 7M: SUCCESS")

LOCAL CORRECT AST PATCHER COMPILE: PASS
Expected SHA-256: aae467f06a5804199dce119b337a4e326659343ee28f31bd039e3c0621ae292f
Remote patcher SHA-256: aae467f06a5804199dce119b337a4e326659343ee28f31bd039e3c0621ae292f
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
CORRECT AST POPULATION PATCH
Action: already_removed
Driver: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py
Backup: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py.before_correct_ast_patch.20261107T151715Z
Stale program removed: False
Manifest: /home/ubuntu/ghsl_multicity/hyderabad/batch/correct_ast_population_patch.json

Population generator AST verification: PASS
Driver compilation: PASS
Required transformations retained: PASS
Original Bengaluru driver modified: NO
Spark launched: NO
HDFS modified: NO
CORRECT AST PATCH: SUCCESS

REMOTE STDERR:


CELL 7M RESULT
Return code: 0
Spark launched: NO
HDFS modified: NO
CELL 7M: SUCCESS


In [ ]:
# ============================================================
# CELL 7N: GENERALIZE THE HYDERABAD SMOD TEMPLATE
#
# Removes only:
#   VALIDATE GRID IDS AGAINST SMOD 1975
#
# Retains:
#   - population-to-SMOD alignment
#   - SMOD class validation
#   - 4,900-cell validation
#   - HDFS write
#   - HDFS read-back validation
#
# Updates only Hyderabad files.
# Does not modify the successful population output.
# Does not launch Spark or modify HDFS.
# ============================================================

import base64
import hashlib


# ============================================================
# VALIDATE NOTEBOOK STATE
# ============================================================

for required_name in [
    "ACTIVE_CITY",
    "ACTIVE_CITY_SLUG",
    "run_remote",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is unavailable. "
            "Rerun Cells 1, 2, and 3B."
        )


if ACTIVE_CITY_SLUG != "hyderabad":
    raise RuntimeError(
        "This patch is specifically for Hyderabad."
    )


root = ACTIVE_CITY["remote_work_dir"]

template_path = (
    root
    + "/templates/smod_epoch_template.py"
)

driver_path = (
    root
    + "/batch/run_city_epochs.py"
)

stale_program_path = (
    root
    + "/programs/epoch_1975/smod_1975.py"
)


# ============================================================
# BUILD REMOTE PATCH PROGRAM
# ============================================================

remote_lines = [
    "from pathlib import Path",
    "import ast",
    "import hashlib",
    "import json",
    "import py_compile",
    "import shutil",
    "import time",
    "",
    f"TEMPLATE = Path({template_path!r})",
    f"DRIVER = Path({driver_path!r})",
    f"STALE_PROGRAM = Path({stale_program_path!r})",
    "",
    "",
    "def sha256_file(path):",
    "    digest = hashlib.sha256()",
    "",
    "    with path.open('rb') as handle:",
    "        for block in iter(",
    "            lambda: handle.read(1024 * 1024),",
    "            b'',",
    "        ):",
    "            digest.update(block)",
    "",
    "    return digest.hexdigest()",
    "",
    "",
    "def utc_stamp():",
    "    return time.strftime(",
    "        '%Y%m%dT%H%M%SZ',",
    "        time.gmtime(),",
    "    )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Validate Hyderabad-specific files.",
    "# --------------------------------------------------------",
    "",
    "if not TEMPLATE.is_file():",
    "    raise FileNotFoundError(TEMPLATE)",
    "",
    "if not DRIVER.is_file():",
    "    raise FileNotFoundError(DRIVER)",
    "",
    "",
    "# --------------------------------------------------------",
    "# Read files before creating backups or modifying anything.",
    "# --------------------------------------------------------",
    "",
    "template_text = TEMPLATE.read_text(",
    "    encoding='utf-8'",
    ")",
    "",
    "driver_text = DRIVER.read_text(",
    "    encoding='utf-8'",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Identify the endpoint-only SMOD comparison block.",
    "# --------------------------------------------------------",
    "",
    "template_lines = template_text.splitlines(",
    "    keepends=True",
    ")",
    "",
    "start_label = (",
    "    'VALIDATE GRID IDS AGAINST SMOD 1975'",
    ")",
    "",
    "end_label = (",
    "    'WRITE SCALAR SMOD PARQUET TO HDFS'",
    ")",
    "",
    "patch_marker = (",
    "    'endpoint-only SMOD cross-epoch comparison removed'",
    ")",
    "",
    "start_matches = [",
    "    index",
    "    for index, line in enumerate(template_lines)",
    "    if start_label in line",
    "]",
    "",
    "end_matches = [",
    "    index",
    "    for index, line in enumerate(template_lines)",
    "    if end_label in line",
    "]",
    "",
    "already_removed = any(",
    "    patch_marker in line",
    "    for line in template_lines",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Create the generalized SMOD template in memory.",
    "# --------------------------------------------------------",
    "",
    "if already_removed:",
    "    patched_template = template_text",
    "    patch_action = 'already_removed'",
    "",
    "else:",
    "    if len(start_matches) != 1:",
    "        raise RuntimeError(",
    "            'Expected one SMOD start marker, found '",
    "            + str(len(start_matches))",
    "        )",
    "",
    "    if len(end_matches) != 1:",
    "        raise RuntimeError(",
    "            'Expected one SMOD end marker, found '",
    "            + str(len(end_matches))",
    "        )",
    "",
    "    start_marker_index = start_matches[0]",
    "    end_marker_index = end_matches[0]",
    "",
    "    if end_marker_index <= start_marker_index:",
    "        raise RuntimeError(",
    "            'The SMOD write block occurs before the '",
    "            'cross-epoch comparison block.'",
    "        )",
    "",
    "    # Include the separator directly above the start label.",
    "    remove_start = max(",
    "        0,",
    "        start_marker_index - 1,",
    "    )",
    "",
    "    # Keep the separator and label for the write section.",
    "    remove_end = max(",
    "        remove_start,",
    "        end_marker_index - 1,",
    "    )",
    "",
    "    replacement_lines = [",
    "        '    # Generalized multi-city SMOD template:\\n',",
    "        '    # endpoint-only SMOD cross-epoch comparison removed.\\n',",
    "        '\\n',",
    "    ]",
    "",
    "    patched_lines = (",
    "        template_lines[:remove_start]",
    "        + replacement_lines",
    "        + template_lines[remove_end:]",
    "    )",
    "",
    "    patched_template = ''.join(",
    "        patched_lines",
    "    )",
    "",
    "    patch_action = (",
    "        'removed_exact_smod_block'",
    "    )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Confirm the invalid self-read logic is gone.",
    "# --------------------------------------------------------",
    "",
    "forbidden_fragments = [",
    "    'smod_1975_df =',",
    "    '.parquet(SMOD_1975_PATH)',",
    "    'missing_from_2020 =',",
    "    'new_in_2020 =',",
    "    '1975-2020 SMOD GRID KEY COMPARISON',",
    "    'The 2020 SMOD grid is missing',",
    "    'The 2020 SMOD grid contains identifiers',",
    "]",
    "",
    "remaining_forbidden = [",
    "    fragment",
    "    for fragment in forbidden_fragments",
    "    if fragment in patched_template",
    "]",
    "",
    "if remaining_forbidden:",
    "    raise RuntimeError(",
    "        'Endpoint-only SMOD logic remains: '",
    "        + repr(remaining_forbidden)",
    "    )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Confirm required generalized processing remains.",
    "# --------------------------------------------------------",
    "",
    "required_fragments = [",
    "    'population_without_smod',",
    "    'smod_without_population',",
    "    'WRITE SCALAR SMOD PARQUET TO HDFS',",
    "    '.parquet(SMOD_OUTPUT_PATH)',",
    "    'READ-BACK VALIDATION',",
    "    'RS_FromGeoTiff',",
    "    'RS_Tile',",
    "    'RS_BandAsArray',",
    "]",
    "",
    "missing_required = [",
    "    fragment",
    "    for fragment in required_fragments",
    "    if fragment not in patched_template",
    "]",
    "",
    "if missing_required:",
    "    raise RuntimeError(",
    "        'Required SMOD logic was removed: '",
    "        + repr(missing_required)",
    "    )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Compile the generalized template before writing it.",
    "# --------------------------------------------------------",
    "",
    "compile(",
    "    patched_template,",
    "    str(TEMPLATE),",
    "    'exec',",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Calculate the future checksum before saving.",
    "# --------------------------------------------------------",
    "",
    "new_smod_checksum = hashlib.sha256(",
    "    patched_template.encode('utf-8')",
    ").hexdigest()",
    "",
    "",
    "# --------------------------------------------------------",
    "# Parse the Hyderabad driver's checksum dictionary.",
    "# --------------------------------------------------------",
    "",
    "driver_ast = ast.parse(",
    "    driver_text",
    ")",
    "",
    "checksum_node = None",
    "checksum_dictionary = None",
    "",
    "for node in driver_ast.body:",
    "    if not isinstance(node, ast.Assign):",
    "        continue",
    "",
    "    for target in node.targets:",
    "        if (",
    "            isinstance(target, ast.Name)",
    "            and target.id",
    "            == 'EXPECTED_TEMPLATE_SHA256'",
    "        ):",
    "            checksum_node = node",
    "            checksum_dictionary = ast.literal_eval(",
    "                node.value",
    "            )",
    "            break",
    "",
    "    if checksum_node is not None:",
    "        break",
    "",
    "if checksum_node is None:",
    "    raise RuntimeError(",
    "        'EXPECTED_TEMPLATE_SHA256 was not found.'",
    "    )",
    "",
    "if not isinstance(",
    "    checksum_dictionary,",
    "    dict,",
    "):",
    "    raise RuntimeError(",
    "        'The template checksum object is not a dictionary.'",
    "    )",
    "",
    "for required_product in [",
    "    'population',",
    "    'smod',",
    "    'built_surface',",
    "]:",
    "    if required_product not in checksum_dictionary:",
    "        raise RuntimeError(",
    "            'Checksum dictionary is missing: '",
    "            + required_product",
    "        )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Update only the Hyderabad SMOD checksum.",
    "# --------------------------------------------------------",
    "",
    "old_smod_checksum = checksum_dictionary['smod']",
    "",
    "checksum_dictionary['smod'] = (",
    "    new_smod_checksum",
    ")",
    "",
    "replacement_source = (",
    "    'EXPECTED_TEMPLATE_SHA256 = '",
    "    + repr(checksum_dictionary)",
    "    + '\\n'",
    ")",
    "",
    "driver_lines = driver_text.splitlines(",
    "    keepends=True",
    ")",
    "",
    "start_index = checksum_node.lineno - 1",
    "end_index = checksum_node.end_lineno",
    "",
    "patched_driver = ''.join(",
    "    driver_lines[:start_index]",
    "    + [replacement_source]",
    "    + driver_lines[end_index:]",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Compile both final files before modifying the disk.",
    "# --------------------------------------------------------",
    "",
    "compile(",
    "    patched_template,",
    "    str(TEMPLATE),",
    "    'exec',",
    ")",
    "",
    "compile(",
    "    patched_driver,",
    "    str(DRIVER),",
    "    'exec',",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Back up Hyderabad-specific files.",
    "# --------------------------------------------------------",
    "",
    "template_backup = TEMPLATE.with_name(",
    "    TEMPLATE.name",
    "    + '.before_cross_epoch_removal.'",
    "    + utc_stamp()",
    ")",
    "",
    "driver_backup = DRIVER.with_name(",
    "    DRIVER.name",
    "    + '.before_smod_checksum_refresh.'",
    "    + utc_stamp()",
    ")",
    "",
    "shutil.copy2(",
    "    TEMPLATE,",
    "    template_backup,",
    ")",
    "",
    "shutil.copy2(",
    "    DRIVER,",
    "    driver_backup,",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Save and compile the Hyderabad SMOD template.",
    "# --------------------------------------------------------",
    "",
    "TEMPLATE.write_text(",
    "    patched_template,",
    "    encoding='utf-8',",
    ")",
    "",
    "py_compile.compile(",
    "    str(TEMPLATE),",
    "    doraise=True,",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Save and compile the Hyderabad driver.",
    "# --------------------------------------------------------",
    "",
    "DRIVER.write_text(",
    "    patched_driver,",
    "    encoding='utf-8',",
    ")",
    "",
    "py_compile.compile(",
    "    str(DRIVER),",
    "    doraise=True,",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Remove the stale generated SMOD program.",
    "# --------------------------------------------------------",
    "",
    "stale_program_removed = False",
    "",
    "if STALE_PROGRAM.is_file():",
    "    STALE_PROGRAM.unlink()",
    "    stale_program_removed = True",
    "",
    "",
    "# --------------------------------------------------------",
    "# Verify saved files.",
    "# --------------------------------------------------------",
    "",
    "saved_template = TEMPLATE.read_text(",
    "    encoding='utf-8'",
    ")",
    "",
    "saved_driver = DRIVER.read_text(",
    "    encoding='utf-8'",
    ")",
    "",
    "actual_saved_checksum = sha256_file(",
    "    TEMPLATE",
    ")",
    "",
    "if actual_saved_checksum != new_smod_checksum:",
    "    raise RuntimeError(",
    "        'Saved SMOD template checksum mismatch. '",
    "        + 'Expected '",
    "        + new_smod_checksum",
    "        + ', actual '",
    "        + actual_saved_checksum",
    "    )",
    "",
    "for fragment in forbidden_fragments:",
    "    if fragment in saved_template:",
    "        raise RuntimeError(",
    "            'Forbidden SMOD fragment remains: '",
    "            + fragment",
    "        )",
    "",
    "for fragment in required_fragments:",
    "    if fragment not in saved_template:",
    "        raise RuntimeError(",
    "            'Required saved SMOD fragment is missing: '",
    "            + fragment",
    "        )",
    "",
    "if new_smod_checksum not in saved_driver:",
    "    raise RuntimeError(",
    "        'New SMOD checksum is absent from the driver.'",
    "    )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Save permanent patch metadata.",
    "# --------------------------------------------------------",
    "",
    "manifest_path = DRIVER.parent / (",
    "    'smod_cross_epoch_patch.json'",
    ")",
    "",
    "manifest_data = {",
    "    'city_slug': 'hyderabad',",
    "    'action': patch_action,",
    "    'smod_template': str(TEMPLATE),",
    "    'old_smod_sha256': old_smod_checksum,",
    "    'new_smod_sha256': new_smod_checksum,",
    "    'template_backup': str(template_backup),",
    "    'driver_backup': str(driver_backup),",
    "    'stale_program_removed': stale_program_removed,",
    "    'population_1975_preserved': True,",
    "    'patched_at_utc': time.strftime(",
    "        '%Y-%m-%dT%H:%M:%SZ',",
    "        time.gmtime(),",
    "    ),",
    "}",
    "",
    "manifest_path.write_text(",
    "    json.dumps(",
    "        manifest_data,",
    "        indent=2,",
    "        sort_keys=True,",
    "    ) + '\\n',",
    "    encoding='utf-8',",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Final report.",
    "# --------------------------------------------------------",
    "",
    "print('=' * 80)",
    "print('SMOD CROSS-EPOCH GENERALIZATION')",
    "print('=' * 80)",
    "",
    "print('Action:', patch_action)",
    "print('Template:', TEMPLATE)",
    "print('Template backup:', template_backup)",
    "print('Driver:', DRIVER)",
    "print('Driver backup:', driver_backup)",
    "print('Old SMOD SHA-256:', old_smod_checksum)",
    "print('New SMOD SHA-256:', new_smod_checksum)",
    "print(",
    "    'Stale SMOD program removed:',",
    "    stale_program_removed,",
    ")",
    "print('Manifest:', manifest_path)",
    "",
    "print()",
    "print('Population-SMOD validation retained: PASS')",
    "print('SMOD self-read removed: PASS')",
    "print('HDFS write logic retained: PASS')",
    "print('Read-back validation retained: PASS')",
    "print('Template compilation: PASS')",
    "print('Driver compilation: PASS')",
    "print('Checksum refresh: PASS')",
    "print('Population 1975 output preserved: YES')",
    "print('Original Bengaluru files modified: NO')",
    "print('Spark launched: NO')",
    "print('HDFS modified: NO')",
    "print('SMOD GENERALIZATION: SUCCESS')",
]


remote_patcher = "\n".join(
    remote_lines
) + "\n"


# ============================================================
# COMPILE LOCALLY
# ============================================================

compile(
    remote_patcher,
    "generalize_smod_template.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print("LOCAL SMOD PATCHER COMPILE: PASS")

print(
    "Expected SHA-256:",
    patcher_sha256,
)


# ============================================================
# TRANSFER AND RUN
# ============================================================

encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

PATCHER="/tmp/generalize_smod_template.py"
EXPECTED_SHA256="{patcher_sha256}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 7N RESULT")
print("=" * 80)

print(
    "Return code:",
    patch_result.returncode,
)

print(
    "Population 1975 output preserved:",
    "YES",
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 7N: SUCCESS")

LOCAL SMOD PATCHER COMPILE: PASS
Expected SHA-256: 81395dd30141b8454151395866e46b97c0bace2c6fb1949a19fe6bb882b193a4
Remote patcher SHA-256: 81395dd30141b8454151395866e46b97c0bace2c6fb1949a19fe6bb882b193a4
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
SMOD CROSS-EPOCH GENERALIZATION
Action: removed_exact_smod_block
Template: /home/ubuntu/ghsl_multicity/hyderabad/templates/smod_epoch_template.py
Template backup: /home/ubuntu/ghsl_multicity/hyderabad/templates/smod_epoch_template.py.before_cross_epoch_removal.20261107T152456Z
Driver: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py
Driver backup: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py.before_smod_checksum_refresh.20261107T152456Z
Old SMOD SHA-256: ccf9a02016143676776af1d10eb31e30b3c37377fa85952f76200f9d99b6325c
New SMOD SHA-256: 9101ea32a9a551cf1fef478f7b03e99501373797bd4b5df61e5b2a72e4177818
Stale SMOD program removed: True
Manifest: /home/ubuntu/ghsl_multicity/hyderabad/batch/smod_cross

In [ ]:
# ============================================================
# CELL 7O REPLACEMENT:
# AST-BASED BUILT-SURFACE GRID REFERENCE PATCH
#
# Replaces:
#   GRID_YEAR_REFERENCE_PATH =
#       .../grid_year/epoch=2020
#
# With:
#   GRID_YEAR_REFERENCE_PATH =
#       .../product=population/epoch=2020/grid_1km
#
# The existing built generator changes epoch=2020 to the
# active epoch. Therefore:
#
#   built 1975 validates against population 1975
#   built 1980 validates against population 1980
#   ...
#
# Updates only Hyderabad files.
# Preserves population 1975 and SMOD 1975 in HDFS.
# Does not launch Spark or modify HDFS.
# ============================================================

import base64
import hashlib


# ============================================================
# VALIDATE NOTEBOOK STATE
# ============================================================

for required_name in [
    "ACTIVE_CITY",
    "ACTIVE_CITY_SLUG",
    "run_remote",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is unavailable. "
            "Rerun Cells 1, 2, and 3B."
        )


if ACTIVE_CITY_SLUG != "hyderabad":
    raise RuntimeError(
        "This patch is specifically for Hyderabad."
    )


root = ACTIVE_CITY["remote_work_dir"]

template_path = (
    root
    + "/templates/"
    + "built_surface_epoch_template.py"
)

driver_path = (
    root
    + "/batch/run_city_epochs.py"
)

stale_program_path = (
    root
    + "/programs/epoch_1975/"
    + "built_surface_1975.py"
)


# ============================================================
# BUILD REMOTE PATCHER
# ============================================================

remote_lines = [
    "from pathlib import Path",
    "import ast",
    "import hashlib",
    "import json",
    "import py_compile",
    "import shutil",
    "import time",
    "",
    f"TEMPLATE = Path({template_path!r})",
    f"DRIVER = Path({driver_path!r})",
    f"STALE_PROGRAM = Path({stale_program_path!r})",
    "",
    "",
    "def sha256_file(path):",
    "    digest = hashlib.sha256()",
    "",
    "    with path.open('rb') as handle:",
    "        for block in iter(",
    "            lambda: handle.read(1024 * 1024),",
    "            b'',",
    "        ):",
    "            digest.update(block)",
    "",
    "    return digest.hexdigest()",
    "",
    "",
    "def utc_stamp():",
    "    return time.strftime(",
    "        '%Y%m%dT%H%M%SZ',",
    "        time.gmtime(),",
    "    )",
    "",
    "",
    "if not TEMPLATE.is_file():",
    "    raise FileNotFoundError(TEMPLATE)",
    "",
    "if not DRIVER.is_file():",
    "    raise FileNotFoundError(DRIVER)",
    "",
    "",
    "# --------------------------------------------------------",
    "# Parse the actual Hyderabad built-surface template.",
    "# --------------------------------------------------------",
    "",
    "template_text = TEMPLATE.read_text(",
    "    encoding='utf-8'",
    ")",
    "",
    "template_ast = ast.parse(",
    "    template_text",
    ")",
    "",
    "reference_node = None",
    "reference_value = None",
    "",
    "for node in template_ast.body:",
    "    if not isinstance(node, ast.Assign):",
    "        continue",
    "",
    "    for target in node.targets:",
    "        if (",
    "            isinstance(target, ast.Name)",
    "            and target.id",
    "            == 'GRID_YEAR_REFERENCE_PATH'",
    "        ):",
    "            if reference_node is not None:",
    "                raise RuntimeError(",
    "                    'Multiple GRID_YEAR_REFERENCE_PATH '",
    "                    'assignments were found.'",
    "                )",
    "",
    "            reference_node = node",
    "",
    "            try:",
    "                reference_value = ast.literal_eval(",
    "                    node.value",
    "                )",
    "            except Exception as error:",
    "                raise RuntimeError(",
    "                    'Could not evaluate the current '",
    "                    'grid-reference path.'",
    "                ) from error",
    "",
    "if reference_node is None:",
    "    raise RuntimeError(",
    "        'GRID_YEAR_REFERENCE_PATH was not found.'",
    "    )",
    "",
    "if not isinstance(reference_value, str):",
    "    raise RuntimeError(",
    "        'GRID_YEAR_REFERENCE_PATH is not a string.'",
    "    )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Define and validate the intended replacement.",
    "# --------------------------------------------------------",
    "",
    "expected_suffix = (",
    "    '/grid_year/epoch=2020'",
    ")",
    "",
    "new_reference = (",
    "    'hdfs://localhost:9000'",
    "    '/data/ghsl/curated/cities/'",
    "    'city=hyderabad/'",
    "    'product=population/'",
    "    'epoch=2020/'",
    "    'grid_1km'",
    ")",
    "",
    "if reference_value == new_reference:",
    "    patch_action = 'already_patched'",
    "",
    "elif reference_value.endswith(",
    "    expected_suffix",
    "):",
    "    patch_action = (",
    "        'grid_year_to_population_reference'",
    "    )",
    "",
    "else:",
    "    raise RuntimeError(",
    "        'Unexpected current grid-reference value: '",
    "        + reference_value",
    "    )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Replace the complete assignment by AST line range.",
    "# --------------------------------------------------------",
    "",
    "template_lines = template_text.splitlines(",
    "    keepends=True",
    ")",
    "",
    "start_index = reference_node.lineno - 1",
    "end_index = reference_node.end_lineno",
    "",
    "replacement_source = (",
    "    'GRID_YEAR_REFERENCE_PATH = (\\n'",
    "    '    \"hdfs://localhost:9000\"\\n'",
    "    '    \"/data/ghsl/curated/cities/\"\\n'",
    "    '    \"city=hyderabad/\"\\n'",
    "    '    \"product=population/\"\\n'",
    "    '    \"epoch=2020/\"\\n'",
    "    '    \"grid_1km\"\\n'",
    "    ')\\n'",
    ")",
    "",
    "replacement_lines = replacement_source.splitlines(",
    "    keepends=True",
    ")",
    "",
    "patched_template = ''.join(",
    "    template_lines[:start_index]",
    "    + replacement_lines",
    "    + template_lines[end_index:]",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Improve only the diagnostic label.",
    "# --------------------------------------------------------",
    "",
    "patched_template = patched_template.replace(",
    "    'print(\"Grid reference:\", '",
    "    'GRID_YEAR_REFERENCE_PATH)',",
    "    'print(\"Population-grid reference:\", '",
    "    'GRID_YEAR_REFERENCE_PATH)',",
    "    1,",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Verify the patched assignment through AST.",
    "# --------------------------------------------------------",
    "",
    "patched_template_ast = ast.parse(",
    "    patched_template",
    ")",
    "",
    "saved_reference_value = None",
    "",
    "for node in patched_template_ast.body:",
    "    if not isinstance(node, ast.Assign):",
    "        continue",
    "",
    "    for target in node.targets:",
    "        if (",
    "            isinstance(target, ast.Name)",
    "            and target.id",
    "            == 'GRID_YEAR_REFERENCE_PATH'",
    "        ):",
    "            saved_reference_value = (",
    "                ast.literal_eval(node.value)",
    "            )",
    "            break",
    "",
    "    if saved_reference_value is not None:",
    "        break",
    "",
    "if saved_reference_value != new_reference:",
    "    raise RuntimeError(",
    "        'Patched reference does not match the '",
    "        'intended population-grid path. '",
    "        + 'Found: '",
    "        + repr(saved_reference_value)",
    "    )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Confirm required processing remains.",
    "# --------------------------------------------------------",
    "",
    "required_template_fragments = [",
    "    'GRID_YEAR_REFERENCE_PATH',",
    "    'OUTPUT_PATH',",
    "    'Built-surface conservation: PASS',",
    "    '.parquet(GRID_YEAR_REFERENCE_PATH)',",
    "    '.parquet(OUTPUT_PATH)',",
    "    'RS_FromGeoTiff',",
    "    'RS_Tile',",
    "    'RS_BandAsArray',",
    "]",
    "",
    "missing_template_fragments = [",
    "    fragment",
    "    for fragment in required_template_fragments",
    "    if fragment not in patched_template",
    "]",
    "",
    "if missing_template_fragments:",
    "    raise RuntimeError(",
    "        'Required built-surface logic is missing: '",
    "        + repr(missing_template_fragments)",
    "    )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Compile the template and calculate its new checksum.",
    "# --------------------------------------------------------",
    "",
    "compile(",
    "    patched_template,",
    "    str(TEMPLATE),",
    "    'exec',",
    ")",
    "",
    "new_built_checksum = hashlib.sha256(",
    "    patched_template.encode('utf-8')",
    ").hexdigest()",
    "",
    "",
    "# --------------------------------------------------------",
    "# Parse the Hyderabad driver's checksum dictionary.",
    "# --------------------------------------------------------",
    "",
    "driver_text = DRIVER.read_text(",
    "    encoding='utf-8'",
    ")",
    "",
    "driver_ast = ast.parse(",
    "    driver_text",
    ")",
    "",
    "checksum_node = None",
    "checksum_dictionary = None",
    "",
    "for node in driver_ast.body:",
    "    if not isinstance(node, ast.Assign):",
    "        continue",
    "",
    "    for target in node.targets:",
    "        if (",
    "            isinstance(target, ast.Name)",
    "            and target.id",
    "            == 'EXPECTED_TEMPLATE_SHA256'",
    "        ):",
    "            checksum_node = node",
    "            checksum_dictionary = ast.literal_eval(",
    "                node.value",
    "            )",
    "            break",
    "",
    "    if checksum_node is not None:",
    "        break",
    "",
    "if checksum_node is None:",
    "    raise RuntimeError(",
    "        'EXPECTED_TEMPLATE_SHA256 was not found.'",
    "    )",
    "",
    "for product in [",
    "    'population',",
    "    'smod',",
    "    'built_surface',",
    "]:",
    "    if product not in checksum_dictionary:",
    "        raise RuntimeError(",
    "            'Checksum dictionary is missing '",
    "            + product",
    "        )",
    "",
    "old_built_checksum = (",
    "    checksum_dictionary['built_surface']",
    ")",
    "",
    "checksum_dictionary['built_surface'] = (",
    "    new_built_checksum",
    ")",
    "",
    "checksum_replacement = (",
    "    'EXPECTED_TEMPLATE_SHA256 = '",
    "    + repr(checksum_dictionary)",
    "    + '\\n'",
    ")",
    "",
    "driver_lines = driver_text.splitlines(",
    "    keepends=True",
    ")",
    "",
    "checksum_start = checksum_node.lineno - 1",
    "checksum_end = checksum_node.end_lineno",
    "",
    "patched_driver = ''.join(",
    "    driver_lines[:checksum_start]",
    "    + [checksum_replacement]",
    "    + driver_lines[checksum_end:]",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Confirm the driver still contains the built generator.",
    "# --------------------------------------------------------",
    "",
    "required_driver_fragments = [",
    "    'def generate_built_program(',",
    "    'EXPECTED_TEMPLATE_SHA256',",
    "]",
    "",
    "# --------------------------------------------------------",
    "# Confirm the driver still contains the built generator.",
    "# --------------------------------------------------------",
    "",
    "required_driver_fragments = [",
    "    'def generate_built_program(',",
    "    'EXPECTED_TEMPLATE_SHA256',",
    "]",
    "",
    "# --------------------------------------------------------",
    "# Confirm the driver still contains the built generator.",
    "# --------------------------------------------------------",
    "",
    "required_driver_fragments = [",
    "    'def generate_built_program(',",
    "    'EXPECTED_TEMPLATE_SHA256',",
    "]",
    "",
    "# --------------------------------------------------------",
    "# Confirm the driver still contains the built generator.",
    "# --------------------------------------------------------",
    "",
    "required_driver_fragments = [",
    "    'def generate_built_program(',",
    "    'EXPECTED_TEMPLATE_SHA256',",
    "]",
    "",
    "# --------------------------------------------------------",
    "# Confirm the driver still contains the built generator.",
    "# --------------------------------------------------------",
    "",
    "required_driver_fragments = [",
    "    'def generate_built_program(',",
    "    'EXPECTED_TEMPLATE_SHA256',",
    "]",
    "",
    "# --------------------------------------------------------",
    "# Confirm the driver still contains the built generator.",
    "# --------------------------------------------------------",
    "",
    "required_driver_fragments = [",
    "    'def generate_built_program(',",
    "    'EXPECTED_TEMPLATE_SHA256',",
    "]",
    "",
    "# --------------------------------------------------------",
    "# Confirm the driver still contains the built generator.",
    "# --------------------------------------------------------",
    "",
    "required_driver_fragments = [",
    "    'def generate_built_program(',",
    "    'EXPECTED_TEMPLATE_SHA256',",
    "]",
    "",
    "# --------------------------------------------------------",
    "# Confirm the driver still contains the built generator.",
    "# --------------------------------------------------------",
    "",
    "required_driver_fragments = [",
    "    'def generate_built_program(',",
    "    'EXPECTED_TEMPLATE_SHA256',",
    "]",
    "",
    "# --------------------------------------------------------",
    "# Confirm the driver still contains the built generator.",
    "# --------------------------------------------------------",
    "",
    "required_driver_fragments = [",
    "    'def generate_built_program(',",
    "    'EXPECTED_TEMPLATE_SHA256',",
    "]",
    "",
    "# --------------------------------------------------------",
    "# Confirm the driver still contains the built generator.",
    "# --------------------------------------------------------",
    "",
    "required_driver_fragments = [",
    "    'def generate_built_program(',",
    "    'EXPECTED_TEMPLATE_SHA256',",
    "]",
    "",
    "missing_driver_fragments = [",
    "    fragment",
    "    for fragment in required_driver_fragments",
    "    if fragment not in patched_driver",
    "]",
    "",
    "if missing_driver_fragments:",
    "    raise RuntimeError(",
    "        'Required built generator logic is missing: '",
    "        + repr(missing_driver_fragments)",
    "    )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Compile both final files before modifying disk.",
    "# --------------------------------------------------------",
    "",
    "compile(",
    "    patched_template,",
    "    str(TEMPLATE),",
    "    'exec',",
    ")",
    "",
    "compile(",
    "    patched_driver,",
    "    str(DRIVER),",
    "    'exec',",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Back up only Hyderabad-specific files.",
    "# --------------------------------------------------------",
    "",
    "template_backup = TEMPLATE.with_name(",
    "    TEMPLATE.name",
    "    + '.before_ast_grid_reference.'",
    "    + utc_stamp()",
    ")",
    "",
    "driver_backup = DRIVER.with_name(",
    "    DRIVER.name",
    "    + '.before_ast_built_checksum.'",
    "    + utc_stamp()",
    ")",
    "",
    "shutil.copy2(",
    "    TEMPLATE,",
    "    template_backup,",
    ")",
    "",
    "shutil.copy2(",
    "    DRIVER,",
    "    driver_backup,",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Save and compile Hyderabad-specific files.",
    "# --------------------------------------------------------",
    "",
    "TEMPLATE.write_text(",
    "    patched_template,",
    "    encoding='utf-8',",
    ")",
    "",
    "py_compile.compile(",
    "    str(TEMPLATE),",
    "    doraise=True,",
    ")",
    "",
    "DRIVER.write_text(",
    "    patched_driver,",
    "    encoding='utf-8',",
    ")",
    "",
    "py_compile.compile(",
    "    str(DRIVER),",
    "    doraise=True,",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Remove stale generated built program.",
    "# --------------------------------------------------------",
    "",
    "stale_program_removed = False",
    "",
    "if STALE_PROGRAM.is_file():",
    "    STALE_PROGRAM.unlink()",
    "    stale_program_removed = True",
    "",
    "",
    "# --------------------------------------------------------",
    "# Verify saved files.",
    "# --------------------------------------------------------",
    "",
    "saved_template_text = TEMPLATE.read_text(",
    "    encoding='utf-8'",
    ")",
    "",
    "saved_template_ast = ast.parse(",
    "    saved_template_text",
    ")",
    "",
    "final_reference = None",
    "",
    "for node in saved_template_ast.body:",
    "    if not isinstance(node, ast.Assign):",
    "        continue",
    "",
    "    for target in node.targets:",
    "        if (",
    "            isinstance(target, ast.Name)",
    "            and target.id",
    "            == 'GRID_YEAR_REFERENCE_PATH'",
    "        ):",
    "            final_reference = ast.literal_eval(",
    "                node.value",
    "            )",
    "            break",
    "",
    "    if final_reference is not None:",
    "        break",
    "",
    "if final_reference != new_reference:",
    "    raise RuntimeError(",
    "        'Saved built reference is incorrect: '",
    "        + repr(final_reference)",
    "    )",
    "",
    "actual_saved_checksum = sha256_file(",
    "    TEMPLATE",
    ")",
    "",
    "if actual_saved_checksum != new_built_checksum:",
    "    raise RuntimeError(",
    "        'Saved built template checksum mismatch.'",
    "    )",
    "",
    "saved_driver = DRIVER.read_text(",
    "    encoding='utf-8'",
    ")",
    "",
    "if new_built_checksum not in saved_driver:",
    "    raise RuntimeError(",
    "        'New built checksum is absent from the driver.'",
    "    )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Save patch manifest.",
    "# --------------------------------------------------------",
    "",
    "manifest_path = DRIVER.parent / (",
    "    'ast_built_grid_reference_patch.json'",
    ")",
    "",
    "manifest_data = {",
    "    'city_slug': 'hyderabad',",
    "    'action': patch_action,",
    "    'old_reference': reference_value,",
    "    'new_reference': new_reference,",
    "    'old_built_sha256': old_built_checksum,",
    "    'new_built_sha256': new_built_checksum,",
    "    'template': str(TEMPLATE),",
    "    'template_backup': str(template_backup),",
    "    'driver': str(DRIVER),",
    "    'driver_backup': str(driver_backup),",
    "    'stale_program_removed': stale_program_removed,",
    "    'population_1975_preserved': True,",
    "    'smod_1975_preserved': True,",
    "    'patched_at_utc': time.strftime(",
    "        '%Y-%m-%dT%H:%M:%SZ',",
    "        time.gmtime(),",
    "    ),",
    "}",
    "",
    "manifest_path.write_text(",
    "    json.dumps(",
    "        manifest_data,",
    "        indent=2,",
    "        sort_keys=True,",
    "    ) + '\\n',",
    "    encoding='utf-8',",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Final report.",
    "# --------------------------------------------------------",
    "",
    "print('=' * 80)",
    "print('AST BUILT-SURFACE GRID-REFERENCE PATCH')",
    "print('=' * 80)",
    "",
    "print('Action:', patch_action)",
    "print('Old reference:', reference_value)",
    "print('New reference:', new_reference)",
    "print('Template:', TEMPLATE)",
    "print('Template backup:', template_backup)",
    "print('Driver:', DRIVER)",
    "print('Driver backup:', driver_backup)",
    "print('Old built SHA-256:', old_built_checksum)",
    "print('New built SHA-256:', new_built_checksum)",
    "print(",
    "    'Stale built program removed:',",
    "    stale_program_removed,",
    ")",
    "print('Manifest:', manifest_path)",
    "",
    "print()",
    "print('AST reference validation: PASS')",
    "print('Population-grid reference configured: PASS')",
    "print('Built conservation retained: PASS')",
    "print('Built-to-population alignment retained: PASS')",
    "print('Template compilation: PASS')",
    "print('Driver compilation: PASS')",
    "print('Checksum refresh: PASS')",
    "print('Population 1975 output preserved: YES')",
    "print('SMOD 1975 output preserved: YES')",
    "print('Original Bengaluru files modified: NO')",
    "print('Spark launched: NO')",
    "print('HDFS modified: NO')",
    "print('AST BUILT REFERENCE PATCH: SUCCESS')",
]


remote_patcher = "\n".join(
    remote_lines
) + "\n"


# ============================================================
# COMPILE LOCALLY
# ============================================================

compile(
    remote_patcher,
    "ast_patch_built_grid_reference.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print("LOCAL AST BUILT PATCHER COMPILE: PASS")

print(
    "Expected SHA-256:",
    patcher_sha256,
)


# ============================================================
# TRANSFER AND RUN
# ============================================================

encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

PATCHER="/tmp/ast_patch_built_grid_reference.py"
EXPECTED_SHA256="{patcher_sha256}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 7O RESULT")
print("=" * 80)

print(
    "Return code:",
    patch_result.returncode,
)

print(
    "Population 1975 output preserved:",
    "YES",
)

print(
    "SMOD 1975 output preserved:",
    "YES",
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 7O: SUCCESS")

LOCAL AST BUILT PATCHER COMPILE: PASS
Expected SHA-256: 700b8076cbd2114fd6baaa46534d86ee948ea28cef6544795e9852d3f2534605
Remote patcher SHA-256: 700b8076cbd2114fd6baaa46534d86ee948ea28cef6544795e9852d3f2534605
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
AST BUILT-SURFACE GRID-REFERENCE PATCH
Action: grid_year_to_population_reference
Old reference: hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/grid_year/epoch=2020
New reference: hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/product=population/epoch=2020/grid_1km
Template: /home/ubuntu/ghsl_multicity/hyderabad/templates/built_surface_epoch_template.py
Template backup: /home/ubuntu/ghsl_multicity/hyderabad/templates/built_surface_epoch_template.py.before_ast_grid_reference.20261107T153742Z
Driver: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py
Driver backup: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py.before_ast_built_checksum.20261107T153742Z
Old built SHA-

In [ ]:
# ============================================================
# CELL 7P: UPDATE BUILT GENERATOR PATH COUNT
#
# The corrected built-surface template contains two active
# epoch paths:
#
#   1. Population-grid reference
#   2. Built-surface output
#
# Therefore, generate_built_program() must replace exactly
# two occurrences of:
#
#       "epoch=2020/"
#
# instead of one.
#
# Updates only the Hyderabad driver.
# Preserves completed population and SMOD 1975 outputs.
# Does not launch Spark or modify HDFS.
# ============================================================

import base64
import hashlib


# ============================================================
# VALIDATE NOTEBOOK STATE
# ============================================================

for required_name in [
    "ACTIVE_CITY",
    "ACTIVE_CITY_SLUG",
    "run_remote",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is unavailable. "
            "Rerun Cells 1, 2, and 3B."
        )


if ACTIVE_CITY_SLUG != "hyderabad":
    raise RuntimeError(
        "This patch is specifically for Hyderabad."
    )


root = ACTIVE_CITY["remote_work_dir"]

driver_path = (
    root
    + "/batch/run_city_epochs.py"
)

stale_program_path = (
    root
    + "/programs/epoch_1975/"
    + "built_surface_1975.py"
)


# ============================================================
# BUILD AST-BASED REMOTE PATCHER
# ============================================================

remote_lines = [
    "from pathlib import Path",
    "import ast",
    "import json",
    "import py_compile",
    "import shutil",
    "import time",
    "",
    f"DRIVER = Path({driver_path!r})",
    f"STALE_PROGRAM = Path({stale_program_path!r})",
    "",
    "",
    "if not DRIVER.is_file():",
    "    raise FileNotFoundError(DRIVER)",
    "",
    "",
    "# --------------------------------------------------------",
    "# Parse the Hyderabad driver.",
    "# --------------------------------------------------------",
    "",
    "driver_text = DRIVER.read_text(",
    "    encoding='utf-8'",
    ")",
    "",
    "driver_ast = ast.parse(",
    "    driver_text",
    ")",
    "",
    "built_function = None",
    "",
    "for node in driver_ast.body:",
    "    if (",
    "        isinstance(node, ast.FunctionDef)",
    "        and node.name",
    "        == 'generate_built_program'",
    "    ):",
    "        built_function = node",
    "        break",
    "",
    "if built_function is None:",
    "    raise RuntimeError(",
    "        'generate_built_program was not found.'",
    "    )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Locate the replace_exact_count() call associated with",
    "# the built-surface epoch path.",
    "# --------------------------------------------------------",
    "",
    "target_call = None",
    "count_node = None",
    "",
    "for node in ast.walk(built_function):",
    "    if not isinstance(node, ast.Call):",
    "        continue",
    "",
    "    if not (",
    "        isinstance(node.func, ast.Name)",
    "        and node.func.id",
    "        == 'replace_exact_count'",
    "    ):",
    "        continue",
    "",
    "    string_values = [",
    "        child.value",
    "        for child in ast.walk(node)",
    "        if (",
    "            isinstance(child, ast.Constant)",
    "            and isinstance(child.value, str)",
    "        )",
    "    ]",
    "",
    "    has_epoch_marker = (",
    "        '    \"epoch=2020/\"'",
    "        in string_values",
    "    )",
    "",
    "    is_built_path_call = any(",
    "        'Built' in value",
    "        and 'path' in value.lower()",
    "        for value in string_values",
    "    )",
    "",
    "    if not (",
    "        has_epoch_marker",
    "        and is_built_path_call",
    "    ):",
    "        continue",
    "",
    "    if target_call is not None:",
    "        raise RuntimeError(",
    "            'Multiple built epoch-path replacement '",
    "            'calls were found.'",
    "        )",
    "",
    "    target_call = node",
    "",
    "    # replace_exact_count arguments:",
    "    #   program",
    "    #   old marker",
    "    #   new marker",
    "    #   expected count",
    "    #   description",
    "    if len(node.args) < 4:",
    "        raise RuntimeError(",
    "            'The built path replacement call has '",
    "            'fewer than four arguments.'",
    "        )",
    "",
    "    count_node = node.args[3]",
    "",
    "",
    "if target_call is None:",
    "    raise RuntimeError(",
    "        'Built epoch-path replacement call '",
    "        'was not found.'",
    "    )",
    "",
    "if not isinstance(count_node, ast.Constant):",
    "    raise RuntimeError(",
    "        'Built path expected count is not a constant.'",
    "    )",
    "",
    "if not isinstance(count_node.value, int):",
    "    raise RuntimeError(",
    "        'Built path expected count is not an integer.'",
    "    )",
    "",
    "old_expected_count = count_node.value",
    "",
    "",
    "# --------------------------------------------------------",
    "# Patch only the integer count by AST source position.",
    "# --------------------------------------------------------",
    "",
    "if old_expected_count == 2:",
    "    patched_driver = driver_text",
    "    patch_action = 'already_two'",
    "",
    "elif old_expected_count == 1:",
    "    source_lines = driver_text.splitlines(",
    "        keepends=True",
    "    )",
    "",
    "    if (",
    "        count_node.lineno",
    "        != count_node.end_lineno",
    "    ):",
    "        raise RuntimeError(",
    "            'Expected-count integer spans multiple lines.'",
    "        )",
    "",
    "    line_index = count_node.lineno - 1",
    "    original_line = source_lines[line_index]",
    "",
    "    start_column = count_node.col_offset",
    "    end_column = count_node.end_col_offset",
    "",
    "    original_token = original_line[",
    "        start_column:end_column",
    "    ]",
    "",
    "    if original_token.strip() != '1':",
    "        raise RuntimeError(",
    "            'Unexpected built expected-count token: '",
    "            + repr(original_token)",
    "        )",
    "",
    "    source_lines[line_index] = (",
    "        original_line[:start_column]",
    "        + '2'",
    "        + original_line[end_column:]",
    "    )",
    "",
    "    patched_driver = ''.join(",
    "        source_lines",
    "    )",
    "",
    "    patch_action = 'one_to_two'",
    "",
    "else:",
    "    raise RuntimeError(",
    "        'Unexpected built path expected count: '",
    "        + str(old_expected_count)",
    "    )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Verify the patched driver using AST.",
    "# --------------------------------------------------------",
    "",
    "patched_ast = ast.parse(",
    "    patched_driver",
    ")",
    "",
    "patched_built_function = None",
    "",
    "for node in patched_ast.body:",
    "    if (",
    "        isinstance(node, ast.FunctionDef)",
    "        and node.name",
    "        == 'generate_built_program'",
    "    ):",
    "        patched_built_function = node",
    "        break",
    "",
    "if patched_built_function is None:",
    "    raise RuntimeError(",
    "        'Patched built generator was not found.'",
    "    )",
    "",
    "verified_counts = []",
    "",
    "for node in ast.walk(",
    "    patched_built_function",
    "):",
    "    if not isinstance(node, ast.Call):",
    "        continue",
    "",
    "    if not (",
    "        isinstance(node.func, ast.Name)",
    "        and node.func.id",
    "        == 'replace_exact_count'",
    "    ):",
    "        continue",
    "",
    "    string_values = [",
    "        child.value",
    "        for child in ast.walk(node)",
    "        if (",
    "            isinstance(child, ast.Constant)",
    "            and isinstance(child.value, str)",
    "        )",
    "    ]",
    "",
    "    if (",
    "        '    \"epoch=2020/\"'",
    "        not in string_values",
    "    ):",
    "        continue",
    "",
    "    if not any(",
    "        'Built' in value",
    "        and 'path' in value.lower()",
    "        for value in string_values",
    "    ):",
    "        continue",
    "",
    "    if len(node.args) < 4:",
    "        raise RuntimeError(",
    "            'Patched built call is malformed.'",
    "        )",
    "",
    "    verified_count_node = node.args[3]",
    "",
    "    if not isinstance(",
    "        verified_count_node,",
    "        ast.Constant,",
    "    ):",
    "        raise RuntimeError(",
    "            'Patched built count is not constant.'",
    "        )",
    "",
    "    verified_counts.append(",
    "        verified_count_node.value",
    "    )",
    "",
    "if verified_counts != [2]:",
    "    raise RuntimeError(",
    "        'Expected one built path replacement count '",
    "        'equal to 2, found '",
    "        + repr(verified_counts)",
    "    )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Confirm required generator logic remains.",
    "# --------------------------------------------------------",
    "",
    "required_fragments = [",
    "    'def generate_built_program(',",
    "    'replace_exact_count(',",
    "    'EXPECTED_TEMPLATE_SHA256',",
    "    'ensure_stage(',",
    "]",
    "",
    "missing_fragments = [",
    "    fragment",
    "    for fragment in required_fragments",
    "    if fragment not in patched_driver",
    "]",
    "",
    "if missing_fragments:",
    "    raise RuntimeError(",
    "        'Required driver logic is missing: '",
    "        + repr(missing_fragments)",
    "    )",
    "",
    "",
    "# --------------------------------------------------------",
    "# Compile before modifying disk.",
    "# --------------------------------------------------------",
    "",
    "compile(",
    "    patched_driver,",
    "    str(DRIVER),",
    "    'exec',",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Back up and save only the Hyderabad driver.",
    "# --------------------------------------------------------",
    "",
    "timestamp = time.strftime(",
    "    '%Y%m%dT%H%M%SZ',",
    "    time.gmtime(),",
    ")",
    "",
    "backup_path = DRIVER.with_name(",
    "    DRIVER.name",
    "    + '.before_built_path_count_patch.'",
    "    + timestamp",
    ")",
    "",
    "shutil.copy2(",
    "    DRIVER,",
    "    backup_path,",
    ")",
    "",
    "DRIVER.write_text(",
    "    patched_driver,",
    "    encoding='utf-8',",
    ")",
    "",
    "py_compile.compile(",
    "    str(DRIVER),",
    "    doraise=True,",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Remove stale generated built program.",
    "# --------------------------------------------------------",
    "",
    "stale_program_removed = False",
    "",
    "if STALE_PROGRAM.is_file():",
    "    STALE_PROGRAM.unlink()",
    "    stale_program_removed = True",
    "",
    "",
    "# --------------------------------------------------------",
    "# Save patch manifest.",
    "# --------------------------------------------------------",
    "",
    "manifest_path = DRIVER.parent / (",
    "    'built_path_count_patch.json'",
    ")",
    "",
    "manifest_data = {",
    "    'city_slug': 'hyderabad',",
    "    'action': patch_action,",
    "    'old_expected_count': old_expected_count,",
    "    'new_expected_count': 2,",
    "    'driver': str(DRIVER),",
    "    'backup': str(backup_path),",
    "    'stale_program_removed': stale_program_removed,",
    "    'population_1975_preserved': True,",
    "    'smod_1975_preserved': True,",
    "    'patched_at_utc': time.strftime(",
    "        '%Y-%m-%dT%H:%M:%SZ',",
    "        time.gmtime(),",
    "    ),",
    "}",
    "",
    "manifest_path.write_text(",
    "    json.dumps(",
    "        manifest_data,",
    "        indent=2,",
    "        sort_keys=True,",
    "    ) + '\\n',",
    "    encoding='utf-8',",
    ")",
    "",
    "",
    "# --------------------------------------------------------",
    "# Final report.",
    "# --------------------------------------------------------",
    "",
    "print('=' * 80)",
    "print('BUILT GENERATOR PATH-COUNT PATCH')",
    "print('=' * 80)",
    "",
    "print('Action:', patch_action)",
    "print('Old expected count:', old_expected_count)",
    "print('New expected count:', 2)",
    "print('Driver:', DRIVER)",
    "print('Backup:', backup_path)",
    "print(",
    "    'Stale built program removed:',",
    "    stale_program_removed,",
    ")",
    "print('Manifest:', manifest_path)",
    "",
    "print()",
    "print('AST path-count verification: PASS')",
    "print('Built generator retained: PASS')",
    "print('Driver compilation: PASS')",
    "print('Population 1975 output preserved: YES')",
    "print('SMOD 1975 output preserved: YES')",
    "print('Original Bengaluru driver modified: NO')",
    "print('Spark launched: NO')",
    "print('HDFS modified: NO')",
    "print('BUILT PATH COUNT PATCH: SUCCESS')",
]


remote_patcher = "\n".join(
    remote_lines
) + "\n"


# ============================================================
# COMPILE LOCALLY
# ============================================================

compile(
    remote_patcher,
    "patch_built_path_count.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print("LOCAL BUILT COUNT PATCHER COMPILE: PASS")

print(
    "Expected SHA-256:",
    patcher_sha256,
)


# ============================================================
# TRANSFER AND RUN
# ============================================================

encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

PATCHER="/tmp/patch_built_path_count.py"
EXPECTED_SHA256="{patcher_sha256}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 7P RESULT")
print("=" * 80)

print(
    "Return code:",
    patch_result.returncode,
)

print(
    "Population 1975 output preserved:",
    "YES",
)

print(
    "SMOD 1975 output preserved:",
    "YES",
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 7P: SUCCESS")

LOCAL BUILT COUNT PATCHER COMPILE: PASS
Expected SHA-256: de593befdc6a8437d6a7f4a5a7381acc7c5b2a1c30f056d703407e676341408b
Remote patcher SHA-256: de593befdc6a8437d6a7f4a5a7381acc7c5b2a1c30f056d703407e676341408b
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
BUILT GENERATOR PATH-COUNT PATCH
Action: one_to_two
Old expected count: 1
New expected count: 2
Driver: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py
Backup: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py.before_built_path_count_patch.20261107T154225Z
Stale built program removed: False
Manifest: /home/ubuntu/ghsl_multicity/hyderabad/batch/built_path_count_patch.json

AST path-count verification: PASS
Built generator retained: PASS
Driver compilation: PASS
Population 1975 output preserved: YES
SMOD 1975 output preserved: YES
Original Bengaluru driver modified: NO
Spark launched: NO
HDFS modified: NO
BUILT PATH COUNT PATCH: SUCCESS

REMOTE STDERR:


CELL 7P RESULT
Return code: 0
Population

In [ ]:
# ============================================================
# CELL 7Q: REMOVE OBSOLETE POPULATION 1975 REQUIREMENT
#
# Updates only Hyderabad's run_city_epochs.py.
# Preserves every completed 1975 HDFS output.
# Does not launch Spark.
# Does not modify HDFS.
# ============================================================

import ast
import base64
import hashlib
import json
import textwrap


root = ACTIVE_CITY["remote_work_dir"]

driver_path = (
    root
    + "/batch/run_city_epochs.py"
)

stale_program_path = (
    root
    + "/programs/epoch_1980/population_1980.py"
)


remote_patcher = textwrap.dedent(
    """
    from pathlib import Path
    import ast
    import json
    import py_compile
    import shutil
    import sys
    import time


    DRIVER = Path(sys.argv[1])
    STALE_PROGRAM = Path(sys.argv[2])

    OBSOLETE_VALUE = '    "epoch=1975/"'
    PATCH_MARKER = (
        "obsolete population 1975 requirement removed"
    )


    if not DRIVER.is_file():
        raise FileNotFoundError(DRIVER)


    source = DRIVER.read_text(
        encoding="utf-8"
    )

    tree = ast.parse(source)


    # --------------------------------------------------------
    # Find generate_population_program().
    # --------------------------------------------------------

    population_function = None

    for node in tree.body:
        if (
            isinstance(node, ast.FunctionDef)
            and node.name
            == "generate_population_program"
        ):
            population_function = node
            break

    if population_function is None:
        raise RuntimeError(
            "generate_population_program was not found."
        )


    # --------------------------------------------------------
    # Find the obsolete literal inside the population
    # require_fragments() list.
    # --------------------------------------------------------

    obsolete_nodes = []

    for node in ast.walk(
        population_function
    ):
        if (
            isinstance(node, ast.Constant)
            and node.value == OBSOLETE_VALUE
        ):
            obsolete_nodes.append(node)


    if len(obsolete_nodes) == 1:
        obsolete_node = obsolete_nodes[0]
        action = "removed_by_ast"

    elif (
        len(obsolete_nodes) == 0
        and PATCH_MARKER in source
    ):
        obsolete_node = None
        action = "already_removed"

    else:
        raise RuntimeError(
            "Expected one obsolete requirement, found "
            + str(len(obsolete_nodes))
        )


    # --------------------------------------------------------
    # Remove the complete source line containing the literal.
    # --------------------------------------------------------

    if obsolete_node is not None:
        lines = source.splitlines(
            keepends=True
        )

        start_index = (
            obsolete_node.lineno - 1
        )

        end_index = (
            obsolete_node.end_lineno
        )

        original_line = lines[
            start_index
        ]

        if "epoch=1975/" not in original_line:
            raise RuntimeError(
                "AST found the literal, but its source "
                "line did not contain epoch=1975/."
            )

        replacement = [
            "                # Generalized multi-city "
            "pipeline:\\n",
            "                # obsolete population 1975 "
            "requirement removed.\\n",
        ]

        lines[
            start_index:end_index
        ] = replacement

        patched_source = "".join(lines)

    else:
        patched_source = source


    # --------------------------------------------------------
    # Verify the obsolete requirement is absent specifically
    # from generate_population_program().
    # --------------------------------------------------------

    patched_tree = ast.parse(
        patched_source
    )

    patched_function = None

    for node in patched_tree.body:
        if (
            isinstance(node, ast.FunctionDef)
            and node.name
            == "generate_population_program"
        ):
            patched_function = node
            break

    if patched_function is None:
        raise RuntimeError(
            "Patched population generator is missing."
        )

    remaining = [
        node.lineno
        for node in ast.walk(
            patched_function
        )
        if (
            isinstance(node, ast.Constant)
            and node.value
            == OBSOLETE_VALUE
        )
    ]

    if remaining:
        raise RuntimeError(
            "Obsolete requirements remain at lines: "
            + repr(remaining)
        )


    # --------------------------------------------------------
    # Confirm essential population generation remains.
    # --------------------------------------------------------

    required_markers = [
        "def generate_population_program(",
        "Population output path",
        "Population active epoch",
        "Population catalog view",
        "Population catalog SQL reference",
        "RS_FromGeoTiff",
        "RS_Tile",
        "RS_BandAsArray",
        ".parquet(OUTPUT_PATH)",
    ]

    missing = [
        marker
        for marker in required_markers
        if marker not in patched_source
    ]

    if missing:
        raise RuntimeError(
            "Required population logic is missing: "
            + repr(missing)
        )


    # --------------------------------------------------------
    # Compile before writing.
    # --------------------------------------------------------

    compile(
        patched_source,
        str(DRIVER),
        "exec",
    )


    # --------------------------------------------------------
    # Back up and save only the Hyderabad driver.
    # --------------------------------------------------------

    timestamp = time.strftime(
        "%Y%m%dT%H%M%SZ",
        time.gmtime(),
    )

    backup_path = DRIVER.with_name(
        DRIVER.name
        + ".before_population_1975_requirement_patch."
        + timestamp
    )

    shutil.copy2(
        DRIVER,
        backup_path,
    )

    DRIVER.write_text(
        patched_source,
        encoding="utf-8",
    )

    py_compile.compile(
        str(DRIVER),
        doraise=True,
    )


    # --------------------------------------------------------
    # Remove an incomplete generated 1980 program if present.
    # --------------------------------------------------------

    stale_removed = False

    if STALE_PROGRAM.is_file():
        STALE_PROGRAM.unlink()
        stale_removed = True


    # --------------------------------------------------------
    # Save patch metadata.
    # --------------------------------------------------------

    manifest_path = (
        DRIVER.parent
        / "population_1975_requirement_patch.json"
    )

    manifest_path.write_text(
        json.dumps(
            {
                "city_slug": "hyderabad",
                "action": action,
                "driver": str(DRIVER),
                "backup": str(backup_path),
                "stale_population_1980_removed":
                    stale_removed,
                "epoch_1975_complete_preserved":
                    True,
                "patched_at_utc": time.strftime(
                    "%Y-%m-%dT%H:%M:%SZ",
                    time.gmtime(),
                ),
            },
            indent=2,
            sort_keys=True,
        )
        + "\\n",
        encoding="utf-8",
    )


    print("=" * 80)
    print("POPULATION 1975 REQUIREMENT PATCH")
    print("=" * 80)

    print("Action:", action)
    print("Driver:", DRIVER)
    print("Backup:", backup_path)
    print(
        "Stale population 1980 removed:",
        stale_removed,
    )
    print("Manifest:", manifest_path)

    print()
    print(
        "Obsolete 1975 requirement removed: PASS"
    )
    print("Population generator retained: PASS")
    print("Driver compilation: PASS")
    print("Hyderabad epoch 1975 preserved: YES")
    print("Original Bengaluru driver modified: NO")
    print("Spark launched: NO")
    print("HDFS modified: NO")
    print(
        "POPULATION REQUIREMENT PATCH: SUCCESS"
    )
    """
).strip() + "\n"


# ============================================================
# LOCAL VALIDATION
# ============================================================

compile(
    remote_patcher,
    "patch_population_1975_requirement.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print(
    "LOCAL POPULATION REQUIREMENT PATCHER "
    "COMPILE: PASS"
)

print(
    "Expected SHA-256:",
    patcher_sha256,
)


# ============================================================
# TRANSFER AND RUN
# ============================================================

encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

PATCHER="/tmp/patch_population_1975_requirement.py"
EXPECTED_SHA256="{patcher_sha256}"
DRIVER="{driver_path}"
STALE_PROGRAM="{stale_program_path}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}" \
    "${{DRIVER}}" \
    "${{STALE_PROGRAM}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 7Q RESULT")
print("=" * 80)

print(
    "Return code:",
    patch_result.returncode,
)

print(
    "Hyderabad epoch 1975 preserved:",
    "YES",
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 7Q: SUCCESS")

LOCAL POPULATION REQUIREMENT PATCHER COMPILE: PASS
Expected SHA-256: 5c034466bb9f3227c7f51f65c028c18a0fd02fa8bbd4afa84337b2ad9433a382
Remote patcher SHA-256: 5c034466bb9f3227c7f51f65c028c18a0fd02fa8bbd4afa84337b2ad9433a382
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
POPULATION 1975 REQUIREMENT PATCH
Action: removed_by_ast
Driver: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py
Backup: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py.before_population_1975_requirement_patch.20261107T155546Z
Stale population 1980 removed: False
Manifest: /home/ubuntu/ghsl_multicity/hyderabad/batch/population_1975_requirement_patch.json

Obsolete 1975 requirement removed: PASS
Population generator retained: PASS
Driver compilation: PASS
Hyderabad epoch 1975 preserved: YES
Original Bengaluru driver modified: NO
Spark launched: NO
HDFS modified: NO
POPULATION REQUIREMENT PATCH: SUCCESS

REMOTE STDERR:


CELL 7Q RESULT
Return code: 0
Hyderabad epoch 1975 preserved:

In [8]:
# ============================================================
# CELL 7R: ROBUST AST FIX FOR POPULATION 2020 GUARD
#
# Changes only Hyderabad's run_city_epochs.py.
#
# Old:
#   if "EPOCH = 2020" in program:
#
# New:
#   if (
#       epoch != 2020
#       and "EPOCH = 2020" in program
#   ):
#
# Completed Hyderabad outputs from 1975-2015 remain untouched.
# No Spark job is launched.
# No HDFS data is modified.
# ============================================================

import base64
import hashlib
import textwrap


root = ACTIVE_CITY["remote_work_dir"]

driver_path = (
    root
    + "/batch/run_city_epochs.py"
)


remote_patcher = textwrap.dedent(
    """
    from pathlib import Path
    import ast
    import json
    import py_compile
    import shutil
    import sys
    import time


    DRIVER = Path(sys.argv[1])

    if not DRIVER.is_file():
        raise FileNotFoundError(DRIVER)


    original_source = DRIVER.read_text(
        encoding="utf-8"
    )

    original_tree = ast.parse(
        original_source
    )


    # --------------------------------------------------------
    # Find generate_population_program().
    # --------------------------------------------------------

    population_function = None

    for node in original_tree.body:
        if (
            isinstance(node, ast.FunctionDef)
            and node.name
            == "generate_population_program"
        ):
            population_function = node
            break

    if population_function is None:
        raise RuntimeError(
            "generate_population_program was not found."
        )


    # --------------------------------------------------------
    # Identify the guard through its RuntimeError message.
    # --------------------------------------------------------

    target_guard = None

    for node in ast.walk(
        population_function
    ):
        if not isinstance(node, ast.If):
            continue

        has_target_message = False

        for descendant in ast.walk(node):
            if not isinstance(
                descendant,
                ast.Raise,
            ):
                continue

            if descendant.exc is None:
                continue

            messages = [
                value.value
                for value in ast.walk(
                    descendant.exc
                )
                if (
                    isinstance(
                        value,
                        ast.Constant,
                    )
                    and isinstance(
                        value.value,
                        str,
                    )
                )
            ]

            if any(
                "retains active 2020"
                in message
                for message in messages
            ):
                has_target_message = True
                break

        if has_target_message:
            if target_guard is not None:
                raise RuntimeError(
                    "Multiple active-2020 population "
                    "guards were found."
                )

            target_guard = node


    if target_guard is None:
        raise RuntimeError(
            "The population active-2020 guard "
            "was not found."
        )


    # --------------------------------------------------------
    # Determine whether it is already generalized.
    # --------------------------------------------------------

    def is_epoch_not_2020(node):
        return (
            isinstance(node, ast.Compare)
            and isinstance(node.left, ast.Name)
            and node.left.id == "epoch"
            and len(node.ops) == 1
            and isinstance(
                node.ops[0],
                ast.NotEq,
            )
            and len(
                node.comparators
            ) == 1
            and isinstance(
                node.comparators[0],
                ast.Constant,
            )
            and node.comparators[0].value
            == 2020
        )


    def is_program_contains_active_2020(node):
        return (
            isinstance(node, ast.Compare)
            and isinstance(
                node.left,
                ast.Constant,
            )
            and node.left.value
            == "EPOCH = 2020"
            and len(node.ops) == 1
            and isinstance(
                node.ops[0],
                ast.In,
            )
            and len(
                node.comparators
            ) == 1
            and isinstance(
                node.comparators[0],
                ast.Name,
            )
            and node.comparators[0].id
            == "program"
        )


    already_generalized = False

    if (
        isinstance(
            target_guard.test,
            ast.BoolOp,
        )
        and isinstance(
            target_guard.test.op,
            ast.And,
        )
    ):
        has_epoch_condition = any(
            is_epoch_not_2020(condition)
            for condition
            in target_guard.test.values
        )

        has_program_condition = any(
            is_program_contains_active_2020(
                condition
            )
            for condition
            in target_guard.test.values
        )

        already_generalized = (
            has_epoch_condition
            and has_program_condition
        )


    # --------------------------------------------------------
    # Replace the guard condition structurally.
    # --------------------------------------------------------

    if already_generalized:
        action = "already_generalized"

    else:
        if not is_program_contains_active_2020(
            target_guard.test
        ):
            current_test = ast.unparse(
                target_guard.test
            )

            raise RuntimeError(
                "Unexpected original guard condition: "
                + current_test
            )

        target_guard.test = ast.BoolOp(
            op=ast.And(),
            values=[
                ast.Compare(
                    left=ast.Name(
                        id="epoch",
                        ctx=ast.Load(),
                    ),
                    ops=[
                        ast.NotEq(),
                    ],
                    comparators=[
                        ast.Constant(
                            value=2020
                        ),
                    ],
                ),
                ast.Compare(
                    left=ast.Constant(
                        value="EPOCH = 2020"
                    ),
                    ops=[
                        ast.In(),
                    ],
                    comparators=[
                        ast.Name(
                            id="program",
                            ctx=ast.Load(),
                        ),
                    ],
                ),
            ],
        )

        action = "guard_generalized_by_ast"


    ast.fix_missing_locations(
        original_tree
    )


    # --------------------------------------------------------
    # Generate valid Python source from the transformed AST.
    # --------------------------------------------------------

    patched_source = (
        ast.unparse(original_tree)
        + "\\n"
    )

    compile(
        patched_source,
        str(DRIVER),
        "exec",
    )


    # --------------------------------------------------------
    # Verify the transformed source through a fresh AST parse.
    # --------------------------------------------------------

    verification_tree = ast.parse(
        patched_source
    )

    verified = False

    for function_node in verification_tree.body:
        if not (
            isinstance(
                function_node,
                ast.FunctionDef,
            )
            and function_node.name
            == "generate_population_program"
        ):
            continue

        for node in ast.walk(
            function_node
        ):
            if not isinstance(
                node,
                ast.If,
            ):
                continue

            messages = [
                value.value
                for value in ast.walk(node)
                if (
                    isinstance(
                        value,
                        ast.Constant,
                    )
                    and isinstance(
                        value.value,
                        str,
                    )
                )
            ]

            if not any(
                "retains active 2020"
                in message
                for message in messages
            ):
                continue

            if not (
                isinstance(
                    node.test,
                    ast.BoolOp,
                )
                and isinstance(
                    node.test.op,
                    ast.And,
                )
            ):
                raise RuntimeError(
                    "The saved guard is not an AND."
                )

            has_epoch_condition = any(
                is_epoch_not_2020(condition)
                for condition
                in node.test.values
            )

            has_program_condition = any(
                is_program_contains_active_2020(
                    condition
                )
                for condition
                in node.test.values
            )

            if not (
                has_epoch_condition
                and has_program_condition
            ):
                raise RuntimeError(
                    "The transformed guard does not "
                    "contain both required conditions."
                )

            verified = True


    if not verified:
        raise RuntimeError(
            "The generalized population guard "
            "could not be verified."
        )


    # --------------------------------------------------------
    # Confirm essential driver functionality remains.
    # --------------------------------------------------------

    required_names = [
        "generate_population_program",
        "generate_smod_program",
        "generate_built_program",
        "ensure_stage",
        "verify_template",
        "replace_exact_count",
    ]

    verification_namespace = {
        node.name
        for node in verification_tree.body
        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        )
    }

    missing_names = [
        name
        for name in required_names
        if name
        not in verification_namespace
    ]

    if missing_names:
        raise RuntimeError(
            "Required driver functions are missing: "
            + repr(missing_names)
        )


    # --------------------------------------------------------
    # Back up and save only Hyderabad's driver.
    # --------------------------------------------------------

    timestamp = time.strftime(
        "%Y%m%dT%H%M%SZ",
        time.gmtime(),
    )

    backup_path = DRIVER.with_name(
        DRIVER.name
        + ".before_robust_2020_guard_patch."
        + timestamp
    )

    shutil.copy2(
        DRIVER,
        backup_path,
    )

    DRIVER.write_text(
        patched_source,
        encoding="utf-8",
    )

    py_compile.compile(
        str(DRIVER),
        doraise=True,
    )


    # --------------------------------------------------------
    # Remove any incomplete generated 2020 program.
    # --------------------------------------------------------

    stale_program = (
        DRIVER.parent.parent
        / "programs"
        / "epoch_2020"
        / "population_2020.py"
    )

    stale_removed = False

    if stale_program.is_file():
        stale_program.unlink()
        stale_removed = True


    # --------------------------------------------------------
    # Save patch metadata.
    # --------------------------------------------------------

    manifest_path = (
        DRIVER.parent
        / "robust_population_2020_guard_patch.json"
    )

    manifest_path.write_text(
        json.dumps(
            {
                "city_slug": "hyderabad",
                "action": action,
                "driver": str(DRIVER),
                "backup": str(backup_path),
                "stale_population_2020_removed":
                    stale_removed,
                "completed_epochs_preserved": [
                    1975,
                    1980,
                    1985,
                    1990,
                    1995,
                    2000,
                    2005,
                    2010,
                    2015,
                ],
                "patched_at_utc": time.strftime(
                    "%Y-%m-%dT%H:%M:%SZ",
                    time.gmtime(),
                ),
            },
            indent=2,
            sort_keys=True,
        )
        + "\\n",
        encoding="utf-8",
    )


    print("=" * 80)
    print("ROBUST POPULATION 2020 GUARD PATCH")
    print("=" * 80)

    print("Action:", action)
    print("Driver:", DRIVER)
    print("Backup:", backup_path)
    print(
        "Stale population 2020 removed:",
        stale_removed,
    )
    print("Manifest:", manifest_path)

    print()
    print("2020 population epoch allowed: PASS")
    print("Non-2020 protection retained: PASS")
    print("AST transformation: PASS")
    print("Fresh AST verification: PASS")
    print("Driver compilation: PASS")
    print("Required driver functions retained: PASS")
    print("Completed epochs preserved: 1975-2015")
    print("Original Bengaluru driver modified: NO")
    print("Spark launched: NO")
    print("HDFS modified: NO")
    print(
        "ROBUST POPULATION 2020 GUARD: SUCCESS"
    )
    """
).strip() + "\n"


# ============================================================
# LOCAL VALIDATION
# ============================================================

compile(
    remote_patcher,
    "robust_patch_population_2020_guard.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print(
    "LOCAL ROBUST 2020 PATCHER COMPILE: PASS"
)

print(
    "Expected SHA-256:",
    patcher_sha256,
)


# ============================================================
# TRANSFER AND RUN
# ============================================================

encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

PATCHER="/tmp/robust_patch_population_2020_guard.py"
EXPECTED_SHA256="{patcher_sha256}"
DRIVER="{driver_path}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}" "${{DRIVER}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 7R RESULT")
print("=" * 80)

print(
    "Return code:",
    patch_result.returncode,
)

print(
    "Completed epochs preserved:",
    "1975-2015",
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 7R: SUCCESS")

LOCAL ROBUST 2020 PATCHER COMPILE: PASS
Expected SHA-256: ac4df09e3f907576a6c7b8abc927a0bb7ed0003c1a4eaa7d5a3365ec727e631e
Remote patcher SHA-256: ac4df09e3f907576a6c7b8abc927a0bb7ed0003c1a4eaa7d5a3365ec727e631e
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
ROBUST POPULATION 2020 GUARD PATCH
Action: guard_generalized_by_ast
Driver: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py
Backup: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py.before_robust_2020_guard_patch.20261108T105139Z
Stale population 2020 removed: False
Manifest: /home/ubuntu/ghsl_multicity/hyderabad/batch/robust_population_2020_guard_patch.json

2020 population epoch allowed: PASS
Non-2020 protection retained: PASS
AST transformation: PASS
Fresh AST verification: PASS
Driver compilation: PASS
Required driver functions retained: PASS
Completed epochs preserved: 1975-2015
Original Bengaluru driver modified: NO
Spark launched: NO
HDFS modified: NO
ROBUST POPULATION 2020 GUARD: SUCC

In [20]:
# ============================================================
# CELL 7S: GENERALIZE ALL REMAINING 2020 GUARDS
#
# Correct rule:
#
#   For epochs before 2020:
#       reject a generated program that accidentally retains
#       active 2020 configuration.
#
#   For the actual 2020 epoch:
#       EPOCH = 2020 is correct and must be allowed.
#
# This cell scans every generator in Hyderabad's driver and
# patches every guard whose error contains:
#
#       "retains active 2020"
#
# Population may already be patched. SMOD and built surface
# are patched if needed.
#
# Preserves all completed HDFS outputs:
#   - all outputs for 1975-2015
#   - population 2020
#
# Does not launch Spark or modify HDFS.
# ============================================================

import base64
import hashlib
import textwrap


root = ACTIVE_CITY["remote_work_dir"]

driver_path = (
    root
    + "/batch/run_city_epochs.py"
)


remote_patcher = textwrap.dedent(
    """
    from pathlib import Path
    import ast
    import json
    import py_compile
    import shutil
    import sys
    import time


    DRIVER = Path(sys.argv[1])

    if not DRIVER.is_file():
        raise FileNotFoundError(DRIVER)


    source = DRIVER.read_text(
        encoding="utf-8"
    )

    tree = ast.parse(source)


    # --------------------------------------------------------
    # Structural helpers.
    # --------------------------------------------------------

    def contains_active_2020_condition(node):
        return (
            isinstance(node, ast.Compare)
            and isinstance(
                node.left,
                ast.Constant,
            )
            and node.left.value
            == "EPOCH = 2020"
            and len(node.ops) == 1
            and isinstance(
                node.ops[0],
                ast.In,
            )
            and len(
                node.comparators
            ) == 1
            and isinstance(
                node.comparators[0],
                ast.Name,
            )
            and node.comparators[0].id
            == "program"
        )


    def epoch_not_2020_condition(node):
        return (
            isinstance(node, ast.Compare)
            and isinstance(
                node.left,
                ast.Name,
            )
            and node.left.id == "epoch"
            and len(node.ops) == 1
            and isinstance(
                node.ops[0],
                ast.NotEq,
            )
            and len(
                node.comparators
            ) == 1
            and isinstance(
                node.comparators[0],
                ast.Constant,
            )
            and node.comparators[0].value
            == 2020
        )


    def has_retains_active_message(if_node):
        messages = [
            value.value
            for value in ast.walk(if_node)
            if (
                isinstance(
                    value,
                    ast.Constant,
                )
                and isinstance(
                    value.value,
                    str,
                )
            )
        ]

        return any(
            "retains active 2020"
            in message
            for message in messages
        )


    # --------------------------------------------------------
    # Find and patch all relevant guards.
    # --------------------------------------------------------

    patch_report = []

    for function_node in tree.body:
        if not isinstance(
            function_node,
            ast.FunctionDef,
        ):
            continue

        for node in ast.walk(
            function_node
        ):
            if not isinstance(node, ast.If):
                continue

            if not has_retains_active_message(
                node
            ):
                continue

            already_generalized = False

            if (
                isinstance(
                    node.test,
                    ast.BoolOp,
                )
                and isinstance(
                    node.test.op,
                    ast.And,
                )
            ):
                has_epoch_condition = any(
                    epoch_not_2020_condition(
                        condition
                    )
                    for condition
                    in node.test.values
                )

                has_program_condition = any(
                    contains_active_2020_condition(
                        condition
                    )
                    for condition
                    in node.test.values
                )

                already_generalized = (
                    has_epoch_condition
                    and has_program_condition
                )

            if already_generalized:
                patch_report.append(
                    {
                        "function":
                            function_node.name,
                        "action":
                            "already_generalized",
                    }
                )

                continue

            if not contains_active_2020_condition(
                node.test
            ):
                raise RuntimeError(
                    "Unexpected active-2020 condition "
                    "inside "
                    + function_node.name
                    + ": "
                    + ast.unparse(node.test)
                )

            node.test = ast.BoolOp(
                op=ast.And(),
                values=[
                    ast.Compare(
                        left=ast.Name(
                            id="epoch",
                            ctx=ast.Load(),
                        ),
                        ops=[
                            ast.NotEq(),
                        ],
                        comparators=[
                            ast.Constant(
                                value=2020
                            ),
                        ],
                    ),
                    ast.Compare(
                        left=ast.Constant(
                            value="EPOCH = 2020"
                        ),
                        ops=[
                            ast.In(),
                        ],
                        comparators=[
                            ast.Name(
                                id="program",
                                ctx=ast.Load(),
                            ),
                        ],
                    ),
                ],
            )

            patch_report.append(
                {
                    "function":
                        function_node.name,
                    "action":
                        "generalized",
                }
            )


    if not patch_report:
        raise RuntimeError(
            "No 'retains active 2020' guards "
            "were found."
        )


    # --------------------------------------------------------
    # Generate valid source from transformed AST.
    # --------------------------------------------------------

    ast.fix_missing_locations(tree)

    patched_source = (
        ast.unparse(tree)
        + "\\n"
    )

    compile(
        patched_source,
        str(DRIVER),
        "exec",
    )


    # --------------------------------------------------------
    # Verify every retained guard is generalized.
    # --------------------------------------------------------

    verification_tree = ast.parse(
        patched_source
    )

    verified_guards = []

    for function_node in (
        verification_tree.body
    ):
        if not isinstance(
            function_node,
            ast.FunctionDef,
        ):
            continue

        for node in ast.walk(
            function_node
        ):
            if not isinstance(node, ast.If):
                continue

            if not has_retains_active_message(
                node
            ):
                continue

            if not (
                isinstance(
                    node.test,
                    ast.BoolOp,
                )
                and isinstance(
                    node.test.op,
                    ast.And,
                )
            ):
                raise RuntimeError(
                    "Guard in "
                    + function_node.name
                    + " is not an AND condition."
                )

            has_epoch_condition = any(
                epoch_not_2020_condition(
                    condition
                )
                for condition
                in node.test.values
            )

            has_program_condition = any(
                contains_active_2020_condition(
                    condition
                )
                for condition
                in node.test.values
            )

            if not (
                has_epoch_condition
                and has_program_condition
            ):
                raise RuntimeError(
                    "Guard in "
                    + function_node.name
                    + " is not fully generalized."
                )

            verified_guards.append(
                function_node.name
            )


    if not verified_guards:
        raise RuntimeError(
            "No generalized guards were verified."
        )


    # --------------------------------------------------------
    # Confirm core production functions remain.
    # --------------------------------------------------------

    function_names = {
        node.name
        for node in verification_tree.body
        if isinstance(
            node,
            ast.FunctionDef,
        )
    }

    required_functions = {
        "generate_population_program",
        "generate_smod_program",
        "generate_built_program",
        "ensure_stage",
        "verify_template",
        "replace_exact_count",
    }

    missing_functions = sorted(
        required_functions
        - function_names
    )

    if missing_functions:
        raise RuntimeError(
            "Required driver functions are missing: "
            + repr(missing_functions)
        )


    # --------------------------------------------------------
    # Back up and save only Hyderabad's driver.
    # --------------------------------------------------------

    timestamp = time.strftime(
        "%Y%m%dT%H%M%SZ",
        time.gmtime(),
    )

    backup_path = DRIVER.with_name(
        DRIVER.name
        + ".before_all_2020_guard_patch."
        + timestamp
    )

    shutil.copy2(
        DRIVER,
        backup_path,
    )

    DRIVER.write_text(
        patched_source,
        encoding="utf-8",
    )

    py_compile.compile(
        str(DRIVER),
        doraise=True,
    )


    # --------------------------------------------------------
    # Remove incomplete generated 2020 programs only.
    #
    # Population 2020 output is preserved in HDFS. Its local
    # program may be regenerated, but checkpoint validation
    # will skip population recomputation.
    # --------------------------------------------------------

    epoch_2020_dir = (
        DRIVER.parent.parent
        / "programs"
        / "epoch_2020"
    )

    removed_programs = []

    for file_name in [
        "smod_2020.py",
        "built_surface_2020.py",
        "enrich_2020.py",
    ]:
        program_path = (
            epoch_2020_dir
            / file_name
        )

        if program_path.is_file():
            program_path.unlink()

            removed_programs.append(
                str(program_path)
            )


    # --------------------------------------------------------
    # Save patch metadata.
    # --------------------------------------------------------

    manifest_path = (
        DRIVER.parent
        / "all_2020_guards_patch.json"
    )

    manifest_path.write_text(
        json.dumps(
            {
                "city_slug": "hyderabad",
                "driver": str(DRIVER),
                "backup": str(backup_path),
                "patch_report": patch_report,
                "verified_guards":
                    verified_guards,
                "removed_incomplete_programs":
                    removed_programs,
                "completed_outputs_preserved": {
                    "epochs_1975_to_2015":
                        "all_four_stages",
                    "population_2020":
                        True,
                },
                "patched_at_utc":
                    time.strftime(
                        "%Y-%m-%dT%H:%M:%SZ",
                        time.gmtime(),
                    ),
            },
            indent=2,
            sort_keys=True,
        )
        + "\\n",
        encoding="utf-8",
    )


    print("=" * 80)
    print("ALL 2020 GUARDS PATCH")
    print("=" * 80)

    print("Driver:", DRIVER)
    print("Backup:", backup_path)

    print()
    print("Guard report:")

    for item in patch_report:
        print(
            " ",
            item["function"],
            "->",
            item["action"],
        )

    print()
    print(
        "Verified guards:",
        verified_guards,
    )

    print(
        "Removed incomplete programs:",
        removed_programs,
    )

    print("Manifest:", manifest_path)

    print()
    print("All 2020 guards generalized: PASS")
    print("Non-2020 protection retained: PASS")
    print("Driver compilation: PASS")
    print("Required driver functions retained: PASS")
    print("Outputs through 2015 preserved: YES")
    print("Population 2020 output preserved: YES")
    print("Original Bengaluru driver modified: NO")
    print("Spark launched: NO")
    print("HDFS modified: NO")
    print("ALL 2020 GUARDS PATCH: SUCCESS")
    """
).strip() + "\n"


# ============================================================
# LOCAL VALIDATION
# ============================================================

compile(
    remote_patcher,
    "patch_all_2020_guards.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print(
    "LOCAL ALL-GUARDS PATCHER COMPILE: PASS"
)

print(
    "Expected SHA-256:",
    patcher_sha256,
)


# ============================================================
# TRANSFER AND RUN
# ============================================================

encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

PATCHER="/tmp/patch_all_2020_guards.py"
EXPECTED_SHA256="{patcher_sha256}"
DRIVER="{driver_path}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}" "${{DRIVER}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 7S RESULT")
print("=" * 80)

print(
    "Return code:",
    patch_result.returncode,
)

print(
    "Completed outputs preserved:",
    "1975-2015 plus population 2020",
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 7S: SUCCESS")

LOCAL ALL-GUARDS PATCHER COMPILE: PASS
Expected SHA-256: c22e6a729cc313e4968a1c9a7e0c5a33f34a6900807eda95d4019653c648086f
Remote patcher SHA-256: c22e6a729cc313e4968a1c9a7e0c5a33f34a6900807eda95d4019653c648086f
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
ALL 2020 GUARDS PATCH
Driver: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py
Backup: /home/ubuntu/ghsl_multicity/hyderabad/batch/run_city_epochs.py.before_all_2020_guard_patch.20261108T111310Z

Guard report:
  generate_population_program -> already_generalized
  generate_smod_program -> generalized
  generate_built_program -> generalized

Verified guards: ['generate_population_program', 'generate_smod_program', 'generate_built_program']
Removed incomplete programs: []
Manifest: /home/ubuntu/ghsl_multicity/hyderabad/batch/all_2020_guards_patch.json

All 2020 guards generalized: PASS
Non-2020 protection retained: PASS
Driver compilation: PASS
Required driver functions retained: PASS
Outputs through 2015 pre

In [30]:
# ============================================================
# CELL 8: MONITOR DETACHED MULTI-CITY JOB
#
# Read-only:
#   - Does not launch another process
#   - Does not stop the running process
#   - Does not modify HDFS
#
# Prints job status at both:
#   1. Beginning of output
#   2. End of output
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]
slug = ACTIVE_CITY_SLUG
city_name = ACTIVE_CITY["city_name"]

monitor_command = f"""
set +e

PID_FILE="{root}/batch/city.pid"
LOG_FILE="{root}/batch/city.log"
MANIFEST="{root}/batch/batch_manifest.tsv"

get_status() {{
    JOB_STATUS="UNKNOWN"
    RECORDED_PID="N/A"

    if [ -f "$PID_FILE" ]; then
        RECORDED_PID="$(cat "$PID_FILE")"

        if kill -0 "$RECORDED_PID" 2>/dev/null; then
            JOB_STATUS="RUNNING"
        else
            JOB_STATUS="STOPPED"
        fi
    fi
}}

print_status() {{
    get_status

    echo "============================================================"
    echo "HYDERABAD JOB STATUS"
    echo "============================================================"
    echo "Recorded PID: $RECORDED_PID"
    echo "STATUS=$JOB_STATUS"

    if [ "$JOB_STATUS" = "RUNNING" ]; then
        ps -p "$RECORDED_PID" \
            -o pid,etime,%cpu,%mem,cmd
    fi
}}

# ============================================================
# STATUS AT BEGINNING
# ============================================================

print_status

echo
echo "============================================================"
echo "LATEST 150 LOG LINES"
echo "============================================================"

if [ -f "$LOG_FILE" ]; then
    tail -n 150 "$LOG_FILE"
else
    echo "Log file is missing:"
    echo "$LOG_FILE"
fi

echo
echo "============================================================"
echo "CURRENT MANIFEST"
echo "============================================================"

if [ -f "$MANIFEST" ]; then
    cat "$MANIFEST"
else
    echo "Manifest has not been created yet."
fi

echo
echo "============================================================"
echo "CURRENT HYDERABAD HDFS OUTPUTS"
echo "============================================================"

hdfs dfs -find \
    "/data/ghsl/curated/cities/city={slug}" \
    2>/dev/null \
    | grep -E 'epoch=[0-9]{{4}}' \
    | sort \
    | tail -100

echo
echo "============================================================"
echo "ACTIVE SPARK / JAVA PROCESSES"
echo "============================================================"

jps

echo
echo "============================================================"
echo "FINAL HYDERABAD JOB STATUS"
echo "============================================================"

# Recheck status because it may have changed while the log
# and HDFS inventory were being printed.

get_status

echo "City: {city_name}"
echo "City slug: {slug}"
echo "Recorded PID: $RECORDED_PID"
echo "FINAL_STATUS=$JOB_STATUS"

if [ "$JOB_STATUS" = "RUNNING" ]; then
    echo
    echo "Remote driver process:"
    ps -p "$RECORDED_PID" \
        -o pid,etime,%cpu,%mem,cmd

    echo
    echo "The detached EC2 job is still running."

elif [ "$JOB_STATUS" = "STOPPED" ]; then
    echo
    echo "The detached EC2 driver is stopped."

    if [ -f "$LOG_FILE" ]; then
        if grep -q \
            "BATCH PIPELINE: SUCCESS" \
            "$LOG_FILE"; then

            echo "COMPLETION_RESULT=SUCCESS"

        elif grep -q \
            "Traceback (most recent call last)" \
            "$LOG_FILE"; then

            echo "COMPLETION_RESULT=FAILED"

            echo
            echo "Latest traceback/error markers:"

            grep -n \
                -E \
                "Traceback|RuntimeError|AnalysisException|ERROR|Return code: 1" \
                "$LOG_FILE" \
                | tail -20 \
                || true
        else
            echo "COMPLETION_RESULT=UNKNOWN"
        fi
    fi

else
    echo
    echo "PID file is missing or unreadable."
    echo "COMPLETION_RESULT=UNKNOWN"
fi

echo
echo "============================================================"
echo "MONITOR CHECK COMPLETE"
echo "============================================================"

exit 0
"""

monitor_result = run_remote(
    monitor_command,
    timeout=180,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 8 RESULT")
print("=" * 80)

print(
    "Monitor return code:",
    monitor_result.returncode,
)

print("No new job was launched.")
print("No running process was stopped.")
print("No HDFS data was modified.")

HYDERABAD JOB STATUS
Recorded PID: 122114
STATUS=STOPPED

LATEST 150 LOG LINES
26/11/08 11:28:53 INFO BlockManagerMaster: Registering BlockManager BlockManagerId(driver, ip-172-31-30-94.ec2.internal, 32999, None)
26/11/08 11:28:53 INFO BlockManagerMasterEndpoint: Registering block manager ip-172-31-30-94.ec2.internal:32999 with 434.4 MiB RAM, BlockManagerId(driver, ip-172-31-30-94.ec2.internal, 32999, None)
26/11/08 11:28:53 INFO BlockManagerMaster: Registered BlockManager BlockManagerId(driver, ip-172-31-30-94.ec2.internal, 32999, None)
26/11/08 11:28:53 INFO BlockManager: Initialized BlockManager: BlockManagerId(driver, ip-172-31-30-94.ec2.internal, 32999, None)
VALIDATION: enriched 2020
Rows: 4900
Duplicate keys: 0
Invalid rows: 0
VALIDATION: PASS



Return code: 0
Elapsed seconds: 11.77
enriched 2015: VALID, SKIPPING RECOMPUTATION

EPOCH 2015: COMPLETE
Epoch elapsed minutes: 0.89

################################################################################
EPOCH 2020 (10/10)
##

In [32]:
# ============================================================
# CELL 9: HYDERABAD COMPLETION CHECK
#
# Verifies:
#   - Successful batch completion marker
#   - 10 population paths
#   - 10 SMOD paths
#   - 10 built-surface paths
#   - 10 enriched paths
#
# Read-only:
#   - Does not launch Spark
#   - Does not modify HDFS
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]
slug = ACTIVE_CITY_SLUG

cmd = f"""
set -e

# Load the same environment used by the production pipeline.
source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

LOG="{root}/batch/city.log"

echo "============================================================"
echo "HYDERABAD FINAL COMPLETION AUDIT"
echo "============================================================"

# Confirm required tools and services.
command -v hdfs
hdfs version | head -1

jps | awk '{{print $2}}' | grep -qx NameNode
jps | awk '{{print $2}}' | grep -qx DataNode

hdfs dfsadmin -safemode get \
    | grep -qi "Safe mode is OFF"

echo "HDFS environment: PASS"

# Confirm successful batch completion.
test -f "$LOG"

grep -q \
    "BATCH PIPELINE: SUCCESS" \
    "$LOG"

echo "Batch success marker: PASS"

checked_paths=0

for epoch in \
    1975 1980 1985 1990 1995 \
    2000 2005 2010 2015 2020

do
    for product in \
        population \
        smod \
        built_surface

    do
        path="/data/ghsl/curated/cities/city={slug}/product=${{product}}/epoch=${{epoch}}/grid_1km"

        if ! hdfs dfs -test -e "$path"; then
            echo "MISSING: $path"
            exit 31
        fi

        checked_paths=$((checked_paths + 1))
    done

    enriched_path="/data/ghsl/curated/cities/city={slug}/enriched_grid_year/epoch=${{epoch}}"

    if ! hdfs dfs -test -e "$enriched_path"; then
        echo "MISSING: $enriched_path"
        exit 32
    fi

    checked_paths=$((checked_paths + 1))

    echo "Epoch $epoch: 4/4 paths PASS"
done

if [ "$checked_paths" -ne 40 ]; then
    echo "ERROR: Expected 40 paths, checked $checked_paths"
    exit 33
fi

echo
echo "Checked HDFS paths: $checked_paths"
echo "ALL_40_PATHS_PASS"
echo "CITY_COMPLETE"
"""

audit_result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 9 RESULT")
print("=" * 80)

print(
    "Return code:",
    audit_result.returncode,
)

print("Expected HDFS paths: 40")
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 9: SUCCESS")

HYDERABAD FINAL COMPLETION AUDIT
/usr/local/hadoop/bin/hdfs
Hadoop 3.4.1
HDFS environment: PASS
Batch success marker: PASS
Epoch 1975: 4/4 paths PASS
Epoch 1980: 4/4 paths PASS
Epoch 1985: 4/4 paths PASS
Epoch 1990: 4/4 paths PASS
Epoch 1995: 4/4 paths PASS
Epoch 2000: 4/4 paths PASS
Epoch 2005: 4/4 paths PASS
Epoch 2010: 4/4 paths PASS
Epoch 2015: 4/4 paths PASS
Epoch 2020: 4/4 paths PASS

Checked HDFS paths: 40
ALL_40_PATHS_PASS
CITY_COMPLETE

REMOTE STDERR:


CELL 9 RESULT
Return code: 0
Expected HDFS paths: 40
Spark launched: NO
HDFS modified: NO
CELL 9: SUCCESS


HYDERABAD PRODUCTION DATA LOCATION
==================================

Status:
- Hyderabad processing is COMPLETE.
- 10 historical epochs processed:
  1975, 1980, 1985, 1990, 1995,
  2000, 2005, 2010, 2015, 2020.
- 4 validated datasets per epoch.
- Total validated HDFS datasets: 40.
- Canonical grid: 70 x 70 = 4,900 cells per epoch.
- CRS: ESRI:54009.
- Grid resolution: 1,000 metres.
- Final audit result:
  ALL_40_PATHS_PASS
  CITY_COMPLETE


PRIMARY HDFS DATA ROOT
----------------------

hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/


POPULATION DATA
---------------

Pattern:

hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/product=population/epoch=<YEAR>/grid_1km

Available years:

1975, 1980, 1985, 1990, 1995,
2000, 2005, 2010, 2015, 2020

Example:

hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/product=population/epoch=2020/grid_1km


SMOD DATA
---------

Pattern:

hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/product=smod/epoch=<YEAR>/grid_1km

Example:

hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/product=smod/epoch=2020/grid_1km


BUILT-SURFACE DATA
------------------

Pattern:

hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/product=built_surface/epoch=<YEAR>/grid_1km

Example:

hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/product=built_surface/epoch=2020/grid_1km


ENRICHED GRID-YEAR DATA
-----------------------

This is the main dataset for future Hyderabad expansion analysis.

Pattern:

hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/enriched_grid_year/epoch=<YEAR>

Example:

hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/enriched_grid_year/epoch=2020

The enriched dataset combines the same-epoch:

- population
- SMOD class
- built-surface area
- non-residential built-surface area
- canonical 1 km grid-cell identifiers

Use this dataset for:

- 1975-to-2020 population growth
- built-surface expansion
- SMOD urban-class transitions
- urban-centre and urban-cluster expansion
- intermediate five-year changes
- multi-city comparison and model features


REMOTE EC2 PRODUCTION ARTIFACTS
-------------------------------

Hyderabad working directory:

/home/ubuntu/ghsl_multicity/hyderabad/

Important contents:

/home/ubuntu/ghsl_multicity/hyderabad/city_config.json

/home/ubuntu/ghsl_multicity/hyderabad/templates/
    population_epoch_template.py
    smod_epoch_template.py
    built_surface_epoch_template.py

/home/ubuntu/ghsl_multicity/hyderabad/programs/
    epoch_1975/
    epoch_1980/
    epoch_1985/
    epoch_1990/
    epoch_1995/
    epoch_2000/
    epoch_2005/
    epoch_2010/
    epoch_2015/
    epoch_2020/

/home/ubuntu/ghsl_multicity/hyderabad/batch/
    run_city_epochs.py
    city.log
    city.pid
    batch_manifest.tsv
    template_checksums.json
    patch manifests and driver backups


MAIN EXECUTION LOG
------------------

/home/ubuntu/ghsl_multicity/hyderabad/batch/city.log

The log contains:

- Spark and Sedona execution details
- epoch-level totals
- validation results
- stage timings
- final BATCH PIPELINE: SUCCESS marker


FINAL BATCH MANIFEST
--------------------

/home/ubuntu/ghsl_multicity/hyderabad/batch/batch_manifest.tsv

The manifest records the final status, elapsed time, and HDFS output
path for every population, SMOD, built-surface, and enriched stage.


IMPORTANT
---------

- The HDFS Parquet datasets are stored persistently on the EC2
  instance's HDFS/EBS storage.
- Closing Colab does not delete these datasets.
- Stopping EC2 stops HDFS temporarily but should retain EBS data.
- Do not terminate EC2 without first checking EBS
  DeleteOnTermination settings or creating an EBS snapshot.
- Do not rerun the Hyderabad production-launch cell.
- Future analytical notebooks should read the enriched_grid_year
  paths directly.
- Hyderabad expansion summaries have not yet been materialized as
  separate analytics tables. They will be calculated from the stored
  enriched grid-year datasets without rerunning raster processing.


  hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/enriched_grid_year/epoch=<YEAR>

## Next city
After Cell 9 passes, change `ACTIVE_CITY_SLUG` in Cell 3 and rerun Cells 3–7. Use Cell 8 to monitor. Process one city at a time. Ahmedabad and Surat remain last.

In [33]:
# ============================================================
# CELL 10: FREEZE FINAL MULTI-CITY PRODUCTION BASELINE
#
# Source:
#   Final corrected Hyderabad driver and templates
#
# Destination:
#   /home/ubuntu/ghsl_multicity_baseline
#
# The baseline contains:
#   - final population template
#   - final SMOD template
#   - final built-surface template
#   - final checkpointed city driver
#   - SHA-256 inventory
#   - provenance metadata
#
# It excludes:
#   - generated epoch programs
#   - PID files
#   - execution logs
#   - old manifests
#   - Hyderabad HDFS data
#
# Safety:
#   - Does not modify Hyderabad source files
#   - Does not modify Bengaluru source files
#   - Does not launch Spark
#   - Does not modify HDFS
# ============================================================

import base64
import hashlib
import json
import textwrap


# ============================================================
# VALIDATE NOTEBOOK STATE
# ============================================================

for required_name in [
    "ACTIVE_CITY",
    "ACTIVE_CITY_SLUG",
    "run_remote",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is unavailable. "
            "Rerun Cells 1, 2, and 3B."
        )


if ACTIVE_CITY_SLUG != "hyderabad":
    raise RuntimeError(
        "Freeze the baseline while Hyderabad is the "
        "active completed city."
    )


source_root = ACTIVE_CITY["remote_work_dir"]

baseline_root = (
    "/home/ubuntu/ghsl_multicity_baseline"
)


# ============================================================
# BUILD REMOTE FREEZE PROGRAM
# ============================================================

remote_freezer = textwrap.dedent(
    f"""
    from pathlib import Path
    import ast
    import hashlib
    import json
    import os
    import py_compile
    import shutil
    import stat
    import time


    SOURCE_ROOT = Path({source_root!r})

    SOURCE_TEMPLATE_ROOT = (
        SOURCE_ROOT / "templates"
    )

    SOURCE_DRIVER = (
        SOURCE_ROOT
        / "batch"
        / "run_city_epochs.py"
    )

    SOURCE_CONFIG = (
        SOURCE_ROOT
        / "city_config.json"
    )

    BASELINE_ROOT = Path(
        {baseline_root!r}
    )

    BASELINE_TEMPLATE_ROOT = (
        BASELINE_ROOT / "templates"
    )

    BASELINE_BATCH_ROOT = (
        BASELINE_ROOT / "batch"
    )

    BASELINE_DRIVER = (
        BASELINE_BATCH_ROOT
        / "run_city_epochs.py"
    )

    BASELINE_CONFIG = (
        BASELINE_ROOT
        / "source_city_config.json"
    )

    TEMPLATE_NAMES = [
        "population_epoch_template.py",
        "smod_epoch_template.py",
        "built_surface_epoch_template.py",
    ]


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(
                    1024 * 1024
                ),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    def utc_now():
        return time.strftime(
            "%Y-%m-%dT%H:%M:%SZ",
            time.gmtime(),
        )


    # --------------------------------------------------------
    # Confirm Hyderabad production artifacts exist.
    # --------------------------------------------------------

    if not SOURCE_ROOT.is_dir():
        raise FileNotFoundError(
            SOURCE_ROOT
        )

    if not SOURCE_DRIVER.is_file():
        raise FileNotFoundError(
            SOURCE_DRIVER
        )

    if not SOURCE_CONFIG.is_file():
        raise FileNotFoundError(
            SOURCE_CONFIG
        )

    source_templates = {{}}

    for template_name in TEMPLATE_NAMES:
        template_path = (
            SOURCE_TEMPLATE_ROOT
            / template_name
        )

        if not template_path.is_file():
            raise FileNotFoundError(
                template_path
            )

        source_templates[
            template_name
        ] = template_path


    # --------------------------------------------------------
    # Validate source Python files before freezing.
    # --------------------------------------------------------

    py_compile.compile(
        str(SOURCE_DRIVER),
        doraise=True,
    )

    for template_path in (
        source_templates.values()
    ):
        py_compile.compile(
            str(template_path),
            doraise=True,
        )


    # --------------------------------------------------------
    # Validate final driver structure.
    # --------------------------------------------------------

    driver_text = SOURCE_DRIVER.read_text(
        encoding="utf-8"
    )

    driver_tree = ast.parse(
        driver_text
    )

    function_names = {{
        node.name
        for node in driver_tree.body
        if isinstance(
            node,
            ast.FunctionDef,
        )
    }}

    required_functions = {{
        "generate_population_program",
        "generate_smod_program",
        "generate_built_program",
        "ensure_stage",
        "verify_template",
        "replace_exact_count",
    }}

    missing_functions = sorted(
        required_functions
        - function_names
    )

    if missing_functions:
        raise RuntimeError(
            "Final driver is missing required "
            "functions: "
            + repr(missing_functions)
        )

    required_driver_fragments = [
        "EXPECTED_TEMPLATE_SHA256",
        "EPOCHS =",
        "spark-submit",
        "BATCH PIPELINE: SUCCESS",
        "existing_valid_skipped",
    ]

    missing_driver_fragments = [
        fragment
        for fragment
        in required_driver_fragments
        if fragment not in driver_text
    ]

    if missing_driver_fragments:
        raise RuntimeError(
            "Final driver is missing required "
            "production fragments: "
            + repr(
                missing_driver_fragments
            )
        )


    # --------------------------------------------------------
    # Validate generalized templates.
    # --------------------------------------------------------

    population_text = source_templates[
        "population_epoch_template.py"
    ].read_text(
        encoding="utf-8"
    )

    smod_text = source_templates[
        "smod_epoch_template.py"
    ].read_text(
        encoding="utf-8"
    )

    built_text = source_templates[
        "built_surface_epoch_template.py"
    ].read_text(
        encoding="utf-8"
    )

    if (
        ".parquet(population_1975_path)"
        in population_text
    ):
        raise RuntimeError(
            "Population template still contains "
            "endpoint-only 1975 self-comparison."
        )

    if (
        ".parquet(SMOD_1975_PATH)"
        in smod_text
    ):
        raise RuntimeError(
            "SMOD template still contains "
            "endpoint-only 1975 self-comparison."
        )

    if (
        "product=population/"
        not in built_text
        or "GRID_YEAR_REFERENCE_PATH"
        not in built_text
    ):
        raise RuntimeError(
            "Built template does not use the "
            "same-epoch population-grid reference."
        )

    for template_name, template_text in [
        (
            "population",
            population_text,
        ),
        (
            "built_surface",
            built_text,
        ),
    ]:
        if (
            "EXPECTED_SELECTED_TILES = 16"
            not in template_text
        ):
            raise RuntimeError(
                f"{{template_name}} template does "
                "not contain the validated Hyderabad "
                "16-tile configuration."
            )


    # --------------------------------------------------------
    # Refuse to overwrite an existing frozen baseline.
    #
    # If a baseline already exists, compare it with the source.
    # --------------------------------------------------------

    if BASELINE_ROOT.exists():
        required_existing_files = [
            BASELINE_DRIVER,
            BASELINE_CONFIG,
            BASELINE_ROOT
            / "baseline_manifest.json",
            BASELINE_ROOT
            / "SHA256SUMS",
        ]

        for existing_path in (
            required_existing_files
        ):
            if not existing_path.is_file():
                raise RuntimeError(
                    "An incomplete baseline directory "
                    "already exists: "
                    + str(BASELINE_ROOT)
                )

        existing_matches = (
            sha256_file(
                BASELINE_DRIVER
            )
            == sha256_file(
                SOURCE_DRIVER
            )
        )

        for template_name in TEMPLATE_NAMES:
            existing_template = (
                BASELINE_TEMPLATE_ROOT
                / template_name
            )

            if not existing_template.is_file():
                existing_matches = False

            elif (
                sha256_file(
                    existing_template
                )
                != sha256_file(
                    source_templates[
                        template_name
                    ]
                )
            ):
                existing_matches = False

        if existing_matches:
            print(
                "Existing baseline matches the "
                "final Hyderabad source."
            )

            print(
                "BASELINE_ALREADY_FROZEN"
            )

            print(
                "MULTICITY BASELINE: SUCCESS"
            )

            raise SystemExit(0)

        raise RuntimeError(
            "A different baseline already exists at "
            + str(BASELINE_ROOT)
            + ". It was not overwritten."
        )


    # --------------------------------------------------------
    # Create baseline directories.
    # --------------------------------------------------------

    BASELINE_TEMPLATE_ROOT.mkdir(
        parents=True,
        exist_ok=False,
    )

    BASELINE_BATCH_ROOT.mkdir(
        parents=True,
        exist_ok=False,
    )


    # --------------------------------------------------------
    # Copy only reusable production files.
    # --------------------------------------------------------

    shutil.copy2(
        SOURCE_DRIVER,
        BASELINE_DRIVER,
    )

    shutil.copy2(
        SOURCE_CONFIG,
        BASELINE_CONFIG,
    )

    copied_templates = []

    for template_name in TEMPLATE_NAMES:
        source_path = (
            source_templates[
                template_name
            ]
        )

        destination_path = (
            BASELINE_TEMPLATE_ROOT
            / template_name
        )

        shutil.copy2(
            source_path,
            destination_path,
        )

        copied_templates.append(
            destination_path
        )


    # --------------------------------------------------------
    # Compile copied files independently.
    # --------------------------------------------------------

    py_compile.compile(
        str(BASELINE_DRIVER),
        doraise=True,
    )

    for template_path in (
        copied_templates
    ):
        py_compile.compile(
            str(template_path),
            doraise=True,
        )


    # --------------------------------------------------------
    # Confirm byte-for-byte copy fidelity.
    # --------------------------------------------------------

    if (
        sha256_file(BASELINE_DRIVER)
        != sha256_file(SOURCE_DRIVER)
    ):
        raise RuntimeError(
            "Baseline driver checksum differs "
            "from source driver."
        )

    for template_name in TEMPLATE_NAMES:
        source_path = (
            SOURCE_TEMPLATE_ROOT
            / template_name
        )

        destination_path = (
            BASELINE_TEMPLATE_ROOT
            / template_name
        )

        if (
            sha256_file(
                destination_path
            )
            != sha256_file(
                source_path
            )
        ):
            raise RuntimeError(
                "Template checksum differs after "
                "copy: "
                + template_name
            )


    # --------------------------------------------------------
    # Save baseline manifest.
    # --------------------------------------------------------

    manifest = {{
        "baseline_name":
            "GHSL Multi-City Production Baseline",
        "baseline_version":
            "1.0.0",
        "created_at_utc":
            utc_now(),
        "source_city":
            "Hyderabad Metropolitan Region",
        "source_city_slug":
            "hyderabad",
        "source_root":
            str(SOURCE_ROOT),
        "baseline_root":
            str(BASELINE_ROOT),
        "validated_epochs": [
            1975,
            1980,
            1985,
            1990,
            1995,
            2000,
            2005,
            2010,
            2015,
            2020,
        ],
        "validated_outputs": 40,
        "canonical_grid": {{
            "crs": "ESRI:54009",
            "resolution_m": 1000,
            "columns": 70,
            "rows": 70,
            "grid_cells": 4900,
        }},
        "validated_source_tile_count": {{
            "population": 16,
            "built_surface": 16,
        }},
        "files": {{
            "driver": {{
                "path": str(
                    BASELINE_DRIVER
                ),
                "sha256": sha256_file(
                    BASELINE_DRIVER
                ),
            }},
            "source_city_config": {{
                "path": str(
                    BASELINE_CONFIG
                ),
                "sha256": sha256_file(
                    BASELINE_CONFIG
                ),
            }},
            "templates": {{
                template_name: {{
                    "path": str(
                        BASELINE_TEMPLATE_ROOT
                        / template_name
                    ),
                    "sha256": sha256_file(
                        BASELINE_TEMPLATE_ROOT
                        / template_name
                    ),
                }}
                for template_name
                in TEMPLATE_NAMES
            }},
        }},
        "usage_notes": [
            (
                "Generate each future city's working "
                "copy from this baseline, not from "
                "the original Bengaluru prototype."
            ),
            (
                "Parameterize city name, slug, "
                "Mollweide extent, paths, selected "
                "tile count, and checksums."
            ),
            (
                "Do not run files directly from the "
                "read-only baseline directory."
            ),
            (
                "Jaipur remains the held-out "
                "generalization city."
            ),
        ],
    }}

    manifest_path = (
        BASELINE_ROOT
        / "baseline_manifest.json"
    )

    manifest_path.write_text(
        json.dumps(
            manifest,
            indent=2,
            sort_keys=True,
        )
        + "\\n",
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # Write standard SHA-256 inventory.
    # --------------------------------------------------------

    checksum_paths = [
        BASELINE_DRIVER,
        BASELINE_CONFIG,
        *copied_templates,
        manifest_path,
    ]

    checksum_lines = []

    for path in sorted(
        checksum_paths,
        key=lambda item: str(item),
    ):
        relative_path = path.relative_to(
            BASELINE_ROOT
        )

        checksum_lines.append(
            sha256_file(path)
            + "  "
            + str(relative_path)
        )

    checksum_path = (
        BASELINE_ROOT
        / "SHA256SUMS"
    )

    checksum_path.write_text(
        "\\n".join(checksum_lines)
        + "\\n",
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # Write baseline README.
    # --------------------------------------------------------

    readme_path = (
        BASELINE_ROOT
        / "README.txt"
    )

    readme_path.write_text(
        "GHSL MULTI-CITY PRODUCTION BASELINE\\n"
        "===================================\\n"
        "\\n"
        "Source city: Hyderabad Metropolitan Region\\n"
        "Validated epochs: 1975-2020, five-year intervals\\n"
        "Validated outputs: 40\\n"
        "Grid: 70 x 70 cells at 1 km resolution\\n"
        "CRS: ESRI:54009\\n"
        "\\n"
        "This baseline contains the final corrected production\\n"
        "driver and templates derived from the completed\\n"
        "Hyderabad processing run.\\n"
        "\\n"
        "Do not execute or modify files directly in this\\n"
        "directory. Create a city-specific working copy first.\\n"
        "\\n"
        "Next production city: Pune Metropolitan Region\\n",
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # Remove generated bytecode from baseline.
    # --------------------------------------------------------

    for pycache in BASELINE_ROOT.rglob(
        "__pycache__"
    ):
        shutil.rmtree(
            pycache
        )


    # --------------------------------------------------------
    # Make baseline files read-only.
    #
    # Directories retain execute permission so they remain
    # traversable.
    # --------------------------------------------------------

    for path in BASELINE_ROOT.rglob("*"):
        if path.is_file():
            path.chmod(
                stat.S_IRUSR
                | stat.S_IRGRP
                | stat.S_IROTH
            )

        elif path.is_dir():
            path.chmod(
                stat.S_IRUSR
                | stat.S_IWUSR
                | stat.S_IXUSR
                | stat.S_IRGRP
                | stat.S_IXGRP
                | stat.S_IROTH
                | stat.S_IXOTH
            )

    BASELINE_ROOT.chmod(
        stat.S_IRUSR
        | stat.S_IWUSR
        | stat.S_IXUSR
        | stat.S_IRGRP
        | stat.S_IXGRP
        | stat.S_IROTH
        | stat.S_IXOTH
    )


    # --------------------------------------------------------
    # Final read-only verification.
    # --------------------------------------------------------

    required_baseline_files = [
        BASELINE_DRIVER,
        BASELINE_CONFIG,
        manifest_path,
        checksum_path,
        readme_path,
        *copied_templates,
    ]

    for path in required_baseline_files:
        if not path.is_file():
            raise RuntimeError(
                "Required baseline file missing: "
                + str(path)
            )

    for path in required_baseline_files:
        permissions = stat.S_IMODE(
            path.stat().st_mode
        )

        if permissions & stat.S_IWUSR:
            raise RuntimeError(
                "Baseline file remains writable: "
                + str(path)
            )


    print("=" * 80)
    print("GHSL MULTI-CITY BASELINE FREEZE")
    print("=" * 80)

    print("Source:", SOURCE_ROOT)
    print("Baseline:", BASELINE_ROOT)
    print("Driver:", BASELINE_DRIVER)

    print()
    print("Templates:")

    for template_path in (
        copied_templates
    ):
        print(" ", template_path)

    print()
    print("Manifest:", manifest_path)
    print("SHA-256 inventory:", checksum_path)
    print("README:", readme_path)

    print()
    print("Source Python compilation: PASS")
    print("Baseline Python compilation: PASS")
    print("Copy checksum verification: PASS")
    print("Generalized-template verification: PASS")
    print("Required driver functions: PASS")
    print("Baseline files read-only: PASS")
    print("Hyderabad HDFS data modified: NO")
    print("Hyderabad source artifacts modified: NO")
    print("Bengaluru artifacts modified: NO")
    print("Spark launched: NO")
    print("HDFS modified: NO")
    print("MULTICITY BASELINE: SUCCESS")
    """
).strip() + "\n"


# ============================================================
# COMPILE LOCALLY
# ============================================================

compile(
    remote_freezer,
    "freeze_multicity_baseline.py",
    "exec",
)

freezer_sha256 = hashlib.sha256(
    remote_freezer.encode("utf-8")
).hexdigest()

print(
    "LOCAL BASELINE FREEZER COMPILE: PASS"
)

print(
    "Expected SHA-256:",
    freezer_sha256,
)


# ============================================================
# TRANSFER AND RUN
# ============================================================

encoded_freezer = base64.b64encode(
    remote_freezer.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

FREEZER="/tmp/freeze_multicity_baseline.py"
EXPECTED_SHA256="{freezer_sha256}"

echo "{encoded_freezer}" \
| base64 -d \
> "${{FREEZER}}"

ACTUAL_SHA256="$(
    sha256sum "${{FREEZER}}" \
    | awk '{{print $1}}'
)"

echo "Remote freezer SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Freezer checksum mismatch."
    exit 20
fi

echo "REMOTE FREEZER CHECKSUM: PASS"

python3 -m py_compile "${{FREEZER}}"

echo "REMOTE FREEZER COMPILE: PASS"

python3 "${{FREEZER}}"
"""


freeze_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 10 RESULT")
print("=" * 80)

print(
    "Return code:",
    freeze_result.returncode,
)

print(
    "Baseline:",
    baseline_root,
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 10: SUCCESS")

LOCAL BASELINE FREEZER COMPILE: PASS
Expected SHA-256: 73cb9a6f1bfdfb0645a2e03972e7db1b7e3fff9a7b3c628a422aa7686c05ae78
Remote freezer SHA-256: 73cb9a6f1bfdfb0645a2e03972e7db1b7e3fff9a7b3c628a422aa7686c05ae78
REMOTE FREEZER CHECKSUM: PASS
REMOTE FREEZER COMPILE: PASS
GHSL MULTI-CITY BASELINE FREEZE
Source: /home/ubuntu/ghsl_multicity/hyderabad
Baseline: /home/ubuntu/ghsl_multicity_baseline
Driver: /home/ubuntu/ghsl_multicity_baseline/batch/run_city_epochs.py

Templates:
  /home/ubuntu/ghsl_multicity_baseline/templates/population_epoch_template.py
  /home/ubuntu/ghsl_multicity_baseline/templates/smod_epoch_template.py
  /home/ubuntu/ghsl_multicity_baseline/templates/built_surface_epoch_template.py

Manifest: /home/ubuntu/ghsl_multicity_baseline/baseline_manifest.json
SHA-256 inventory: /home/ubuntu/ghsl_multicity_baseline/SHA256SUMS
README: /home/ubuntu/ghsl_multicity_baseline/README.txt

Source Python compilation: PASS
Baseline Python compilation: PASS
Copy checksum verification: PASS


We now have a permanent, read-only production baseline on EC2:

Plain Text
/home/ubuntu/ghsl_multicity_baseline/
├── README.txt
├── SHA256SUMS
├── baseline_manifest.json
├── source_city_config.json
├── batch/
│ └── run_city_epochs.py
└── templates/
├── population_epoch_template.py
├── smod_epoch_template.py
└── built_surface_epoch_template.py

Confirmed:

Plain Text
Hyderabad data: unchanged
Hyderabad source code: unchanged
Bengaluru artifacts: unchanged
Baseline checksums: verified
Baseline files: read-only
Spark launched: no
HDFS modified: no

NOTEBOOK 06 FINAL STATUS
========================

Hyderabad Metropolitan Region production processing is complete.

Historical epochs:
1975, 1980, 1985, 1990, 1995,
2000, 2005, 2010, 2015, 2020

Validated outputs:
10 population datasets
10 SMOD datasets
10 built-surface datasets
10 enriched grid-year datasets
Total: 40/40

Hyderabad HDFS root:
hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/

Reusable multi-city production baseline:
/home/ubuntu/ghsl_multicity_baseline/

Final audit:
ALL_40_PATHS_PASS
CITY_COMPLETE
MULTICITY BASELINE: SUCCESS

Do not rerun the Hyderabad detached-launch cell.
Future cities must be generated from the frozen baseline,
not from the original Bengaluru prototype.